# Second Attempts or Second Thoughts? Adaptive reasoning, RL and LLM judges in small language models
This notebook runs the whole study of the paper: data, baselines, the LLM-judge study, LoRA SFT, GRPO (R1, R1b, R2, R3,
three seeds, mechanism controls), DPO, evaluation, multiple attempts and MPV, the second base model, and every table
and figure. Two-stage training: **(A) LoRA SFT** teaches the dual-mode format (`<think>…</think>` + answer, or
`<short>…</short>`), then **(B) GRPO** trains *when* to think, with a verifiable mode-aware reward.

## ▶ How to run
1. Google Colab with an **A100 GPU** (40 or 80 GB). Add your Hugging Face token as a Colab secret named `HF_TOKEN`.
2. In cell **5)**, choose `SCALE` (`"pilot"` = a small wiring test of every stage; `"full"` = the paper study) and, for the
   full study, `SESSION`. The study ran as eleven sessions, **S1 → S11** (plus S2b), one "Run all" each; run them in order
   to reproduce it. Finished stages skip themselves, and all outputs are cached on Google Drive (`MyDrive/slmreasoning`).
3. **Runtime → Run all.** If Colab asks to restart after the install cell, restart and **Run all** again.
4. Cell 42) closes the session (energy and CO2 report) and writes a small results zip to Drive.

Pilot numbers are never results. Greedy decoding on a GPU is not bit-exact across hardware and library versions, so a
rerun can differ slightly from the reported numbers; the paper's numbers come from the outputs released with this code.

## 🧠 Out-of-memory safety (Colab)
Colab may give an **A100-40GB or -80GB**, and memory can fragment over a long session. The notebook adapts by itself:
* **Before every heavy stage** it drops any model the notebook still holds, releases cached GPU memory, and prints `[mem] GPU free … | RAM free …`.
* **vLLM evaluation** sizes its memory share from the memory that is actually free, and on out-of-memory retries with a smaller share and fewer parallel sequences.
* **GRPO** on out-of-memory retries with half and then a quarter of the micro-batch, less vLLM memory, a smaller judge batch, and finally vLLM sleep mode. **The effective batch (64 rollouts per step) never changes.** It resumes from the last checkpoint.
* **The judge inside R2** halves its own batch on out-of-memory, without restarting the run.
* **Inline SFT and DPO** catch out-of-memory, halve the micro-batch and double the gradient accumulation (same effective batch), and resume from the checkpoint.
* **On a 40 GB A100** the micro-batches start halved (same effective batches) and are printed in cell 5.
* **The demo** is skipped on out-of-memory (it is qualitative only).

If you ever see *"still out of memory after N attempts"*: Runtime → Restart session → Run all. Finished stages are cached on Drive.


## 1–3) System, PyTorch and training stack
✅ FIX: v1 uninstalled and reinstalled torch and fought dependency conflicts. Here we install **TRL with its matching vLLM** in one resolver call and keep Colab's CUDA build.

In [ ]:
# 1–3) Install (about 3–5 min). If Colab asks to restart the session, restart and Run all again.
import os, subprocess, sys
if not os.path.exists("/content/.slm_deps_ok"):
    !pip -q install -U "trl[vllm]" peft accelerate datasets "math-verify[antlr4_13_2]" codecarbon bitsandbytes "torchao>=0.16.0"
    !pip -q install -U fastapi uvicorn tensorboard seaborn
    open("/content/.slm_deps_ok", "w").write("ok")
    print("Installed. If imports fail below: Runtime → Restart session, then continue from P0.2.")
else:
    print("Dependencies already installed in this runtime.")

## 3.1 + 4) Versions and CUDA check

In [ ]:
# 3.1 + 4) Imports, library versions, CUDA / A100 check (TF32 on)
import os
# Must be set before torch touches the GPU: reduces fragmentation OOMs in the in-notebook SFT/DPO training.
# (run_script removes it for the vLLM subprocesses, which manage their own memory pool.)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc, glob, hashlib, json, math, random, re, shutil, time, textwrap
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

assert torch.cuda.is_available(), "Select an A100 GPU runtime."
GPU = torch.cuda.get_device_name(0)
assert "A100" in GPU, f"This project is budgeted for A100 only; got {GPU}."
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
VERSIONS = {p: version(p) for p in ["torch", "transformers", "trl", "peft", "vllm", "datasets",
                                     "math-verify", "accelerate", "codecarbon"]}
print(GPU, f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GiB")
print(VERSIONS)

## 5) Drive, HF login, **run scale**, configuration
✅ FIX: v1 kept everything on the runtime disk, so a disconnect lost the work. Now everything lives on Drive, and every stage skips itself once its output exists.

In [ ]:
# 5) Drive, run scale, configuration and stage flags
from google.colab import drive
drive.mount("/content/drive")

SCALE = "full"               # "full" = the paper study · "pilot" = a small wiring test of every stage (not results)
assert SCALE in ("pilot", "full")
PILOT = SCALE == "pilot"

ROOT = Path("/content/drive/MyDrive/" + ("slmreasoning_pilot" if PILOT else "slmreasoning"))
CODE = Path("/content/slm")
DATA, EVAL, RUNS = ROOT / "data", ROOT / "eval", ROOT / "runs"
FIG, TAB, EXP = ROOT / "paper_figures", ROOT / "paper_tables", ROOT / "experiment_data"
for p in (CODE, DATA, EVAL, EVAL / "preds", EVAL / "jobs", RUNS, FIG, TAB, EXP):
    p.mkdir(parents=True, exist_ok=True)
# Optional: an earlier GRPO adapter from a preliminary pipeline (not released); skipped when absent
V1_BUNDLE = Path("/content/drive/MyDrive/full_paper_experiment_bundle")

CFG = dict(
    seed=42,
    policy_base="Qwen/Qwen2.5-Math-1.5B-Instruct",
    # S10–S11: the second base model, a general instruct model with the same size, vocabulary
    # and chat template (checked 2026-10-13: identical vocab/merges/special tokens and prompt rendering). Published
    # (Qwen2.5 blog, instruct table): GSM8K 73.2, MATH 55.2; reported next to our harness, not a gate.
    policy_base2="Qwen/Qwen2.5-1.5B-Instruct", m2_published=dict(gsm8k=0.732, math500=0.552),
    fallback_base="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",   # decision D1 + reference model
    judge_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",     # the v1 judge (studied, not trusted)
    context=4096, vllm_enforce_eager=PILOT,
    # data
    sft_source="Vinnnf/Hybrid-OpenThoughts2-1M-1.5B",
    sft_source_rows=400_000,   # streamed prefix; the pilot found 154,719 usable pairs there (we need 3,000)
    sft_pairs_target=3000, sft_eval_frac=0.05,
    # S2b (2026-10-09): think responses ≤1500 tokens (budget-fit: under GRPO's 2048 cap). The first SFT used ≤3400
    # (median 1787) and its think mode looped (G1/D1 failed); it is kept as the "sft_long" ablation.
    sft_max_target_tokens=1500,
    rl_n_prompts=3000, rl_gsm8k_frac=0.2, dpo_n_prompts=600, rl_max_prompt_tokens=512, ngram=13,
    # evaluation
    eval_max_tokens=3584, ref_max_tokens=8192, aime_samples=4, aime_temperature=0.6,
    aime_samples_main=16,   # G2: AIME has only 30 problems → 16 samples for the models the claims rest on
    aime_main_models=["base", "sft_v2", "grpo_ours", "grpo_stdlen", "grpo_ours_s2", "grpo_ours_s3", "grpo_ours_gstd",
                      "grpo_stdlen_gstd"],
    difficulty_samples=4, multi_models=["base", "grpo_ours"],
    # 12c) multi-perspective reasoning with memory + validation (MPV)
    mpv_models=["base", "sft_v2", "grpo_ours", "dpo_v2"], mpv_n=100, case_n=30,
    mpv_early_models=["base", "sft_v2"],   # S6 (P7_mpv): MPV early on the models that exist, so a weak MPV can be fixed before S7
    mpv_temperature=0.7, mpv_max_tokens=2048, mpv_verify_tokens=1024, mpv_vote_k=8,
    # 12e) S9: k independent samples per problem on the FULL GSM8K + MATH-500 (majority vote
    # with enough items to test the gain; Table 18 breadth vs depth at matched tokens)
    vote_k=8, vote_models=["base", "grpo_ours"],
    judge_large="Qwen/Qwen2.5-7B-Instruct", judge_long_tokens=2048,   # 6b) judge controls
    judge_v1_tokens=512,   # 6b) the v1 setting; the offline study needs the full text to expose the v1 parser artifact
    judge_items_per_class=750, grpo_smoke_steps=15, pilot_bench=None, demo_max_new_tokens=1536,
    demo_n=6, latency_n=10, ctx_n=5, judge_run_steps=None,
    # Stage A: SFT
    sft=dict(lora_r=16, lora_alpha=32, epochs=2, lr=1e-4, batch=8, grad_accum=2, eval_steps=100,
             max_len=4096, max_steps=-1),
    # Stage B: GRPO
    grpo=dict(max_steps=200, per_device_batch=16, grad_accum=4, num_generations=8,
              max_prompt_length=512, max_completion_length=2048, lr=1e-5, beta=0.0,
              vllm_gpu_mem=0.30, save_steps=25, max_hours=4.5, g2_after_step=60,   # a run stops cleanly after max_hours and resumes
              stop_on_collapse=False,    # G2 reported, not enforced (since S4: R1 collapsed to <short> in S3)
              # Advantages = (r − group mean) / batch std (S6). With TRL's default group std,
              # all-correct groups turned the ≤0.2 brevity bonus into a ±1 advantage: R1/R3 compressed their CoT after
              # step ≈130 (S5 greedy MATH: R1 74.2 at step 60 → 50.4 at step 200). Those runs are kept as *_gstd.
              scale_rewards="batch",
              judge_batch=64,
              judge_max_new_tokens=16,   # R2: the verdict is the first ~12 chars (pilot: 80/80 identical decisions vs 512)
              judge_parser="fixed"),   # R2: the judge's actual verdict ("v1" = v1's first-number parser)
    # DPO ablation
    dpo=dict(samples=4, beta=0.1, lr=1e-5, epochs=1, batch=4, grad_accum=4, max_length=2560,
             max_prompt_length=512, lora_r=16, lora_alpha=32),
)
if PILOT:   # every code path at tiny scale
    CFG.update(sft_pairs_target=200, rl_n_prompts=200, dpo_n_prompts=40,
               multi_models=["grpo_ours"], mpv_models=["sft_v2", "grpo_ours"], mpv_n=8,
               judge_items_per_class=40, ref_max_tokens=4096,
               pilot_bench=dict(gsm8k=50, math500=50, aime2026=5), demo_max_new_tokens=512,
               demo_n=3, latency_n=3, ctx_n=3, judge_run_steps=3)
    CFG["sft"].update(epochs=1, eval_steps=20, max_steps=25)
    CFG["grpo"].update(max_steps=5, save_steps=2, max_hours=0.3, g2_after_step=10_000)

# Adaptive to the A100 Colab assigns (40 GB or 80 GB): smaller micro-batches on 40 GB, SAME effective batches
# (batch × grad-accum), so results stay comparable across sessions. OOM retries shrink further if ever needed.
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
if GPU_GIB < 60:
    CFG["sft"].update(batch=CFG["sft"]["batch"] // 2, grad_accum=CFG["sft"]["grad_accum"] * 2)
    CFG["grpo"].update(per_device_batch=CFG["grpo"]["per_device_batch"] // 2, grad_accum=CFG["grpo"]["grad_accum"] * 2,
                       judge_batch=CFG["grpo"]["judge_batch"] // 2)
    CFG["dpo"].update(batch=max(1, CFG["dpo"]["batch"] // 2), grad_accum=CFG["dpo"]["grad_accum"] * 2)
print(f"GPU {GPU_GIB:.0f} GiB → SFT micro-batch {CFG['sft']['batch']}×{CFG['sft']['grad_accum']}, "
      f"GRPO {CFG['grpo']['per_device_batch']}×{CFG['grpo']['grad_accum']}, DPO {CFG['dpo']['batch']}×{CFG['dpo']['grad_accum']}")

# Which GPU stages run in THIS session. The study ran as S1 → S11 (plus S2b), one Colab session per "Run all";
# finished stages skip themselves and everything else reads the cache on Drive. Run them in order to reproduce it.
SESSION = "S1"
SESSION_STAGES = {"S1": ["P1_data", "P2_baselines", "P3_judge"],        # data, benchmarks, baselines, judge study
                  "S2": ["P4_sft"],                                     # SFT with think traces <= 3,400 tokens (sft_long)
                  "S2b": ["P1_data", "P4_sft"],                         # SFT with think traces <= 1,500 tokens (the main SFT)
                  "S3": ["P5_ours"],                                    # R1, group std; stopped by the step-60 collapse check
                  "S4": ["P5_ours", "P5_balanced", "P5_stdlen"],        # R1 (group std) to 200 steps, R1b, R3 (group std)
                  "S5": ["P6_dpo", "P7_eval"],                          # DPO + greedy evaluation of the trained models
                  "S6": ["P5_ours", "P5_stdlen", "P7_eval", "P7_mpv"],  # R1 and R3 with batch std (the group-std runs are
                                                                        # kept as *_gstd), their evaluation, MPV on base + SFT
                  "S7": ["P5_judge", "P5_seed2"],                       # R2 (LLM-judge reward) and R1 seed 2
                  "S8": ["P5_ctrl_nobonus", "P5_ctrl_nostd", "P5_ctrl_b01", "P7_eval"],   # mechanism controls C1-C3 + eval
                  "S9": ["P5_seed3", "P7_final", "P3_judge_onpolicy", "P7_votes", "P10_demo", "P11_bundle"],
                  # R1 seed 3, final evaluation, MPV (rest), Table 3c, full-benchmark votes, demo, bundle
                  "S10": ["P4_m2_sft", "P7_m2_eval", "P5_m2_ours"],     # second base model: SFT, R1, evaluation
                  "S11": ["P5_m2_gstd", "P7_m2_eval", "P11_bundle"]}    # second base model: R1 with group std, evaluation
if not PILOT:   # the settings that changed during the study (each change is described in the paper)
    if SESSION in ("S1", "S2"):
        CFG["sft_max_target_tokens"] = 3400       # the first SFT data used think traces up to 3,400 tokens (sft_long)
    if SESSION in ("S3", "S4", "S5"):
        CFG["grpo"]["scale_rewards"] = "group"    # TRL's default; from S6 on batch std, and the group-std runs become *_gstd
    if SESSION == "S3":
        CFG["grpo"]["stop_on_collapse"] = True    # S3 stopped R1 at the step-60 mode-collapse check (its step-60 adapter)
STAGES = {k: k in SESSION_STAGES[SESSION] for k in
          ("P1_data", "P2_baselines", "P3_judge", "P4_sft", "P5_smoke", "P5_ours", "P5_balanced", "P5_judge", "P5_stdlen",
           "P5_seed2", "P6_dpo", "P7_eval", "P7_mpv", "P7_final", "P10_demo", "P11_bundle", "P5_ctrl_nobonus",
           "P5_ctrl_nostd", "P5_ctrl_b01", "P5_seed3", "P3_judge_onpolicy", "P7_votes", "P4_m2_sft", "P5_m2_ours",
           "P5_m2_gstd", "P7_m2_eval")}
if PILOT:
    STAGES = {k: True for k in STAGES}
    STAGES["P5_smoke"] = False
    STAGES["P5_stdlen"] = False     # R3 = R1 with the standard length bonus (same code path); full study only
    STAGES["P5_balanced"] = False   # R1b = R1 with balanced rollouts; full study only (mock-tested offline)
    STAGES["P5_seed2"] = False      # same code path as R1; full study only
    STAGES["P5_seed3"] = False
    for _k in ("P4_m2_sft", "P5_m2_ours", "P5_m2_gstd", "P7_m2_eval"):
        STAGES[_k] = False          # the second model is full-study only (same code paths as SFT, R1 and eval)
(ROOT / "config.json").write_text(json.dumps({"SCALE": SCALE, "CFG": CFG, "VERSIONS": VERSIONS, "GPU": GPU}, indent=1))
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print(f"SCALE={SCALE} → {ROOT} | session {SESSION if not PILOT else 'pilot'} | "
      f"stages on: {[k for k, v in STAGES.items() if v is True]}")

In [ ]:
# 5.1) Hugging Face login (Colab secret HF_TOKEN; never hard-code tokens)
from google.colab import userdata
from huggingface_hub import login, HfApi
try:
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped (public models/datasets still work):", e)
HF = HfApi()

## 5.2) Carbon tracking
CodeCarbon measures the energy use and emissions of every session (Table 4).

In [ ]:
# 5.2) Carbon tracking (energy and CO2 of every session, Table 4) and stage markers
from codecarbon import EmissionsTracker
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def mark_stage(stage, note=""):
    """Prints the end of a stage."""
    print(f"[stage] {stage} done ({datetime.now(timezone.utc).isoformat(timespec='seconds')})")

tracker = EmissionsTracker(project_name="slm_adaptive_reasoning_v2", output_dir=str(ROOT),
                           output_file="emissions.csv", log_level="error")
tracker.start()
print("Session", SESSION_ID, "started; carbon tracking on.")

## 5.2b) Stage gate
A training stage runs only if its final adapter does not exist yet, so an interrupted study resumes where it stopped and
a finished run is never trained twice.

In [ ]:
# 5.2b) Stage gate: skip a training stage whose final adapter already exists
_ARTEFACT = {"P4_sft": ("sft_v2", "adapter/adapter_config.json"),
             "P5_ours": ("grpo_ours", "adapter/adapter_config.json"),
             "P5_balanced": ("grpo_balanced", "adapter/adapter_config.json"),
             "P5_judge": ("grpo_judge", "adapter/adapter_config.json"),
             "P5_stdlen": ("grpo_stdlen", "adapter/adapter_config.json"),
             "P5_seed2": ("grpo_ours_s2", "adapter/adapter_config.json"),
             "P5_seed3": ("grpo_ours_s3", "adapter/adapter_config.json"),
             "P5_ctrl_nobonus": ("grpo_nobonus_gstd", "adapter/adapter_config.json"),
             "P5_ctrl_nostd": ("grpo_ours_nostd", "adapter/adapter_config.json"),
             "P5_ctrl_b01": ("grpo_ours_gstd_b01", "adapter/adapter_config.json"),
             "P6_dpo": ("dpo_v2", "merged/config.json"),
             "P4_m2_sft": ("m2_sft", "adapter/adapter_config.json"),
             "P5_m2_ours": ("m2_grpo", "adapter/adapter_config.json"),
             "P5_m2_gstd": ("m2_grpo_gstd", "adapter/adapter_config.json")}

def stage_gate(key):
    """Call before a training stage. Returns (run, overrides); no overrides in the public notebook."""
    if key in _ARTEFACT and (RUNS / _ARTEFACT[key][0] / _ARTEFACT[key][1]).exists():
        print(f"[stage] {key}: already trained (runs/{_ARTEFACT[key][0]}), skipped")
        return False, {}
    return True, {}

## 5.3) Shared code (written to `/content/slm/`)
* `common.py`: **one** prompt builder, the response-mode parser, the **strict scorer**, the GRPO rewards, the v1 judge prompt/parsers and the statistics. Training and evaluation import the same file, so they can never silently disagree.
* `gen_worker.py`: batched **vLLM** generation in a fresh process. ✅ FIX: v1 generated one question at a time with HF `generate` (about 3 A100-h for 270 rows). vLLM evaluates full benchmarks in minutes, with no repetition penalty.
* `train_grpo.py`: Stage B, run as a separate process. Its colocated vLLM engine cannot be freed inside a notebook, and as a process it resumes from checkpoints after a disconnect.
* `backend_api.py`: single-pass serving.

In [ ]:
%%writefile /content/slm/common.py
# common.py: shared code for SFT, GRPO, DPO, evaluation and the backend.
# The prompt, the response modes, the strict scorer and the rewards are defined ONCE here
# so training and evaluation can never silently disagree.
import math, os, re, json, hashlib, unicodedata, zlib
from collections import Counter, defaultdict
import numpy as np
from fractions import Fraction
from functools import lru_cache

try:
    from math_verify import LatexExtractionConfig, ExprExtractionConfig, parse, verify
except ImportError:  # allows importing for pure-python checks without math-verify
    parse = verify = LatexExtractionConfig = ExprExtractionConfig = None
try:     # math-verify's TimeoutException subclasses BaseException, so `except Exception` does NOT catch it
    from math_verify.errors import TimeoutException as _MVTimeout
except ImportError:
    class _MVTimeout(BaseException):
        pass
MV_ERRORS = (Exception, _MVTimeout)   # every math-verify call: a timeout = "not verified", never a crash
# math-verify enforces timeouts with signals on Linux (Colab); on Windows it would spawn processes.
import threading


def _to():
    """5 s math-verify timeout on Linux main thread; disabled where signals are unavailable."""
    if os.name == "nt" or threading.current_thread() is not threading.main_thread():
        return None
    return 5

# --------------------------------------------------------------------------------------
# Prompting
# --------------------------------------------------------------------------------------
# One system prompt for every trained v2 policy (SFT, GRPO, DPO) and for their evaluation.
POLICY_SYSTEM = (
    "You are a careful mathematical reasoner. Choose exactly one response mode. "
    "For problems that need deliberation, write <think> your step-by-step reasoning </think> "
    "and then the final answer in \\boxed{}. For problems you can answer directly, write "
    "<short> a brief solution ending with the final answer in \\boxed{} </short>."
)
# Official Qwen2.5-Math CoT prompt, used ONLY for the untouched base-model baseline (gate G0).
OFFICIAL_QWEN_MATH_SYSTEM = "Please reason step by step, and put your final answer within \\boxed{}."
# DeepSeek-R1-Distill recommendation: no system prompt, instruction appended to the user turn.
R1_USER_SUFFIX = "\nPlease reason step by step, and put your final answer within \\boxed{}."
# v1 adapters were trained on raw ChatML without a system prompt.
V1_TEMPLATE = "<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

PROMPT_STYLES = ("policy", "official", "r1", "v1")


def build_prompt(tokenizer, question, style="policy", prefill=""):
    """Return the full prompt string (chat template applied). `prefill` forces a mode."""
    if style == "v1":
        return V1_TEMPLATE.format(q=question) + prefill
    if style == "policy":
        msgs = [{"role": "system", "content": POLICY_SYSTEM}, {"role": "user", "content": question}]
    elif style == "official":
        msgs = [{"role": "system", "content": OFFICIAL_QWEN_MATH_SYSTEM}, {"role": "user", "content": question}]
    elif style == "r1":
        msgs = [{"role": "user", "content": question + R1_USER_SUFFIX}]
    elif style == "thinkless":      # the Thinkless model card: instruction first, then the problem, one user turn
        msgs = [{"role": "user", "content": R1_USER_SUFFIX.strip() + "\n" + question}]
    else:
        raise ValueError(style)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill


def text_hash(text):
    text = " ".join(unicodedata.normalize("NFC", str(text)).split())
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# --------------------------------------------------------------------------------------
# Response modes
# --------------------------------------------------------------------------------------
TAG_RE = re.compile(r"</?(?:think|short)>", re.I)
SHORT_RE = re.compile(r"\s*<short>(.*?)</short>\s*", re.S | re.I)
THINK_RE = re.compile(r"\s*<think>(.*?)</think>(.*)", re.S | re.I)


def parse_policy_output(text):
    """Classify a v2-policy completion.
    Returns (mode, answer_region, think_text):
      short     : <short> ... </short> and nothing else
      think     : <think> ... </think> followed by a non-empty answer region
      untagged  : no mode tags at all
      malformed : anything else (unclosed/truncated think, nested or extra tags, text after </short>)
    """
    t = str(text)
    m = SHORT_RE.fullmatch(t)
    if m and not TAG_RE.search(m[1]) and m[1].strip():
        return "short", m[1].strip(), ""
    m = THINK_RE.fullmatch(t)
    if m and not TAG_RE.search(m[1]) and not TAG_RE.search(m[2]) and m[1].strip() and m[2].strip():
        return "think", m[2].strip(), m[1].strip()
    if not TAG_RE.search(t):
        return "untagged", t.strip(), ""
    return "malformed", None, ""


def parse_plain_output(text):
    """Base / reference models: answer region is the text after the last </think> if present."""
    t = str(text)
    if re.search(r"</think>", t, re.I):
        head, tail = re.split(r"</think>", t, flags=re.I)[-2:]
        return "think", tail.strip(), head
    if re.search(r"<think>", t, re.I):  # opened but never closed = truncated reasoning
        return "malformed", None, t
    return "untagged", t.strip(), ""


def parse_hybrid_output(text):
    """Published hybrid models (prior-work baselines). Thinkless opens with <short> (unclosed, as in its
    Hybrid-OpenThoughts2 training data) or <think> ... </think>; AdaptThink's NoThinking mode closes an EMPTY
    thinking block (its chat template opens <think>). Mode = whether any reasoning precedes the last </think>."""
    t = str(text)
    low = t.lower()
    end = low.rfind("</think>")
    if end >= 0:
        thought = TAG_RE.sub("", t[:end]).strip()
        return ("think" if thought else "short"), t[end + len("</think>"):].strip(), thought
    if re.match(r"\s*<short>", t, re.I):
        return "short", TAG_RE.sub("", t).strip(), ""
    if "<think>" in low:                       # opened but never closed = truncated reasoning
        return "malformed", None, t
    return "untagged", t.strip(), ""


def classify(text, fmt):
    if fmt == "policy":
        return parse_policy_output(text)
    return parse_hybrid_output(text) if fmt == "hybrid" else parse_plain_output(text)


def format_ok(text):
    """Strict v2 format: exactly one mode, closed tags, final answer in \\boxed{}."""
    mode, region, _ = parse_policy_output(text)
    if mode not in ("think", "short"):
        return False
    boxes = boxed_answers(region)
    return bool(boxes) and bool(boxes[-1])

# --------------------------------------------------------------------------------------
# Strict scorer (from the validated v1 evaluation notebook, extended for modes and MC)
# --------------------------------------------------------------------------------------

def boxed_answers(text):
    """Balanced-brace \\boxed{...} extraction. Returns None for an unfinished box."""
    answers = []
    for m in re.finditer(r"\\boxed\s*\{", str(text)):
        start, depth, end = m.end(), 1, m.end()
        while end < len(text) and depth:
            ch = text[end]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
            end += 1
        if depth:
            return None
        answers.append(text[start:end - 1].strip())
    return answers


def rational(text):
    s = str(text).strip().replace("\u2212", "-")
    s = re.sub(r"^\$|\$$", "", s).strip()
    s = re.sub(r"^\\\(|\\\)$", "", s).strip()
    s = re.sub(r"\\(?:text|mathrm)\{[^}]*\}", "", s).strip()   # units such as \text{ dollars}
    s = s.replace("\\$", "").replace("\\%", "").replace("%", "").strip()
    s = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", s)            # thousands separators only
    frac = re.fullmatch(r"([+-]?)\\(?:d|t)?frac\s*\{([+-]?\d+)\}\s*\{([+-]?\d+)\}", s)
    if frac:
        if int(frac[3]) == 0:
            return None
        return (-1 if frac[1] == "-" else 1) * Fraction(int(frac[2]), int(frac[3]))
    s = re.sub(r"\s+", "", s)
    if not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:/[+-]?\d+(?:\.\d*)?)?", s):
        return None
    try:
        parts = s.split("/")
        return Fraction(parts[0]) / Fraction(parts[1]) if len(parts) == 2 else Fraction(s)
    except (ValueError, ZeroDivisionError):
        return None


def _mv_parse(expr):
    return parse("\\boxed{" + expr + "}", extraction_config=[LatexExtractionConfig()],
                 fallback_mode="no_fallback", extraction_mode="first_match",
                 parsing_timeout=_to(), raise_on_error=True)


@lru_cache(maxsize=8192)
def symbolic_gold(gold):
    result = _mv_parse(gold)
    if not result:
        raise ValueError("Unparseable authoritative gold answer: " + repr(gold))
    return result


def extract_final(text, fmt="policy"):
    """Return (mode, answer_or_None, extraction_status)."""
    mode, region, _ = classify(text, fmt)
    if region is None:
        return mode, None, "malformed_or_truncated"
    boxes = boxed_answers(region)
    if boxes is None:
        return mode, None, "unfinished_box"
    if boxes:
        return (mode, boxes[-1], "boxed") if boxes[-1] else (mode, None, "empty_box")
    if mode == "short" and len(region) <= 40:   # e.g. <short>42</short>
        return mode, region, "short_text"
    return mode, None, "missing_final_answer"


NUMERIC_TASKS = ("gsm8k", "aime2026")

def score_text(text, gold, task, fmt="policy"):
    """Strict correctness of one completion. Failures stay in the denominator."""
    mode, answer, extraction = extract_final(text, fmt)
    res = {"correct": False, "final_answer": answer, "extraction": extraction, "mode": mode,
           "needs_review": False}
    if answer is None:
        res["status"] = "final_parse_failure"
        return res
    if task in NUMERIC_TASKS:
        expected = rational(gold)
        if expected is None:
            raise ValueError("Invalid numeric gold: " + repr(gold))
        actual = rational(answer)
        if actual is None and parse is not None:      # e.g. \boxed{18\text{ apples}} or 3\cdot 6
            try:
                res["correct"] = bool(verify(_mv_parse(str(gold)), _mv_parse(answer), strict=True,
                                             float_rounding=12, numeric_precision=30, timeout_seconds=_to()))
            except MV_ERRORS:
                res["correct"] = False
            res["status"] = "match" if res["correct"] else "final_parse_failure"
            res["needs_review"] = not res["correct"]
            return res
        res["correct"] = actual == expected
        res["status"] = "match" if res["correct"] else "numeric_mismatch"
        return res
    if task in ("math500", "math_train"):
        if rational(answer) is not None and rational(gold) is not None:
            res["correct"] = rational(answer) == rational(gold)
            res["status"] = "match" if res["correct"] else "numeric_mismatch"
            return res
        try:
            expected = symbolic_gold(str(gold))
            actual = _mv_parse(answer)
            if not actual:
                res.update(status="symbolic_parse_failure", needs_review=True)
                return res
            res["correct"] = bool(verify(expected, actual, strict=True, float_rounding=12,
                                         numeric_precision=30, timeout_seconds=_to(), raise_on_error=True))
            res.update(status="match" if res["correct"] else "symbolic_nonmatch",
                       needs_review=not res["correct"])
        except MV_ERRORS as exc:
            res.update(status="verifier_error", needs_review=True, verifier_error=repr(exc)[:200])
        return res
    raise ValueError("No validated scorer for task: " + task)


def second_scorer(text, gold, fmt="policy"):
    """Independent, lenient math-verify scorer (default extraction over the answer region).
    Used ONLY to report scorer agreement, never as the headline metric."""
    _, region, _ = classify(text, fmt)
    if region is None or parse is None:
        return False
    try:
        g = parse("$" + str(gold) + "$", parsing_timeout=_to())
        a = parse(region, parsing_timeout=_to())
        return bool(g and a and verify(g, a, timeout_seconds=_to()))
    except MV_ERRORS:
        return False

# --------------------------------------------------------------------------------------
# Repetition and trace statistics
# --------------------------------------------------------------------------------------

LOOP_MIN_REPS, LOOP_MIN_WORDS, LOOP_MAX_PERIOD = 5, 50, 200
LOOP_TAIL_CHARS, LOOP_TAIL_ZLIB = 3000, 0.15


def has_repetition_loop(text, min_reps=LOOP_MIN_REPS, min_words=LOOP_MIN_WORDS, max_period=LOOP_MAX_PERIOD,
                        tail_chars=LOOP_TAIL_CHARS, tail_zlib=LOOP_TAIL_ZLIB):
    """Degenerate-loop detector (v2, 2026-10-09). Flags (a) a block of 1..max_period words repeated back-to-back
    >= min_reps times and covering >= min_words words, anywhere in the text, or (b) a near-verbatim loop: the last
    tail_chars characters compress (zlib) below tail_zlib. Calibrated on the 12,205 full-scale S1 outputs: 0 of 8,928
    correct untruncated answers flagged, 52% of truncated ones. v1's rule (any word 4-gram seen 6 times, or one
    duplicated line) flagged 21% of normal base answers and 51% of R1-Distill answers: it measured length, not loops."""
    t = str(text)
    if len(t) >= tail_chars:
        tail = t[-tail_chars:].encode("utf-8", "ignore")
        if len(zlib.compress(tail, 9)) < tail_zlib * len(tail):
            return True
    words = t.split()
    n = len(words)
    if n < min_words:
        return False
    ids = {}
    a = np.fromiter((ids.setdefault(w, len(ids)) for w in words), dtype=np.int64, count=n)
    for p in range(1, max_period + 1):
        need = max((min_reps - 1) * p, min_words - p)     # run of equal words at lag p ⇔ (run + p) / p repetitions
        if need > n - p:
            break
        run = np.concatenate(([0], np.cumsum(a[p:] == a[:-p])))
        if (run[need:] - run[:-need] == need).any():
            return True
    return False


def chose_think(mode, text, template_opens_think=False):
    """The model's mode DECISION, used for every think-rate in the analysis: a closed think block, or a thinking block
    cut off before </think> (by the token limit). template_opens_think: the chat template itself opens <think>
    (R1-style models such as AdaptThink), so text without </think> is unfinished thinking. Format failures are
    reported separately (format_ok, truncated)."""
    if mode == "think":
        return True
    t = str(text).lstrip()
    if t.startswith("<think>"):
        return True
    return bool(template_opens_think) and "</think>" not in t


SELF_CHECK_RE = re.compile(r"\b(wait|let me (?:check|verify|re-?check)|double[- ]check|verify|hmm|"
                           r"actually|on second thought|let's check)\b", re.I)


def self_check_markers(text):
    return len(SELF_CHECK_RE.findall(str(text)))

# --------------------------------------------------------------------------------------
# GRPO reward functions (TRL signature: prompts, completions, **dataset columns)
# Dataset columns used: gold, task.  TRL passes completion_ids in recent versions.
# --------------------------------------------------------------------------------------
L_MAX = 2048            # = GRPO max_completion_length
LENGTH_ALPHA = 0.2


def _texts(completions):
    out = []
    for c in completions:
        if isinstance(c, list):               # conversational format fallback
            c = c[-1]["content"] if c else ""
        out.append(str(c))
    return out


def _ntoks(texts, kwargs):
    ids = kwargs.get("completion_ids")
    if ids is not None:
        return [len(x) for x in ids]
    return [max(1, int(len(t) / 3.6)) for t in texts]   # rough fallback, logged as such


def correctness_reward(prompts=None, completions=None, gold=None, task=None, **kwargs):
    """1 if the strictly scored final answer is correct, else 0 (truncated -> malformed -> 0)."""
    return [1.0 if score_text(t, g, k, "policy")["correct"] else 0.0
            for t, g, k in zip(_texts(completions), gold, task)]


def format_reward(completions=None, **kwargs):
    return [0.1 if format_ok(t) else -0.5 for t in _texts(completions)]


def repetition_reward(completions=None, **kwargs):
    return [-0.5 if has_repetition_loop(t) else 0.0 for t in _texts(completions)]


def mode_length_reward(prompts=None, completions=None, gold=None, task=None, **kwargs):
    """Standard length bonus (R3 ablation): CORRECT answers get more for fewer tokens, whatever the problem.
    Never rewards fast wrong answers."""
    texts = _texts(completions)
    lens = _ntoks(texts, kwargs)
    out = []
    for t, n, g, k in zip(texts, lens, gold, task):
        ok = score_text(t, g, k, "policy")["correct"]
        out.append(LENGTH_ALPHA * max(0.0, 1.0 - n / L_MAX) if ok else 0.0)
    return out


# C2 (ours, R1): DIFFICULTY-AWARE MODE REWARD. GRPO already samples a group of G answers per prompt; the group's
# pass rate p is a free, on-policy difficulty estimate (no difficulty model, no extra rollouts). Correct answers get
#   mode bonus   MODE_BETA · (p if <short> else 1 − p if <think>)  → short when the model finds it easy, think when hard
#   length bonus MODE_LEN_ALPHA · (n_max − n)/(n_max − n_min) among the group's correct answers in the SAME mode
# (0.5 · MODE_LEN_ALPHA if that mode has a single correct answer). Wrong answers get 0. The total bonus budget
# (0.1 + 0.1) equals the standard length bonus (LENGTH_ALPHA = 0.2), so R1 vs R3 isolates HOW the bonus is allocated.
MODE_BETA, MODE_LEN_ALPHA = 0.1, 0.1


def adaptive_bonus(groups, correct, modes, lens):
    """Difficulty-aware bonus per completion; `groups` = one key per completion (same key = same prompt)."""
    by = defaultdict(list)
    for i, g in enumerate(groups):
        by[g].append(i)
    out = [0.0] * len(correct)
    for idx in by.values():
        p = sum(bool(correct[i]) for i in idx) / len(idx)
        for mode in ("think", "short"):
            ok = [i for i in idx if correct[i] and modes[i] == mode]
            if not ok:
                continue
            lo, hi = min(lens[i] for i in ok), max(lens[i] for i in ok)
            for i in ok:
                rel = (hi - lens[i]) / (hi - lo) if hi > lo else 0.5
                out[i] = MODE_BETA * (p if mode == "short" else 1.0 - p) + MODE_LEN_ALPHA * rel
    return out


def group_keys(n, prompts=None, kwargs=None):
    """Group id per completion: the dataset `id` column (TRL passes it), else the prompt text, else one group."""
    ids = (kwargs or {}).get("id")
    if ids is not None and len(ids) == n:
        return list(ids)
    if prompts is not None and len(prompts) == n:
        return [str(p) for p in prompts]
    return [0] * n


def adaptive_mode_reward(prompts=None, completions=None, gold=None, task=None, **kwargs):
    texts = _texts(completions)
    correct = [score_text(t, g, k, "policy")["correct"] for t, g, k in zip(texts, gold, task)]
    modes = [parse_policy_output(t)[0] for t in texts]
    return adaptive_bonus(group_keys(len(texts), prompts, kwargs), correct, modes, _ntoks(texts, kwargs))


# Monitoring-only "rewards" (weight 0): TRL logs their means, giving think-rate etc. per step.
def think_rate(completions=None, **kwargs):
    """Mode decision, as in the evaluation (chose_think): thinking cut off at the token limit still counts."""
    return [1.0 if chose_think(parse_policy_output(t)[0], t) else 0.0 for t in _texts(completions)]


def think_correct(prompts=None, completions=None, gold=None, task=None, **kwargs):
    """1 if the rollout chose think AND is correct. Per step: think accuracy = think_correct / think_rate and
    short accuracy = (strict_accuracy − think_correct) / (1 − think_rate), without NaN rewards."""
    texts = _texts(completions)
    acc = correctness_reward(prompts, completions, gold, task, **kwargs)
    return [a if chose_think(parse_policy_output(t)[0], t) else 0.0 for t, a in zip(texts, acc)]


def policy_think_prob(completions=None, policy_think_prob=None, **kwargs):
    """Balanced runs (R1b) force the rollouts 50/50, so think_rate is fixed at 0.5. This logs the policy's OWN
    probability of choosing <think> for the prompt, measured by the rollout function (None if not measured; TRL
    ≥1.15 turns None into NaN, excluded from the logged mean, and the weight is 0)."""
    n = len(_texts(completions))
    return list(policy_think_prob) if policy_think_prob is not None else [None] * n


def all_correct_group(prompts=None, completions=None, gold=None, task=None, **kwargs):
    """1 for every rollout of a group whose rollouts are ALL strictly correct, else 0. Its per-step mean is the
    share of all-correct groups. In those groups only the shaping bonus varies, so group-std normalisation rescales
    the bonus to a unit-size advantage whatever its weight (the S4–S6 CoT collapse, C2f mechanism controls)."""
    keys = group_keys(len(_texts(completions)), prompts, kwargs)
    acc = correctness_reward(prompts, completions, gold, task, **kwargs)
    full = defaultdict(lambda: True)
    for k, a in zip(keys, acc):
        full[k] = full[k] and a == 1.0
    return [1.0 if full[k] else 0.0 for k in keys]


def short_rate(completions=None, **kwargs):
    return [1.0 if parse_policy_output(t)[0] == "short" else 0.0 for t in _texts(completions)]


def malformed_rate(completions=None, **kwargs):
    return [1.0 if parse_policy_output(t)[0] in ("malformed", "untagged") else 0.0
            for t in _texts(completions)]


def strict_accuracy(prompts=None, completions=None, gold=None, task=None, **kwargs):
    return correctness_reward(prompts, completions, gold, task, **kwargs)

# --------------------------------------------------------------------------------------
# LLM-judge reward (v1 judge, kept for the C3 study)
# --------------------------------------------------------------------------------------
JUDGE_PROMPT = """You are a strict, impartial grader. You will see a PROBLEM, a \
REFERENCE SOLUTION, and a CANDIDATE RESPONSE from another AI. Judge ONLY whether \
the candidate's final answer/approach is logically correct and complete, ignoring \
style. Output exactly one line: "SCORE: x.xx" where x.xx is a number from 0.00 \
(completely wrong) to 1.00 (fully correct and complete). No other text.

PROBLEM:
{problem}

REFERENCE SOLUTION:
{reference}

CANDIDATE RESPONSE:
{candidate}

SCORE:"""
JUDGE_DEFAULT_ON_ERROR = 0.5     # v1 behaviour
_JUDGE_NUM_RE = re.compile(r"(\d\.\d+|\d+\.\d+|\d+)")


def judge_prompt(problem, reference, candidate):
    return JUDGE_PROMPT.format(problem=str(problem)[:1500], reference=str(reference)[:1500],
                               candidate=str(candidate)[:1500])


def parse_judge_v1(text):
    """v1 parser (lenient): first number after </think>, clipped to [0,1]; None if absent."""
    if "</think>" in text:
        text = text.split("</think>")[-1]
    m = _JUDGE_NUM_RE.search(text)
    if not m:
        return None
    try:
        return max(0.0, min(1.0, float(m.group(1))))
    except ValueError:
        return None


_SCORE_LINE_RE = re.compile(r"SCORE\s*:\s*\**\s*(-?\d+(?:\.\d+)?)", re.I)
_LEADING_NUM_RE = re.compile(r"\s*\**\s*(-?\d+(?:\.\d+)?)(?![\d.])")


def _valid_score(x):
    v = float(x)
    return v if 0.0 <= v <= 1.0 else None      # an answer like 13535 or 70.00 is not a score


def parse_judge_fixed(raw):
    """Corrected parser: the score the judge actually gave (found in the pilot, 2026-10-08).
    The judge prompt ends with 'SCORE:', so the judge's verdict is the number that COMPLETES it, i.e. the first
    token of the output. The 1.5B R1 judge then writes '</think>' and re-solves the problem. v1 split on
    '</think>' and took the first number of that RE-SOLUTION, clipped to [0,1] → almost always 1.0 = "correct".
      1. a leading number completes 'SCORE:' → that is the verdict (invalid → None if outside [0,1])
      2. otherwise the last explicit 'SCORE: x' (after </think> when present)
    Returns None when the judge gave no valid score."""
    text = str(raw)
    lead = _LEADING_NUM_RE.match(text)
    if lead:
        return _valid_score(lead[1])
    region = re.split(r"</think>", text, flags=re.I)[-1] if re.search(r"</think>", text, re.I) else text
    found = _SCORE_LINE_RE.findall(region)
    return _valid_score(found[-1]) if found else None


def parse_judge_strict(raw):
    """Strict parser: only an exact 'SCORE: x' line (after an optional closed think block)."""
    cleaned = str(raw).strip()
    if "</think>" in cleaned.lower():
        cleaned = re.split(r"</think>", cleaned, flags=re.I)[-1].strip()
    m = re.fullmatch(r"(?:SCORE:\s*)?(0(?:\.\d+)?|1(?:\.0+)?)", cleaned)
    return float(m[1]) if m else None

# --------------------------------------------------------------------------------------
# Multi-perspective reasoning with memory and validation (MPV): test-time method
# 1) K independent attempts, each from a different perspective   (diversity, no error propagation)
# 2) self-validation of every attempt -> VERDICT                   (verification)
# 3) selection: answer groups scored by votes + passed validations (no gold answer used)
# 4) synthesis with a memory of all attempts + verdicts            (final best solution)
# --------------------------------------------------------------------------------------
PERSPECTIVES = [
    ("direct", "Solve it directly, step by step."),
    ("alternative", "Solve it with a different method than the most obvious one (for example work backwards, "
                    "set up equations, or use a formula instead of enumeration)."),
    ("first-principles", "Solve it from first principles: restate what is given and what is asked, "
                         "then derive the answer carefully."),
    ("solve-and-verify", "Solve it, then verify the result (substitute it back or check it another way) "
                         "and correct any mistake you find."),
]
# Validation by INDEPENDENT RE-DERIVATION (pilot, 2026-10-08: a math-specialised 1.5B model does not follow
# "end with VERDICT: ..." meta-instructions; it re-solves instead). So it is asked to do exactly that: re-solve,
# box its own answer; verdict = whether it matches the candidate. An explicit VERDICT line is honoured if written.
VALIDATE_TEMPLATE = ("A previous solution to the problem below concluded that the final answer is {ans}.\n"
                     "Excerpt of that solution:\n{sol}\n\nVerify it: solve the problem again yourself, carefully and "
                     "independently, recomputing every step, and put YOUR final answer in \\boxed{{}}.\n\n"
                     "Problem:\n{q}")
# Synthesis: memory first, problem LAST, so the natural continuation is solving it (the pilot model echoed a
# long memory-after-problem prompt instead of answering).
SYNTH_TEMPLATE = ("Memory of {k} earlier attempts at the problem below:\n{memory}\n\n{selection}\n\n"
                  "Using this memory, solve the problem carefully and write the final answer in \\boxed{{}}.\n\n"
                  "Problem:\n{q}")
VERDICT_RE = re.compile(r"VERDICT\s*[:\-]?\s*\**\s*(CORRECT|INCORRECT)", re.I)


def mpv_attempt_question(question, k):
    return f"{question}\n\nApproach: {PERSPECTIVES[k][1]}"


def parse_verdict(text):
    found = VERDICT_RE.findall(str(text))
    return found[-1].upper() if found else None


def validation_verdict(text, candidate_answer, fmt="policy"):
    """Returns (verdict or None, how): an explicit VERDICT line if present, otherwise whether the validator's own
    re-derived boxed answer matches the candidate's answer. No gold answer is involved."""
    explicit = parse_verdict(text)
    if explicit:
        return explicit, "explicit"
    if candidate_answer is None:
        return None, "no_candidate_answer"
    own = extract_final(text, fmt)[1]
    if own is None:
        return None, "no_rederived_answer"
    return ("CORRECT" if answers_equivalent(own, candidate_answer) else "INCORRECT"), "rederived"


def answers_equivalent(a, b):
    """Candidate-vs-candidate equivalence (no gold involved): exact rational, string, or symbolic."""
    if a is None or b is None:
        return False
    ra, rb = rational(a), rational(b)
    if ra is not None and rb is not None:
        return ra == rb
    if str(a).strip() == str(b).strip():
        return True
    if parse is None:
        return False
    try:
        pa, pb = _mv_parse(str(a)), _mv_parse(str(b))
        return bool(pa and pb and verify(pa, pb, timeout_seconds=_to()) and verify(pb, pa, timeout_seconds=_to()))
    except MV_ERRORS:
        return False


def group_answers(answers):
    """Group equivalent answers; returns [{'answer', 'members': [idx...]}] in first-seen order."""
    groups = []
    for i, a in enumerate(answers):
        if a is None:
            continue
        for g in groups:
            if answers_equivalent(a, g["answer"]):
                g["members"].append(i)
                break
        else:
            groups.append({"answer": a, "members": [i]})
    return groups


def mpv_select(answers, verdicts):
    """Pick the answer group with the most (votes + passed validations); ties -> earliest attempt.
    Returns (selected_index or None, majority_index or None, groups)."""
    groups = group_answers(answers)
    if not groups:
        return None, None, groups
    majority = max(groups, key=lambda g: (len(g["members"]), -g["members"][0]))
    scored = max(groups, key=lambda g: (len(g["members"]) + sum(verdicts[i] == "CORRECT" for i in g["members"]),
                                        -g["members"][0]))
    return scored["members"][0], majority["members"][0], groups

# --------------------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------------------

def wilson(k, n, z=1.959963984540054):
    assert n > 0 and 0 <= k <= n
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def mcnemar_exact(a_only, b_only):
    n = a_only + b_only
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(a_only, b_only) + 1)) / (2 ** n)) if n else 1.0


def holm(pvalues):
    order = sorted(range(len(pvalues)), key=lambda i: pvalues[i])
    adjusted, running = [0.0] * len(pvalues), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * pvalues[i]))
        adjusted[i] = running
    return adjusted


def paired_bootstrap(a, b, draws=2000, seed=42):
    """95% CI of mean(a-b) resampling items (a, b: per-item 0/1 or probabilities)."""
    import numpy as np
    d = np.asarray(a, float) - np.asarray(b, float)
    if np.ptp(d) == 0:                      # Hoeffding fallback for constant samples
        r = 2 * math.sqrt(math.log(2 / 0.05) / (2 * len(d)))
        return float(d.mean()), max(-1.0, d.mean() - r), min(1.0, d.mean() + r)
    rng = np.random.default_rng(seed)
    s = d[rng.integers(0, len(d), size=(draws, len(d)))].mean(axis=1)
    lo, hi = np.quantile(s, [0.025, 0.975])
    return float(d.mean()), float(lo), float(hi)

# --------------------------------------------------------------------------------------
# Self-check (run before any GPU work)
# --------------------------------------------------------------------------------------

def self_check():
    B = "\\boxed"
    # strict numeric scoring
    assert score_text("<short>" + B + "{4.0}</short>", "4", "gsm8k")["correct"]
    assert not score_text("<short>" + B + "{47/3}</short>", "15", "gsm8k")["correct"]
    assert not score_text("<short>Earlier 503. Final " + B + "{10}</short>", "503", "aime2026")["correct"]
    assert score_text("<short>" + B + "{1,248}</short>", "1248", "gsm8k")["correct"]
    assert score_text("<short>" + B + "{\\$18}</short>", "18", "gsm8k")["correct"]
    # refusals and wrong dates never pass
    assert not score_text("<short>I'm here to help with general knowledge.</short>", "393", "aime2026")["correct"]
    assert not score_text("<short>On Wednesday, March 7, 2024 ...</short>", "1248", "gsm8k")["correct"]
    # modes
    assert parse_policy_output("<think>a</think> so " + B + "{3}")[0] == "think"
    assert parse_policy_output("<short>" + B + "{3}</short>")[0] == "short"
    assert parse_policy_output("<think>unfinished reasoning 3")[0] == "malformed"
    assert parse_policy_output("<short>" + B + "{3}</short> extra <short>4</short>")[0] == "malformed"
    assert parse_policy_output("<think>a</think>")[0] == "malformed"           # no answer after think
    assert not score_text("<think>so it is " + B + "{3}", "3", "gsm8k")["correct"]   # truncated think
    assert score_text("<think>work</think>The answer is " + B + "{3}.", "3", "gsm8k")["correct"]
    # answer inside think but none after it does not count
    assert not score_text("<think>" + B + "{3}</think>The end.", "3", "gsm8k")["correct"]
    # plain models (base / R1-distill)
    assert score_text("reasoning </think> final " + B + "{12}", "12", "gsm8k", fmt="plain")["correct"]
    assert score_text("Step 1 ... " + B + "{12}", "12", "gsm8k", fmt="plain")["correct"]
    assert not score_text("<think> long thinking without end " + B + "{12}", "12", "gsm8k", fmt="plain")["correct"]
    # format
    assert format_ok("<short>x " + B + "{1}</short>") and format_ok("<think>w</think> " + B + "{1}")
    assert not format_ok("<short>no box</short>") and not format_ok("plain " + B + "{1}")
    # repetition
    assert has_repetition_loop("<think>" + "the answer is " * 40 + "</think>")
    assert not has_repetition_loop("<think>2 + 3 = 5 so the total is 5</think>")
    # mode decision for think-rates: unfinished thinking counts as thinking
    assert chose_think("think", "<think>a</think> " + B + "{1}") and chose_think("malformed", "<think>cut off at the limit")
    assert not chose_think("short", "<short>" + B + "{1}</short>") and not chose_think("malformed", "<short>never closed")
    assert not chose_think("untagged", "plain answer " + B + "{1}")                        # base model: never thinks
    assert chose_think("untagged", "reasoning cut off", template_opens_think=True)          # AdaptThink, truncated
    assert not chose_think("short", "</think>\n\n" + B + "{1}", template_opens_think=True)  # AdaptThink NoThinking
    # a long normal solution that reuses phrases is NOT a loop (v1's 4-gram rule flagged these)
    steps = " ".join(f"Step {i}: the total number of apples is {3 * i}." for i in range(40))
    assert not has_repetition_loop("<think>" + steps + "</think> " + B + "{117}")
    assert has_repetition_loop("<think>" + steps + " So x = 1. Then" * 12)                  # mid-length loop
    assert has_repetition_loop("Let me compute. " + "We get 10^{3} mod 7 = 6, " * 200)          # long loop, tail check
    # rewards
    comps = ["<short>" + B + "{5}</short>", "<think>long</think> " + B + "{5}", "<short>" + B + "{6}</short>",
             "<think>never closed " + B + "{5}"]
    gold, task = ["5"] * 4, ["gsm8k"] * 4
    assert correctness_reward(None, comps, gold, task) == [1.0, 1.0, 0.0, 0.0]
    ml = mode_length_reward(None, comps, gold, task, completion_ids=[[0] * 50, [0] * 1500, [0] * 10, [0] * 2048])
    assert ml[0] > ml[1] > 0 and ml[2] == 0 and ml[3] == 0
    assert format_reward(comps) == [0.1, 0.1, 0.1, -0.5]
    assert think_rate(comps) == [0.0, 1.0, 0.0, 1.0]                     # cut-off thinking is still a think decision
    assert think_correct(None, comps, gold, task) == [0.0, 1.0, 0.0, 0.0]
    assert policy_think_prob(comps, policy_think_prob=[.2] * 4) == [.2] * 4 and policy_think_prob(comps) == [None] * 4
    # difficulty-aware mode reward: EASY group (4/4 correct) favours short; HARD group (1/4) favours think
    S, T = "<short>" + B + "{5}</short>", "<think>w</think> " + B + "{5}"
    W = "<short>" + B + "{6}</short>"
    easy = [S, S, T, T]
    hard = [T, W, W, W]
    ar = adaptive_mode_reward(None, easy + hard, ["5"] * 8, ["gsm8k"] * 8, id=["e"] * 4 + ["h"] * 4,
                              completion_ids=[[0] * n for n in (50, 100, 500, 1000, 800, 10, 10, 10)])
    assert abs(ar[0] - (MODE_BETA * 1.0 + MODE_LEN_ALPHA * 1.0)) < 1e-9          # shortest correct short, easy
    assert abs(ar[1] - MODE_BETA * 1.0) < 1e-9 and abs(ar[2] - MODE_LEN_ALPHA) < 1e-9 and ar[3] == 0.0
    assert abs(ar[4] - (MODE_BETA * 0.75 + MODE_LEN_ALPHA * 0.5)) < 1e-9 and ar[5:] == [0.0, 0.0, 0.0]
    assert max(ar) <= MODE_BETA + MODE_LEN_ALPHA <= LENGTH_ALPHA + 1e-12        # same bonus budget as R3
    assert group_keys(3, prompts=["a", "a", "b"]) == ["a", "a", "b"] and group_keys(2) == [0, 0]
    assert all_correct_group(None, easy + hard, ["5"] * 8, ["gsm8k"] * 8, id=["e"] * 4 + ["h"] * 4) == [1.0] * 4 + [0.0] * 4
    # published hybrid baselines (Thinkless / AdaptThink output formats)
    assert extract_final("<short>\nso " + B + "{2}", "hybrid")[:2] == ("short", "2")                     # Thinkless short
    assert extract_final("<think>\nhmm</think>\nso " + B + "{2}", "hybrid")[:2] == ("think", "2")        # Thinkless think
    assert extract_final("</think>\n\nDirect: " + B + "{7}", "hybrid")[:2] == ("short", "7")             # AdaptThink NoThinking
    assert extract_final("Let me think... </think> " + B + "{7}", "hybrid")[:2] == ("think", "7")      # AdaptThink Thinking
    assert extract_final("<think>\nnever closed " + B + "{7}", "hybrid")[0] == "malformed"
    # judge parsers
    assert parse_judge_v1(" 1.00") == 1.0 and parse_judge_v1("<think>score 0.3</think> SCORE: 0.9") == 0.9
    assert parse_judge_strict("The answer is 503. SCORE: 0.00") is None
    assert parse_judge_strict("SCORE: 0.75") == 0.75
    # real pilot outputs: the v1 parser reads a number from the judge's own re-solution (34 → clipped 1.0)
    real = " 0.00\n</think>\n\nTo simplify, sqrt(80) = 4 sqrt 5 ... \\boxed{\\dfrac{\\sqrt{179}}{6}}."
    assert parse_judge_v1(real) == 1.0 and parse_judge_fixed(real) == 0.0     # v1 read "80" from the re-solution
    assert parse_judge_fixed(" 13535\n</think>\n\nTo find the greatest integer ...") is None      # an answer, not a score
    assert parse_judge_fixed(" 70.00\n</think>\n\nGeometrically ...") is None
    assert parse_judge_fixed(" 0.67\nThe candidate is partly right. SCORE: 1.00") == 0.67           # completion wins
    assert parse_judge_fixed("Okay, 34 pages...</think>She reads \\boxed{8.5} per day.\n\nSCORE: 0.00") == 0.0
    assert parse_judge_fixed("Okay, let me check 12 and 7 ... (truncated, no verdict)") is None
    assert parse_judge_fixed("<think>2+2=4</think>The candidate is right.\nSCORE: 1.00") == 1.0
    # statistics
    assert abs(mcnemar_exact(3, 5) - 0.7265625) < 1e-12
    lo, hi = wilson(30, 30)
    assert hi > 0.999 and lo < 0.9
    assert all(abs(x - y) < 1e-12 for x, y in zip(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06]))
    # MPV helpers
    assert parse_verdict("... so it holds. VERDICT: INCORRECT") == "INCORRECT"
    assert parse_verdict("**VERDICT: correct**") == "CORRECT" and parse_verdict("no verdict") is None
    sel, maj, groups = mpv_select(["5", "7", "5.0", None], ["INCORRECT", "CORRECT", "INCORRECT", None])
    assert maj == 0 and len(groups) == 2                       # "5" has 2 votes
    assert sel == 0                                            # 2 votes + 0 checks > 1 vote + 1 check
    sel, maj, _ = mpv_select(["5", "7"], ["INCORRECT", "CORRECT"])
    assert maj == 0 and sel == 1                               # tie on votes broken by validation
    assert validation_verdict("I recomputed: total = " + B + "{5}", "5.0") == ("CORRECT", "rederived")
    assert validation_verdict("<short>Recomputing gives " + B + "{7}</short>", "5") == ("INCORRECT", "rederived")
    assert validation_verdict("... VERDICT: INCORRECT", "5") == ("INCORRECT", "explicit")
    assert validation_verdict("I could not finish", "5") == (None, "no_rederived_answer")
    assert validation_verdict(B + "{5}", None) == (None, "no_candidate_answer")
    assert SYNTH_TEMPLATE.format(k=4, memory="m", selection="s", q="Q?").endswith("Problem:\nQ?")
    if parse is not None:   # symbolic checks need math-verify
        assert score_text("<short>" + B + "{2\\sqrt{2}}</short>", "\\sqrt{8}", "math500")["correct"]
        assert not score_text("<short>" + B + "{1.414214}</short>", "\\sqrt{2}", "math500")["correct"]
        assert score_text("<think>w</think> " + B + "{\\frac{1}{2}}", "0.5", "math500")["correct"]
        assert score_text("<short>" + B + "{18\\text{ apples}}</short>", "18", "gsm8k")["correct"]
    return True

# --------------------------------------------------------------------------------------
# Library-version safety: keep only kwargs the installed class accepts, and say what was dropped.
# --------------------------------------------------------------------------------------

def supported_kwargs(cls, kwargs, required=()):
    import dataclasses, inspect
    if dataclasses.is_dataclass(cls):
        names = {f.name for f in dataclasses.fields(cls)}
    else:
        names = set(inspect.signature(cls).parameters)
    kept = {k: v for k, v in kwargs.items() if k in names}
    dropped = sorted(set(kwargs) - set(kept))
    missing = [r for r in required if r not in kept]
    if missing:
        raise RuntimeError(f"{cls.__name__} lacks required options {missing}; check the library version.")
    if dropped:
        print(f"[compat] {cls.__name__} does not support {dropped}; they were NOT applied.")
    return kept

In [ ]:
%%writefile /content/slm/gen_worker.py
# gen_worker.py: batched vLLM generation in a fresh process (clean GPU memory every call).
# Usage: python gen_worker.py job.json
# job = {"model": hf_id_or_dir, "lora": dir_or_null, "max_model_len": int, "gpu_mem": float,
#        "out": path.jsonl, "requests": [{"id", "prompt", "n", "temperature", "top_p",
#        "max_tokens", "seed", "stop"(optional)}]}
import json, sys, time
from pathlib import Path


def main(job_path):
    job = json.loads(Path(job_path).read_text(encoding="utf-8"))
    out = Path(job["out"])
    done_flag = out.with_suffix(".done.json")
    if done_flag.exists():
        print("[gen_worker] already complete:", out)
        return
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    t0 = time.time()
    llm_kwargs = dict(model=job["model"], dtype="bfloat16", max_model_len=job["max_model_len"],
                      gpu_memory_utilization=job.get("gpu_mem", 0.85), seed=0,
                      enable_prefix_caching=True, trust_remote_code=True,
                      enforce_eager=bool(job.get("enforce_eager", False)),   # eager = faster start-up, less memory
                      max_num_seqs=int(job.get("max_num_seqs", 256)))        # lowered automatically after an OOM
    if job.get("lora"):
        llm_kwargs.update(enable_lora=True, max_lora_rank=job.get("max_lora_rank", 64), max_loras=1)
    llm = LLM(**llm_kwargs)
    tok = llm.get_tokenizer()
    load_s = time.time() - t0

    prompts, params, meta = [], [], []
    for r in job["requests"]:
        ptoks = len(tok(r["prompt"], add_special_tokens=False)["input_ids"])
        room = job["max_model_len"] - ptoks
        if room < 16:
            raise ValueError(f"Prompt {r['id']} ({ptoks} tokens) does not fit the context; "
                             "revise the protocol instead of silently dropping it.")
        max_tokens = min(int(r["max_tokens"]), room)
        params.append(SamplingParams(n=int(r.get("n", 1)), temperature=float(r.get("temperature", 0.0)),
                                     top_p=float(r.get("top_p", 1.0)), max_tokens=max_tokens,
                                     seed=r.get("seed"), stop=r.get("stop"),
                                     repetition_penalty=1.0))       # never a repetition penalty
        prompts.append(r["prompt"])
        meta.append({"prompt_tokens": ptoks, "max_tokens": max_tokens,
                     "context_capped": max_tokens < int(r["max_tokens"])})

    # The parameters are explicit, so a model's generation_config.json defaults (e.g. Qwen2.5-1.5B-Instruct:
    # repetition_penalty 1.1, top_k 20) never apply. Record what was actually used.
    sampling = {k: getattr(params[0], k, None) for k in ("temperature", "top_p", "top_k", "repetition_penalty")} if params else {}
    assert all(p.repetition_penalty == 1.0 for p in params), "a repetition penalty slipped into evaluation"

    lora_req = LoRARequest("adapter", 1, job["lora"]) if job.get("lora") else None
    t1 = time.time()
    results = llm.generate(prompts, params, lora_request=lora_req, use_tqdm=True)
    gen_s = time.time() - t1

    tmp = out.with_suffix(".tmp")
    n_out = 0
    with tmp.open("w", encoding="utf-8") as f:
        for r, m, res in zip(job["requests"], meta, results):
            for k, c in enumerate(res.outputs):
                f.write(json.dumps({"id": r["id"], "sample": k, "text": c.text,
                                    "n_tokens": len(c.token_ids), "finish_reason": c.finish_reason,
                                    **m}, ensure_ascii=False) + "\n")
                n_out += 1
    tmp.replace(out)
    done_flag.write_text(json.dumps({"rows": n_out, "load_seconds": load_s, "generate_seconds": gen_s,
                                     "requests": len(prompts), "model": job["model"],
                                     "lora": job.get("lora"), "sampling_first_request": sampling},
                                    indent=2, default=str), encoding="utf-8")
    print(f"[gen_worker] {n_out} completions in {gen_s:.1f}s (load {load_s:.1f}s) -> {out}")


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /content/slm/train_grpo.py
# train_grpo.py: Stage B. GRPO from the SFT-v2 adapter with verifiable (or judge) rewards.
# Usage: python train_grpo.py config.json
# Variants (cfg["variant"]):
#   ours       : strict correctness + format + repetition + difficulty-aware mode reward      (R1, C2)
#   judge      : 1.5B LLM-judge correctness (+ the same difficulty-aware bonus, judge-gated)
#                + format + repetition                                                   (R2, C3)
#   std_length : strict correctness + format + repetition + standard length bonus       (R3 ablation)
#   no_length  : strict correctness + format + repetition                               (control C1, S8)
# cfg["scale_rewards"]: TRL advantage scaling ("group" = TRL default, "batch" = R1/R3 since S6, "none" = Dr. GRPO-style).
# cfg["bonus_scale"]  : multiplies the shaping reward's weight (control C3: 0.1). TRL sums weighted rewards BEFORE
#                       the advantage normalisation, so this is exactly a smaller bonus.
# Every variant LOGS the same monitors (strict accuracy, think/short/malformed rate) with weight 0,
# so training curves are directly comparable and judge reward-hacking is visible.
# cfg["balanced"] = True (R1b): AdaptThink-style balanced think/short exploration (see make_balanced_rollout).
import json, math, os, sys, time
from pathlib import Path

import torch
from datasets import Dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common  # noqa: E402
from common import (correctness_reward, format_reward, repetition_reward, mode_length_reward,
                    adaptive_mode_reward, strict_accuracy, think_rate, short_rate, malformed_rate,
                    think_correct, policy_think_prob, all_correct_group)  # noqa: E402


def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]


# ------------------------------- LLM judge (R2) ---------------------------------------------
class Judge:
    def __init__(self, name, max_new_tokens, batch_size, parser="fixed"):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16,
                                                          attn_implementation="sdpa").cuda().eval()
        self.max_new_tokens, self.batch_size = max_new_tokens, batch_size
        # "fixed" = the judge's actual verdict (the number completing 'SCORE:'); "v1" = v1's first-number parser,
        # which mostly read numbers from the judge's own re-solution (pilot finding). Default: a fair test of the judge.
        self.parse = common.parse_judge_fixed if parser == "fixed" else common.parse_judge_v1
        self.log = []

    @torch.no_grad()
    def score(self, problems, references, candidates):
        """Batched greedy judging. Adaptive: on CUDA OOM the batch is halved (and remembered) instead of crashing."""
        out, i = [], 0
        while i < len(candidates):
            bs = self.batch_size
            prompts = [common.judge_prompt(p, r, c) for p, r, c in
                       zip(problems[i:i + bs], references[i:i + bs], candidates[i:i + bs])]
            try:
                enc = self.tok(prompts, return_tensors="pt", padding=True, truncation=True,
                               max_length=3072).to("cuda")
                gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                          pad_token_id=self.tok.pad_token_id)
            except torch.cuda.OutOfMemoryError:
                enc = gen = None
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                self.batch_size = max(1, bs // 2)
                print(f"[judge] out of memory at batch {bs} → {self.batch_size}", flush=True)
                continue
            for row in gen[:, enc["input_ids"].shape[1]:]:
                raw = self.tok.decode(row, skip_special_tokens=True)
                s = self.parse(raw)
                out.append(common.JUDGE_DEFAULT_ON_ERROR if s is None else s)   # v1 behaviour
            del enc, gen
            i += bs
        return out


JUDGE = None


def judge_correctness(prompts=None, completions=None, question=None, reference=None, completion_ids=None,
                      id=None, **kw):
    """R2 correctness: judge score in [0,1] plus R1's difficulty-aware bonus, computed from the JUDGE's verdicts
    (score ≥ 0.7 counts as correct). R1 and R2 differ only in who decides correctness."""
    texts = common._texts(completions)
    scores = JUDGE.score(question, reference, texts)
    lens = common._ntoks(texts, {"completion_ids": completion_ids})
    modes = [common.parse_policy_output(t)[0] for t in texts]
    keys = common.group_keys(len(texts), prompts, {"id": id})
    bonus = common.adaptive_bonus(keys, [s >= 0.7 for s in scores], modes, lens)
    return [s + b for s, b in zip(scores, bonus)]


# ------------------------------- balanced think/short rollouts (R1b) -------------------------
MODE_TAGS = {"think": "<think>\n", "short": "<short>\n"}


def mode_tag_ids(tok, prompt, prompt_ids):
    """Token ids of each mode tag as written after this prompt (canonical tokenisation of prompt + tag, exactly
    as TRL tokenises text prompts), and the index of the first token where <think> and <short> differ."""
    ids = {}
    for mode, tag in MODE_TAGS.items():
        full = tok(text=[prompt + tag])["input_ids"][0]
        assert list(full[:len(prompt_ids)]) == list(prompt_ids), "the tag merged into the prompt's last token"
        ids[mode] = list(full[len(prompt_ids):])
    k = next(i for i, (a, b) in enumerate(zip(ids["think"], ids["short"])) if a != b)
    return ids, k


def probe_think_prob(trainer, prefixes, think_tok, short_tok):
    """The policy's own share of <think> among the two tags, P(think) / (P(think) + P(short)), at the deciding
    token: one sampled token with the top-20 log-probs from the colocated vLLM engine. None if unavailable."""
    try:
        from vllm import SamplingParams
        outs = trainer.vllm_generation.llm.generate(
            [{"prompt_token_ids": p} for p in prefixes],
            sampling_params=SamplingParams(max_tokens=1, temperature=1.0, logprobs=20), use_tqdm=False)
        res = []
        for o, t, s in zip(outs, think_tok, short_tok):
            lp = o.outputs[0].logprobs[0]
            pt = math.exp(lp[t].logprob) if t in lp else 0.0
            ps = math.exp(lp[s].logprob) if s in lp else 0.0
            res.append(pt / (pt + ps) if pt + ps > 0 else None)
        return res
    except Exception as e:                       # a monitor must never stop training
        print("[balanced] think-probability probe failed:", repr(e)[:200], flush=True)
        return [None] * len(prefixes)


def make_balanced_rollout(tok):
    """AdaptThink-style importance-sampled exploration (Zhang et al. 2025) through TRL's rollout_func. In every
    group of G rollouts the first G/2 are forced to open <think> and the rest <short>, so both modes are always
    trained and compared on the same problem. The forced tag is part of the COMPLETION (scored and trained), and
    its mode-deciding token carries the sampling log-prob log(0.5): TRL's per-token vLLM importance correction
    (mode token_mask) then weights that token by pi_old(mode) / 0.5, the importance-sampled mode gradient. The
    other forced tag tokens carry NaN (ratio 1). Rewards are unchanged (R1's)."""
    def rollout(prompts, trainer):
        G = trainer.num_generations
        assert len(prompts) % G == 0 and all(prompts[i] == prompts[i - i % G] for i in range(len(prompts))), \
            "TRL no longer passes each prompt as G consecutive copies; balanced rollouts rely on it"
        base = [list(x) for x in trainer.processing_class(text=list(prompts))["input_ids"]]
        tags, decide, prefixes, t_tok, s_tok = {}, {}, [], [], []
        for u in range(0, len(prompts), G):
            tags[u], decide[u] = mode_tag_ids(trainer.processing_class, prompts[u], base[u])
            k = decide[u]
            prefixes.append(base[u] + tags[u]["think"][:k])
            t_tok.append(tags[u]["think"][k]); s_tok.append(tags[u]["short"][k])
        p_think = probe_think_prob(trainer, prefixes, t_tok, s_tok)
        gen_prompts, tag_of, lp_pre, forced = [], [], [], []
        for i, ids in enumerate(base):
            u = i - i % G
            mode = "think" if i % G < G // 2 else "short"
            tag = tags[u][mode]
            lp = [float("nan")] * len(tag)
            lp[decide[u]] = math.log(0.5)
            gen_prompts.append(ids + tag); tag_of.append(tag); lp_pre.append(lp); forced.append(mode)
        _, comp, logprobs, _ = trainer.vllm_generation.generate(prompts=gen_prompts, images=None, num_generations=G)
        return {"prompt_ids": base,
                "completion_ids": [t + list(c) for t, c in zip(tag_of, comp)],
                "logprobs": [pre + [l[0] for l in seq] for pre, seq in zip(lp_pre, logprobs)],
                "forced_mode": forced,
                "policy_think_prob": [p_think[i // G] for i in range(len(prompts))]}
    return rollout


# ------------------------------- callbacks --------------------------------------------------
class TimeLimit(TrainerCallback):
    """Stops (and checkpoints) after max_hours, so a long run can continue in the next session."""
    def __init__(self, hours):
        self.limit, self.t0 = hours * 3600, time.time()

    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.limit:
            print(f"[grpo] time limit reached at step {state.global_step}; saving and stopping.")
            control.should_save = True
            control.should_training_stop = True


class CollapseGuard(TrainerCallback):
    """Gate G2: record (and optionally stop) when the mode decision collapses to one mode. metric: think_rate
    (rollouts' own choice) or, for balanced runs whose rollouts are forced 50/50, policy_think_prob. The first
    collapse is written to G2_MODE_COLLAPSE.json and never overwritten."""
    def __init__(self, out, after_step, window, stop, metric="think_rate"):
        self.out, self.after, self.window, self.stop, self.metric = out, after_step, window, stop, metric
        self.hist = []

    def on_log(self, args, state, control, logs=None, **kw):
        logs = logs or {}
        key = next((k for k in logs if self.metric in k and k.endswith("mean")), None)
        if key is None:
            return
        self.hist.append(float(logs[key]))
        if state.global_step % 10 == 0:
            print(f"[grpo] step {state.global_step}: {self.metric}={logs[key]:.3f} "
                  + " ".join(f"{k.split('/')[1]}={v:.3f}" for k, v in logs.items()
                             if k.startswith("rewards/") and k.endswith("/mean")), flush=True)
        if state.global_step >= self.after and len(self.hist) >= self.window:
            m = sum(self.hist[-self.window:]) / self.window
            if m < 0.05 or m > 0.95:
                f = self.out / "G2_MODE_COLLAPSE.json"
                if not f.exists():
                    msg = {"step": state.global_step, "metric": self.metric, "window_mean": m}
                    f.write_text(json.dumps(msg))
                    print("[grpo] *** G2 MODE COLLAPSE ***", msg, flush=True)
                if self.stop:
                    control.should_save = True
                    control.should_training_stop = True


def main(cfg_path):
    global JUDGE
    cfg = json.loads(Path(cfg_path).read_text(encoding="utf-8"))
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    if (out / "adapter" / "adapter_config.json").exists():
        print("[grpo] adapter exists, skipping:", out / "adapter")
        return
    from trl import GRPOConfig, GRPOTrainer
    import trl
    set_seed(cfg["seed"])

    tok = AutoTokenizer.from_pretrained(cfg["base"])
    rows = load_jsonl(cfg["train_file"])
    ds = Dataset.from_list([{k: r[k] for k in ("prompt", "gold", "task", "question", "reference", "level", "id")}
                            for r in rows])
    base = AutoModelForCausalLM.from_pretrained(cfg["base"], dtype=torch.bfloat16, attn_implementation="sdpa")
    model = PeftModel.from_pretrained(base, cfg["init_adapter"], is_trainable=True)
    model.print_trainable_parameters()

    variant = cfg["variant"]
    if variant == "judge":
        JUDGE = Judge(cfg["judge_model"], cfg["judge_max_new_tokens"], cfg["judge_batch"], cfg.get("judge_parser", "fixed"))
        main_fn = judge_correctness
    else:
        main_fn = correctness_reward
    shaping = {"ours": [adaptive_mode_reward], "std_length": [mode_length_reward], "no_length": [],
               "judge": []}[variant]                      # R2's difficulty-aware bonus is inside judge_correctness
    monitors = [strict_accuracy, think_rate, short_rate, malformed_rate, think_correct,   # weight 0: logged only
                all_correct_group]                  # share of all-correct groups (the C2f mechanism)
    balanced = bool(cfg.get("balanced"))
    if balanced:
        monitors.append(policy_think_prob)          # rollouts are forced 50/50; this is the policy's own choice
    funcs = [main_fn, format_reward, repetition_reward] + shaping + monitors
    weights = [1] * 3 + [float(cfg.get("bonus_scale", 1.0))] * len(shaping) + [0] * len(monitors)

    gcfg = dict(
        output_dir=str(out / "checkpoints"), max_steps=cfg["max_steps"],
        per_device_train_batch_size=cfg["per_device_batch"],
        gradient_accumulation_steps=cfg["grad_accum"], num_generations=cfg["num_generations"],
        max_prompt_length=cfg["max_prompt_length"], max_completion_length=cfg["max_completion_length"],
        temperature=1.0, top_p=1.0, repetition_penalty=1.0,
        learning_rate=cfg["lr"], lr_scheduler_type="constant_with_warmup",
        warmup_steps=max(1, int(0.05 * cfg["max_steps"])),   # warmup_ratio is ignored by TRL 1.14
        beta=cfg["beta"], reward_weights=[float(w) for w in weights],
        scale_rewards=cfg.get("scale_rewards", "group"),     # S6: "batch" (group std amplified the brevity bonus)
        use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=cfg["vllm_gpu_mem"],
        vllm_enable_sleep_mode=bool(cfg.get("vllm_enable_sleep_mode", False)),   # last-resort OOM fallback
        bf16=True, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        mask_truncated_completions=False, remove_unused_columns=False,
        **({"vllm_importance_sampling_correction": True,          # balanced: per-token, so only the mode token
            "vllm_importance_sampling_mode": "token_mask"} if balanced else {}),   # gets the pi/0.5 weight
        logging_steps=1, save_steps=cfg["save_steps"], save_total_limit=2,
        report_to="tensorboard", logging_dir=str(out / "tb"), seed=cfg["seed"])
    args = GRPOConfig(**common.supported_kwargs(GRPOConfig, gcfg, required=(
        "use_vllm", "num_generations", "reward_weights", "max_completion_length", "scale_rewards")
        + (("vllm_importance_sampling_correction", "vllm_importance_sampling_mode") if balanced else ())))
    (out / "resolved_grpo_config.json").write_text(json.dumps(
        {"trl_version": trl.__version__, "variant": variant, "reward_funcs": [f.__name__ for f in funcs],
         "reward_weights": weights, "bonus_scale": float(cfg.get("bonus_scale", 1.0)), "balanced": balanced,
         "args": args.to_dict()}, indent=1, default=str))

    trainer = GRPOTrainer(model=model, reward_funcs=funcs, args=args, train_dataset=ds,
                          processing_class=tok,
                          **({"rollout_func": make_balanced_rollout(tok)} if balanced else {}),
                          callbacks=[TimeLimit(cfg["max_hours"]),
                                     CollapseGuard(out, cfg["g2_after_step"], 10, cfg["stop_on_collapse"],
                                                   "policy_think_prob" if balanced else "think_rate")])
    ckpts = sorted((out / "checkpoints").glob("checkpoint-*"))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=bool(ckpts))
    finished = trainer.state.global_step >= cfg["max_steps"] or (
        cfg["stop_on_collapse"] and (out / "G2_MODE_COLLAPSE.json").exists())
    final_dir = out / ("adapter" if finished else "adapter_partial")   # partial runs stay resumable
    trainer.model.save_pretrained(final_dir)
    tok.save_pretrained(final_dir)
    (out / "trainer_log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    (out / "train_summary.json").write_text(json.dumps(
        {"steps": trainer.state.global_step}, indent=1))
    print(f"[grpo] saved {final_dir} (finished={finished}); rerun the cell to resume if not finished.")


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /content/slm/train_sft.py
# train_sft.py: Stage A. LoRA SFT on verified <think>/<short> examples (both modes kept).
# Usage: python train_sft.py config.json
# Reuses the v1 recipe that was correct (LoRA r16/a32 on all projections, prompt-masked labels,
# DataCollatorForSeq2Seq) and fixes the v1 bug that dropped every <think> example.
import json, math, os, sys
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common  # noqa: E402

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]


def main(cfg_path):
    cfg = json.loads(Path(cfg_path).read_text(encoding="utf-8"))
    out = Path(cfg["out_dir"])
    if (out / "adapter" / "adapter_config.json").exists():
        print("[sft] adapter exists, skipping:", out / "adapter")
        return
    set_seed(cfg["seed"])
    tok = AutoTokenizer.from_pretrained(cfg["base"])
    end_ids = tok("<|im_end|>\n", add_special_tokens=False)["input_ids"]

    def encode(ex):
        p = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
        t = tok(ex["target"], add_special_tokens=False)["input_ids"] + end_ids
        return {"input_ids": p + t, "attention_mask": [1] * (len(p) + len(t)),
                "labels": [-100] * len(p) + t}

    train = Dataset.from_list(load_jsonl(cfg["train_file"])).map(encode, remove_columns=["prompt", "target", "mode", "id"])
    evald = Dataset.from_list(load_jsonl(cfg["eval_file"])).map(encode, remove_columns=["prompt", "target", "mode", "id"])
    for d in (train, evald):
        too_long = sum(len(x) > cfg["max_len"] for x in d["input_ids"])
        assert too_long == 0, f"{too_long} examples exceed max_len; P1 filtering must remove them."
    print(f"[sft] train={len(train)} eval={len(evald)}")

    # The same recipe as the notebook's inline SFT (cells 9–15), used for the second base model (S10). The OOM retry
    # (run_adaptive) halves the micro-batch and doubles grad-accum, so the effective batch and the step count stay fixed.
    try:
        import flash_attn  # noqa: F401
        attn = "flash_attention_2"
    except ImportError:
        attn = "sdpa"
    model = AutoModelForCausalLM.from_pretrained(cfg["base"], dtype=torch.bfloat16, attn_implementation=attn)
    print(f"[sft] loaded {cfg['base']} (attn={attn}), pad={tok.pad_token!r}, eos={tok.eos_token!r}")
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                                             lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                             target_modules=LORA_TARGETS))
    model.print_trainable_parameters()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    max_steps = cfg.get("max_steps", -1)
    total = max_steps if max_steps > 0 else math.ceil(len(train) / (cfg["batch"] * cfg["grad_accum"])) * cfg["epochs"]
    args = TrainingArguments(**common.supported_kwargs(TrainingArguments, dict(
        output_dir=str(out / "checkpoints"), num_train_epochs=cfg["epochs"], max_steps=max_steps,
        per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=cfg["batch"],
        gradient_accumulation_steps=cfg["grad_accum"], learning_rate=cfg["lr"],
        lr_scheduler_type="cosine", warmup_steps=max(1, int(0.03 * total)),   # warmup_ratio is ignored by this stack
        max_grad_norm=1.0, bf16=True, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, optim="adamw_torch_fused",
        group_by_length=True, eval_strategy="steps", save_strategy="steps",
        eval_steps=cfg["eval_steps"], save_steps=cfg["eval_steps"], save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=5, report_to="tensorboard", logging_dir=str(out / "tb"),
        dataloader_num_workers=2, seed=cfg["seed"]),
        required=("learning_rate", "bf16", "eval_strategy")))
    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evald,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    ckpts = sorted((out / "checkpoints").glob("checkpoint-*"))
    trainer.train(resume_from_checkpoint=bool(ckpts))
    model.save_pretrained(out / "adapter")
    tok.save_pretrained(out / "adapter")
    (out / "trainer_log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    (out / "train_summary.json").write_text(json.dumps({"base": cfg["base"], "steps": trainer.state.global_step,
                                                         "train": len(train), "eval": len(evald)}, indent=1))
    print("[sft] saved", out / "adapter")


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /content/slm/backend_api.py
# backend_api.py: single-pass adaptive inference server for the v2 GRPO adapter.
# The model chooses <think> or <short> itself in ONE generation (no multi-call workflow,
# no second model rewriting the answer). Run:  MODEL_DIR=./adapter uvicorn backend_api:app
import os
from contextlib import asynccontextmanager
from threading import Lock

import torch
from fastapi import FastAPI, HTTPException
from peft import PeftModel
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

import common

BASE = os.getenv("BASE_MODEL", "Qwen/Qwen2.5-Math-1.5B-Instruct")
ADAPTER = os.getenv("MODEL_DIR", "./adapter")
STATE = {}
LOCK = Lock()


@asynccontextmanager
async def lifespan(app):
    tok = AutoTokenizer.from_pretrained(BASE)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype)
    model = PeftModel.from_pretrained(model, ADAPTER).merge_and_unload().eval()
    if torch.cuda.is_available():
        model = model.cuda()
    STATE.update(tok=tok, model=model)
    yield
    STATE.clear()


app = FastAPI(title="Adaptive reasoning SLM", lifespan=lifespan)


class Query(BaseModel):
    question: str = Field(min_length=1, max_length=6000)
    max_new_tokens: int = Field(default=2048, ge=16, le=3584)
    force_mode: str | None = Field(default=None, pattern="^(think|short)$")


@app.post("/solve")
def solve(q: Query):
    if "model" not in STATE:
        raise HTTPException(503, "model not loaded")
    tok, model = STATE["tok"], STATE["model"]
    prefill = f"<{q.force_mode}>\n" if q.force_mode else ""
    prompt = common.build_prompt(tok, q.question, "policy", prefill)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with LOCK, torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=q.max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    gen = out[0, enc["input_ids"].shape[1]:]
    text = prefill + tok.decode(gen, skip_special_tokens=True)
    mode, answer, status = common.extract_final(text, "policy")
    _, _, think = common.parse_policy_output(text)
    return {"mode": mode, "final_answer": answer, "extraction": status, "reasoning": think,
            "raw": text, "generated_tokens": int(gen.shape[0]),
            "truncated": int(gen.shape[0]) >= q.max_new_tokens}


@app.get("/health")
def health():
    return {"loaded": "model" in STATE, "base": BASE, "adapter": ADAPTER}

In [ ]:
# 5.4) Shared helpers, adaptive memory management + self-checks (must pass before any GPU stage)
sys.path.insert(0, str(CODE))
import importlib, common
importlib.reload(common)
from transformers import AutoTokenizer
assert common.self_check(), "scorer/reward self-check failed"
TOK = AutoTokenizer.from_pretrained(CFG["policy_base"])
# The SFT, GRPO and eval prompt strings must be identical (same builder, same tokens).
_q = "What is 2+3?"
assert common.build_prompt(TOK, _q, "policy") == common.build_prompt(TOK, _q, "policy")
print(common.build_prompt(TOK, _q, "policy"))
print("Self-checks passed ✓  (strict scorer, modes, rewards, statistics)")

def read_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []

def _json_default(o):
    """NumPy scalars (int64, bool_, float32) -> native Python types; anything else -> str.
    (default=str alone turned token counts into strings; found in the pilot.)"""
    return o.item() if hasattr(o, "item") else str(o)

def write_jsonl(p, rows):
    with Path(p).open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=_json_default) + "\n")

# ---------------- adaptive memory management (Colab: GPU can be A100-40GB or -80GB; RAM is finite) ----------
from collections import deque
OOM_PATTERNS = ("cuda out of memory", "outofmemoryerror", "out of memory", "no available memory for the cache blocks",
                "insufficient memory", "less than desired gpu memory", "cublas_status_alloc_failed",
                "not enough memory", "free memory on device")
KERNEL_GPU_VARS = ("trainer", "model", "dpo_trainer", "sft_merged", "merged", "base", "_m")

def gpu_free_gib():
    free, total = torch.cuda.mem_get_info()
    return free / 2**30, total / 2**30

def mem_report(tag=""):
    f, t = gpu_free_gib()
    try:
        import psutil
        vm = psutil.virtual_memory(); ram = f" | RAM free {vm.available/2**30:.0f}/{vm.total/2**30:.0f} GiB"
    except Exception:
        ram = ""
    print(f"[mem]{(' ' + tag) if tag else ''} GPU free {f:.1f}/{t:.1f} GiB{ram}")
    return f, t

def free_gpu(tag="", drop_kernel_models=True):
    """Adaptive cleanup before a heavy stage: drop any model the kernel still holds (e.g. after a crashed
    cell), then release cached CUDA blocks so the next process gets the whole GPU."""
    if drop_kernel_models:
        g = globals()
        for n in KERNEL_GPU_VARS:
            obj = g.get(n)
            if obj is not None and (hasattr(obj, "parameters") or n in ("trainer", "dpo_trainer")):
                del g[n]
        # generic sweep: any other notebook variable that is a model living on the GPU, and IPython's
        # output history (_, __, Out[n]) which can silently keep a displayed model alive
        Module = getattr(getattr(torch, "nn", None), "Module", None)
        def on_gpu(o):
            try:
                return Module is not None and isinstance(o, Module) and next(o.parameters()).is_cuda
            except Exception:
                return False
        for n in [k for k, v in list(g.items()) if not k.startswith("__") and on_gpu(v)]:
            del g[n]
        for n in ("_", "__", "___"):
            if on_gpu(g.get(n)):
                g[n] = None
        if isinstance(g.get("Out"), dict):
            for k in [k for k, v in g["Out"].items() if on_gpu(v)]:
                del g["Out"][k]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    return mem_report(tag)

def is_oom(log):
    low = log.lower()
    return any(p in low for p in OOM_PATTERNS)

def run_script(script, cfg, name, check=True):
    """Run a training/generation script in a fresh process (clean GPU memory), streaming logs.
    Returns (returncode, log tail)."""
    cfg_path = EVAL / "jobs" / f"{name}.json"
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, default=_json_default))
    env = dict(os.environ, TQDM_MININTERVAL="20", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    env.pop("PYTORCH_CUDA_ALLOC_CONF", None)   # vLLM manages its own memory pool (conflicts with expandable segments)
    proc = subprocess.Popen([sys.executable, str(CODE / script), str(cfg_path)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=str(CODE), env=env)
    tail = deque(maxlen=400)
    for line in proc.stdout:
        line = line.split(chr(13))[-1]
        tail.append(line)
        print(line[-400:], end="")   # keep only the latest progress-bar state
    rc = proc.wait()
    log = "".join(tail)
    if check and rc != 0:
        raise RuntimeError(f"{script} failed (see log above)")
    return rc, log

def run_adaptive(script, cfg, name, fallbacks=()):
    """Run a GPU stage; on CUDA OOM free memory and retry with the next (smaller) override.
    Checkpoints make retries resume, so nothing finished is lost. Returns the config that worked."""
    tries = [{}] + list(fallbacks)
    for i, over in enumerate(tries):
        cfg_try = {**cfg, **over}
        free_gpu(f"before {name} (attempt {i+1}/{len(tries)})")
        rc, log = run_script(script, cfg_try, name, check=False)
        if rc == 0:
            if over:
                print(f"[mem] {name} succeeded with reduced memory settings {over}")
            return cfg_try
        if not is_oom(log):
            raise RuntimeError(f"{script} failed for a non-memory reason (see log above)")
        if i + 1 < len(tries):
            print(f"[mem] OUT OF MEMORY in {name} → retrying with {tries[i+1]}")
    raise RuntimeError(f"{name}: still out of memory after {len(tries)} attempts; restart the runtime "
                       "(Runtime → Restart session) and Run all — finished stages are cached.")

def generate(name, model, requests, lora=None, max_model_len=None, gpu_mem=0.85):
    """Batched vLLM generation via gen_worker.py; cached on Drive by job name.
    vLLM's memory share is sized from the memory that is actually free, and retried smaller on OOM."""
    out = EVAL / "jobs" / f"{name}.jsonl"
    if not out.with_suffix(".done.json").exists():
        f, t = free_gpu(f"before {name}")
        util = round(min(gpu_mem, max(0.25, (f - 2.0) / t)), 2)
        job = {"model": model, "lora": str(lora) if lora else None,
               "max_model_len": max_model_len or CFG["context"], "gpu_mem": util, "max_num_seqs": 256,
               "enforce_eager": CFG["vllm_enforce_eager"], "out": str(out), "requests": requests}
        run_adaptive("gen_worker.py", job, name, fallbacks=(
            {"gpu_mem": round(util * 0.8, 2), "max_num_seqs": 128},
            {"gpu_mem": round(util * 0.65, 2), "max_num_seqs": 48, "enforce_eager": True}))
    meta = json.loads(out.with_suffix(".done.json").read_text())
    return read_jsonl(out), meta

## 5.5) Benchmarks frozen *before* any training, plus decontamination
✅ FIX: v1 loaded benchmarks at the very end and never checked for overlap with the training data. Here the benchmarks are frozen first, with dataset commit, question hash and gold answer:
* **GSM8K** test (1319)
* **MATH-500** (500, levels 1–5)
* **AIME-2026** (30, released after the base model)

All training text is then checked against them with a 13-gram index. The pilot uses fixed small subsets.

In [ ]:
# 5.5) Freeze benchmarks (dataset commit + question hash + gold answer)
BENCH_FILE = DATA / "benchmarks.jsonl"
BENCHES = ("gsm8k", "math500", "aime2026")
if not BENCH_FILE.exists():
    from datasets import load_dataset
    def pinned(repo, split, config=None):
        sha = HF.dataset_info(repo).sha
        return load_dataset(repo, config, split=split, revision=sha), sha
    rows = []
    ds, sha = pinned("openai/gsm8k", "test", "main")
    for i, r in enumerate(ds):
        rows.append(dict(bench="gsm8k", idx=i, question=r["question"], gold=r["answer"].rsplit("####", 1)[1].strip(),
                         level=None, subject=None, repo="openai/gsm8k", revision=sha))
    ds, sha = pinned("HuggingFaceH4/MATH-500", "test")
    for i, r in enumerate(ds):
        rows.append(dict(bench="math500", idx=i, question=r["problem"], gold=str(r["answer"]).strip(),
                         level=int(r["level"]), subject=r["subject"], repo="HuggingFaceH4/MATH-500", revision=sha))
    ds, sha = pinned("MathArena/aime_2026", "train")
    for i, r in enumerate(ds):
        rows.append(dict(bench="aime2026", idx=i, question=r["problem"], gold=str(r["answer"]).strip(),
                         level=None, subject=None, repo="MathArena/aime_2026", revision=sha))
    for r in rows:
        r["qhash"] = common.text_hash(r["question"]); r["pid"] = f'{r["bench"]}:{r["idx"]}:{r["qhash"][:12]}'
        if r["bench"] in ("gsm8k", "aime2026"):
            assert common.rational(r["gold"]) is not None, r
        if r["bench"] == "math500":
            common.symbolic_gold(r["gold"])          # every gold must parse, or we stop
    write_jsonl(BENCH_FILE, rows)
BENCH = pd.DataFrame(read_jsonl(BENCH_FILE))
BENCH = BENCH[BENCH.bench.isin(BENCHES)].reset_index(drop=True)   # older freezes also had ARC-Challenge
if CFG["pilot_bench"]:   # pilot: fixed small subsets (MATH-500 stratified over levels)
    parts = []
    for b, n in CFG["pilot_bench"].items():
        x = BENCH[BENCH.bench == b]
        if b == "math500":
            x = x.groupby("level").sample(n // 5, random_state=CFG["seed"])
        else:
            x = x.sample(min(n, len(x)), random_state=CFG["seed"])
        parts.append(x)
    BENCH = pd.concat(parts, ignore_index=True)
print(BENCH.groupby("bench").size())

In [ ]:
# 5.6) 13-gram decontamination index over all benchmark questions
def ngrams(text, n=CFG["ngram"]):
    w = re.findall(r"\w+", str(text).lower())
    return {" ".join(w[i:i+n]) for i in range(len(w) - n + 1)}
BENCH_NGRAMS = set().union(*[ngrams(q) for q in BENCH.question])
def contaminated(text):
    return bool(ngrams(text) & BENCH_NGRAMS)
print(f"{len(BENCH_NGRAMS):,} benchmark {CFG['ngram']}-grams indexed")

## 5.7) Evaluation harness + baselines (✅ NEW: v1 never measured the untouched model)
* **Rows:** the untouched **base** model (official prompt), the **DeepSeek-R1-Distill-1.5B** reference, and your **v1 GRPO adapter** (if uploaded), all on the same harness.
* **Settings:** greedy decoding, no repetition penalty, the policy's own answer, the strict scorer.
* **Gate G0:** base within about 3 points of the published 84.8 GSM8K / 75.8 MATH (full scale only).
* **Difficulty:** the base model is also sampled 4× per MATH problem, giving each problem an empirical difficulty.

In [ ]:
# 5.7) Model registry and evaluation functions
MODELS = {
    "base":       dict(model=CFG["policy_base"], lora=None, fmt="plain", style="official",
                       label="Base (Qwen2.5-Math-1.5B-Inst)"),
    "r1distill":  dict(model=CFG["fallback_base"], lora=None, fmt="plain", style="r1", max_model_len=12288,
                       max_tokens=CFG["ref_max_tokens"], label="R1-Distill-1.5B (reference)"),
    "v1_grpo":    dict(model=CFG["policy_base"], lora=V1_BUNDLE / "qwen2.5-rl-reasoning-adapter", fmt="policy",
                       style="v1", label="v1 GRPO (original)"),
    "sft_v2":     dict(model=CFG["policy_base"], lora=RUNS / "sft_v2" / "adapter", fmt="policy", style="policy",
                       label="SFT (v2)"),
    "sft_long":   dict(model=CFG["policy_base"], lora=RUNS / "sft_long" / "adapter", fmt="policy", style="policy",
                       label="SFT, long think traces (ablation)"),   # S2: think targets ≤3400 tokens; looped
    "grpo_ours":  dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours" / "adapter", fmt="policy",
                       style="policy", label="GRPO ours (R1)"),
    "grpo_judge": dict(model=CFG["policy_base"], lora=RUNS / "grpo_judge" / "adapter", fmt="policy",
                       style="policy", label="GRPO judge reward (R2)"),
    "grpo_stdlen": dict(model=CFG["policy_base"], lora=RUNS / "grpo_stdlen" / "adapter", fmt="policy",
                        style="policy", label="GRPO std. length (R3)"),
    "dpo_v2":     dict(model=str(RUNS / "dpo_v2" / "merged"), lora=None, fmt="policy", style="policy",
                       label="DPO (v2)"),
    "grpo_ours_s2": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_s2" / "adapter", fmt="policy",
                         style="policy", label="GRPO ours (R1, seed 2)"),
    "grpo_ours_s3": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_s3" / "adapter", fmt="policy",
                         style="policy", label="GRPO ours (R1, seed 3)"),   # S9
    "grpo_balanced": dict(model=CFG["policy_base"], lora=RUNS / "grpo_balanced" / "adapter", fmt="policy",
                          style="policy", label="GRPO ours, balanced exploration (R1b)"),
    # R1 at step 60 = the adapter gate G2 stopped in S3 (kept by the P5.3 cell): does training to 200 steps help or hurt?
    "grpo_ours_step60": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_gstd" / "adapter_g2stop_step60",
                             fmt="policy", style="policy", label="GRPO ours (R1) at step 60"),
    # R1 and R3 trained with group-std advantages (S3–S4): their CoT collapsed after step ≈130. Archived in S6 (P5.1)
    "grpo_ours_gstd": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_gstd" / "adapter", fmt="policy",
                           style="policy", label="GRPO ours (R1), group-std advantages"),
    "grpo_stdlen_gstd": dict(model=CFG["policy_base"], lora=RUNS / "grpo_stdlen_gstd" / "adapter", fmt="policy",
                             style="policy", label="GRPO std. length (R3), group-std advantages"),
    # S8 mechanism controls of that collapse (same start, prompts, seed and 200 steps as R1; one factor changes)
    "grpo_nobonus_gstd": dict(model=CFG["policy_base"], lora=RUNS / "grpo_nobonus_gstd" / "adapter", fmt="policy",
                              style="policy", label="Control C1: group-std, no brevity bonus"),
    "grpo_ours_nostd": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_nostd" / "adapter", fmt="policy",
                            style="policy", label="Control C2: R1 bonus, no std scaling (Dr. GRPO-style)"),
    "grpo_ours_gstd_b01": dict(model=CFG["policy_base"], lora=RUNS / "grpo_ours_gstd_b01" / "adapter", fmt="policy",
                               style="policy", label="Control C3: group-std, R1 bonus ×0.1"),
    # Second base model (S10–S11): the same recipe on a general instruct model of the same
    # size and tokenizer (not math-specialised). Same SFT data, prompts, seed, rewards and steps as the main model.
    "m2_base":    dict(model=CFG["policy_base2"], lora=None, fmt="plain", style="official",
                       label="2nd model: base (Qwen2.5-1.5B-Inst)"),
    "m2_sft":     dict(model=CFG["policy_base2"], lora=RUNS / "m2_sft" / "adapter", fmt="policy", style="policy",
                       label="2nd model: SFT"),
    "m2_grpo":    dict(model=CFG["policy_base2"], lora=RUNS / "m2_grpo" / "adapter", fmt="policy", style="policy",
                       label="2nd model: GRPO R1"),
    "m2_grpo_gstd": dict(model=CFG["policy_base2"], lora=RUNS / "m2_grpo_gstd" / "adapter", fmt="policy",
                         style="policy", label="2nd model: GRPO R1, group-std advantages"),
    # prior-work adaptive-thinking models (both R1-Distill-1.5B based), evaluated under OUR harness and scorer
    "thinkless":  dict(model="Vinnnf/Thinkless-1.5B-RL-DeepScaleR", lora=None, fmt="hybrid", style="thinkless",
                       max_model_len=12288, max_tokens=CFG["ref_max_tokens"], label="Thinkless-1.5B (Fang et al.)"),
    "adaptthink": dict(model="THU-KEG/AdaptThink-1.5B-delta0.05", lora=None, fmt="hybrid", style="r1",
                       max_model_len=12288, max_tokens=CFG["ref_max_tokens"], label="AdaptThink-1.5B (Zhang et al.)"),
}
EXTERNAL = ["thinkless", "adaptthink"]

def aime_k(key):
    """AIME samples per model: 16 for the models the paper's claims rest on, 4 for the rest (budget)."""
    return CFG["aime_samples_main"] if key in CFG["aime_main_models"] else CFG["aime_samples"]
V1_ANSWER_INSTRUCTION = ("\n\nPut the final mathematical answer in \\boxed{...} inside whichever response tag "
                         "you choose, then close the tag.")          # as in the v1 strict-eval notebook
PREFILL = {"single": "", "forced_think": "<think>\n", "forced_short": "<short>\n"}
_TOKS = {}

def tok_for(spec):
    name = spec["model"]          # LoRA models: their own base (the second model since S10); DPO: its merged folder
    if name not in _TOKS:
        _TOKS[name] = AutoTokenizer.from_pretrained(name)
    return _TOKS[name]

def model_prompt(spec, question, prefill=""):
    q = question + (V1_ANSWER_INSTRUCTION if spec["style"] == "v1" else "")
    return common.build_prompt(tok_for(spec), q, spec["style"], prefill)

def eval_model(key, conditions=("single",), difficulty=False):
    """One vLLM job per model covering every missing condition (one model load). Cached per condition."""
    if isinstance(conditions, str):
        conditions = (conditions,)
    spec = MODELS[key]
    files = {c: EVAL / "preds" / f"{key}__{c}.jsonl" for c in conditions}
    missing = [c for c in conditions if not files[c].exists()]
    if missing:
        if spec["lora"] is not None:
            assert (Path(spec["lora"]) / "adapter_config.json").exists(), f"missing adapter for {key}: {spec['lora']}"
        max_tokens = spec.get("max_tokens", CFG["eval_max_tokens"])
        reqs = []
        for c in missing:
            for r in BENCH.itertuples():
                p = model_prompt(spec, r.question, PREFILL[c])
                reqs.append(dict(id=f"{c}|{r.pid}|greedy", prompt=p, n=1, temperature=0.0, top_p=1.0, max_tokens=max_tokens))
                if r.bench == "aime2026" and c == "single":
                    reqs.append(dict(id=f"{c}|{r.pid}|sampled", prompt=p, n=aime_k(key),
                                     temperature=CFG["aime_temperature"], top_p=0.95, max_tokens=max_tokens, seed=CFG["seed"]))
                if difficulty and r.bench == "math500" and c == "single":
                    reqs.append(dict(id=f"{c}|{r.pid}|difficulty", prompt=p, n=CFG["difficulty_samples"],
                                     temperature=0.6, top_p=0.95, max_tokens=max_tokens, seed=CFG["seed"] + 1))
        outs, meta = generate(f"eval_{key}_{'+'.join(missing)}", spec["model"], reqs, lora=spec["lora"],
                              max_model_len=spec.get("max_model_len", CFG["context"]))
        by_pid = BENCH.set_index("pid")
        rows = defaultdict(list)
        for o in outs:
            c, pid, kind = o["id"].split("|")
            b = by_pid.loc[pid]
            text = PREFILL[c] + o["text"]
            s = common.score_text(text, b.gold, b.bench, spec["fmt"])
            mode, region, think = common.classify(text, spec["fmt"])
            rows[c].append(dict(model=key, condition=c, kind=kind, sample=o["sample"], pid=pid, bench=b.bench,
                                level=b.level, subject=b.subject, correct=bool(s["correct"]), status=s["status"],
                                final_answer=s["final_answer"], mode=mode, n_tokens=o["n_tokens"],
                                truncated=o["finish_reason"] == "length", context_capped=o["context_capped"],
                                format_ok=common.format_ok(text) if spec["fmt"] == "policy" else None,
                                repetition=common.has_repetition_loop(text),
                                self_checks=common.self_check_markers(think or text),
                                second_scorer=(common.second_scorer(text, b.gold, spec["fmt"])
                                               if b.bench in ("gsm8k", "math500") and kind == "greedy" else None),
                                sec_per_problem=meta["generate_seconds"] / max(1, len(reqs)), text=text))
        for c in missing:
            write_jsonl(files[c], rows[c])
    out = pd.concat([pd.DataFrame(read_jsonl(files[c])) for c in conditions], ignore_index=True)
    out["repetition"] = out["text"].map(common.has_repetition_loop)   # current loop detector, also for cached rows
    out["chose_think"] = [common.chose_think(m, t, spec["style"] == "r1") for m, t in zip(out["mode"], out["text"])]
    return out

def acc_table(df, kind="greedy"):
    g = df[df.kind == kind].groupby(["model", "condition", "bench"])
    rows = []
    for (m, c, b), x in g:
        k, n = int(x.correct.sum()), len(x); lo, hi = common.wilson(k, n)
        rows.append(dict(model=m, condition=c, bench=b, n=n, acc=k / n, ci_low=lo, ci_high=hi,
                         mean_tokens=x.n_tokens.mean(),
                         think_rate=(x.chose_think if "chose_think" in x else x["mode"] == "think").mean(),
                         short_rate=(x["mode"] == "short").mean(),
                         format_ok=x.format_ok.mean() if x.format_ok.notna().any() else np.nan,
                         repetition=x.repetition.mean(), truncated=x.truncated.mean(),
                         sec_per_problem=x.sec_per_problem.mean()))
    return pd.DataFrame(rows)

def show(df, cols=None, digits=3):
    display(df[cols] if cols else df.round(digits))

In [ ]:
# 5.8) Baselines: base (official CoT prompt + difficulty sampling), R1-Distill reference, v1 GRPO
# and the prior-work adaptive-thinking models Thinkless + AdaptThink (published checkpoints, our harness)
if STAGES["P2_baselines"]:
    base_df = eval_model("base", "single", difficulty=True)
    ref_df = eval_model("r1distill", "single")
    for key in EXTERNAL:
        try:
            eval_model(key, "single")
        except Exception as e:      # a missing/changed checkpoint is reported, never replaced by a number
            print(f"⚠ {key} could not be evaluated ({type(e).__name__}: {str(e)[:200]}). It is left out of the tables.")
    if (V1_BUNDLE / "qwen2.5-rl-reasoning-adapter" / "adapter_config.json").exists():
        v1_df = eval_model("v1_grpo", "single")
    else:
        print("v1 bundle not on Drive. Skipping the v1 baseline row (copy full_paper_experiment_bundle to MyDrive to include it).")
    mark_stage("P2_baselines")
base_df = eval_model("base", "single", difficulty=True) if (EVAL / "preds" / "base__single.jsonl").exists() else None
if base_df is not None:
    t = acc_table(base_df); show(t)
    published = {"gsm8k": 0.848, "math500": 0.758}
    for b, pub in (published.items() if not PILOT else []):   # G0 is meaningful only at full scale
        a = t[t.bench == b].acc.iloc[0]
        print(f"G0 {b}: harness {a:.3f} vs published {pub:.3f} → Δ {a-pub:+.3f} {'✓' if abs(a-pub) <= 0.03 else '⚠ CHECK HARNESS'}")

## 6–8) Load Hybrid-OpenThoughts2 → split the two modes → train/eval split
✅ FIX (the bug that removed thinking): v1 kept only rows containing `<short>`, which removed every `<think>` example. Here we keep instructions that have **both** a `<think>` and a `<short>` response, where:
* both final `\boxed{}` answers **agree** (cross-mode verification, since the dataset has no answer key),
* tags are closed,
* the example fits in 4096 tokens (**dropped, never truncated**),
* it is not contaminated.

The train/eval split is **by instruction**, so the two modes of one question never cross sides. The cell prints three raw examples so the real format is visible.

✅ S2b (2026-10-09): think responses are capped at **1,500 tokens** (`sft_max_target_tokens`), which fits GRPO's 2,048-token completion cap. The first SFT allowed 3,400 (median 1,787), and its think mode looped on 41% of MATH problems (G1 and D1 failed). Small models learn short reasoning traces better than long ones (Li et al. 2025; Luo et al. 2025). The first SFT's data, adapter and predictions are archived once as the **`sft_long`** ablation, then the data, SFT and SFT evaluation are rebuilt.

In [ ]:
# 6–8 pre) One-time archive (S2b): when the saved SFT data used another think-length cap, keep that SFT as the
# "sft_long" ablation (data, adapter, predictions, eval cache), so 6–8 and 9–15.1 rebuild from scratch.
def archive_long_sft():
    stats_f = DATA / "sft_stats.json"
    if PILOT or not stats_f.exists():
        return
    cap = json.loads(stats_f.read_text()).get("max_target_tokens", 3400)
    if cap == CFG["sft_max_target_tokens"]:
        return
    if not STAGES["P1_data"]:
        print(f"⚠ the SFT data on Drive allows think targets ≤{cap} tokens, CFG says ≤{CFG['sft_max_target_tokens']}: "
              "run SESSION = 'S2b' to rebuild it."); return
    arch = DATA / "sft_long"
    assert not arch.exists() and not (RUNS / "sft_long").exists(), "an sft_long archive already exists; archive by hand"
    arch.mkdir()
    for f in ("sft_train.jsonl", "sft_eval.jsonl", "sft_pairs_full.jsonl", "sft_stats.json"):
        if (DATA / f).exists():
            shutil.move(str(DATA / f), str(arch / f))
    if (RUNS / "sft_v2").exists():
        shutil.move(str(RUNS / "sft_v2"), str(RUNS / "sft_long"))
    for p in sorted((EVAL / "preds").glob("sft_v2__*.jsonl")):
        rows = read_jsonl(p)
        for r in rows:
            r["model"] = "sft_long"
        write_jsonl(EVAL / "preds" / p.name.replace("sft_v2__", "sft_long__"), rows); p.unlink()
    (EVAL / "jobs" / "sft_long").mkdir(exist_ok=True)
    for p in sorted((EVAL / "jobs").glob("eval_sft_v2_*")):   # else the new SFT eval would reuse the old generations
        shutil.move(str(p), str(EVAL / "jobs" / "sft_long" / p.name))
    if (FIG / "fig0_sft_loss.png").exists():
        shutil.move(str(FIG / "fig0_sft_loss.png"), str(FIG / "fig0_sft_loss_long.png"))
    (arch / "ARCHIVED.json").write_text(json.dumps({"utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                                                    "max_target_tokens": cap, "reason": "think mode looped; G1/D1 failed (S2)"}))
    print(f"archived the first SFT (think targets ≤{cap} tokens) as 'sft_long'; rebuilding with ≤{CFG['sft_max_target_tokens']}")
archive_long_sft()

In [ ]:
# 6–8) Verified dual-mode SFT data: both modes, cross-mode verified, fits the context, decontaminated
SFT_TRAIN, SFT_EVAL, SFT_STATS = DATA / "sft_train.jsonl", DATA / "sft_eval.jsonl", DATA / "sft_stats.json"
if STAGES["P1_data"] and not SFT_TRAIN.exists():
    from datasets import load_dataset
    if CFG["sft_source_rows"]:   # pilot: stream only the first N rows (≈1.5 GB instead of 9 GB)
        from itertools import islice
        from datasets import Dataset as _DS
        _stream = load_dataset(CFG["sft_source"], split="train", streaming=True)
        def _gen():
            for r in islice(_stream, CFG["sft_source_rows"]):
                yield {"instruction": r["instruction"], "output": r["output"]}
        raw = _DS.from_generator(_gen)
    else:
        raw = load_dataset(CFG["sft_source"], split="train")   # ≈9 GB, memory-mapped
    sha = HF.dataset_info(CFG["sft_source"]).sha
    print("rows:", len(raw))
    for k in range(3):   # show the raw format once (mode tags, where the boxed answer sits)
        o = str(raw[k]["output"]); print(f"--- raw example {k}: head={o[:120]!r} ... tail={o[-160:]!r}")
    # Pass 1: index rows by instruction hash and mode (no full-text materialisation)
    by_hash = defaultdict(lambda: {"think": [], "short": []})
    for start in range(0, len(raw), 200_000):
        chunk = raw[start:start + 200_000]
        for j, (ins, out) in enumerate(zip(chunk["instruction"], chunk["output"])):
            head = str(out).lstrip()[:8].lower()
            mode = "think" if head.startswith("<think>") else "short" if head.startswith("<short>") else None
            if mode:
                by_hash[common.text_hash(ins)][mode].append(start + j)
    paired = [h for h, v in by_hash.items() if len(v["think"]) == 1 and len(v["short"]) == 1]
    print(f"instructions with exactly one think + one short: {len(paired):,}")
    rng = random.Random(CFG["seed"]); rng.shuffle(paired)

    def normalise(text):
        t = str(text).strip()
        for a, b in (("<THINK>", "<think>"), ("</THINK>", "</think>"), ("<SHORT>", "<short>"), ("</SHORT>", "</short>")):
            t = t.replace(a, b)
        if t.startswith("<short>") and not t.rstrip().endswith("</short>"):
            t = t.rstrip() + "</short>"     # the dataset never closes <short> (found in the pilot)
        return t

    # RL/DPO prompts already built (S2b rebuild) stay fixed, so the new SFT questions must avoid them
    rl_hashes = {common.text_hash(r["question"]) for f in ("rl_train.jsonl", "dpo_prompts.jsonl")
                 if (DATA / f).exists() for r in read_jsonl(DATA / f)}
    stats = Counter(); kept = []
    for h in paired:
        if len(kept) >= CFG["sft_pairs_target"]:
            break
        if h in rl_hashes:
            stats["overlaps_rl_prompts"] += 1; continue
        v = by_hash[h]
        think_row, short_row = raw[int(v["think"][0])], raw[int(v["short"][0])]
        q = str(think_row["instruction"]).strip()
        t_out, s_out = normalise(think_row["output"]), normalise(short_row["output"])
        t_mode, t_ans, _ = common.extract_final(t_out, "policy")
        s_mode, s_ans, _ = common.extract_final(s_out, "policy")
        if t_mode != "think" or s_mode != "short":
            stats["bad_format"] += 1; continue
        if t_ans is None or s_ans is None:
            stats["no_boxed_answer"] += 1; continue           # also removes code / open-ended rows
        prompt = common.build_prompt(TOK, q, "policy")
        lt = len(TOK(t_out, add_special_tokens=False)["input_ids"])
        lp = len(TOK(prompt, add_special_tokens=False)["input_ids"])
        if lt > CFG["sft_max_target_tokens"] or lp + lt + 2 > CFG["sft"]["max_len"]:
            stats["too_long"] += 1; continue                   # dropped, never truncated (checked before the slower verifier)
        agree = common.score_text(f"<short>\\boxed{{{t_ans}}}</short>", s_ans, "math500")
        if not agree["correct"]:                             # cross-mode verification
            stats["verifier_timeout_or_error" if agree["status"] == "verifier_error" else "modes_disagree"] += 1
            continue
        if contaminated(q):
            stats["contaminated"] += 1; continue
        kept.append(dict(qhash=h, question=q, prompt=prompt, think=t_out, short=s_out, think_tokens=lt))
    stats["kept_pairs"] = len(kept)
    assert len(kept) >= min(500, CFG["sft_pairs_target"] // 2), (f"Only {len(kept)} verified pairs (stats={dict(stats)}). The raw format differs from "
                              "what extract_final expects; share this output before training.")
    # Split by instruction (both modes of a question go to the same side)
    n_eval = max(50, int(len(kept) * CFG["sft_eval_frac"]))
    rows_eval, rows_train = kept[:n_eval], kept[n_eval:]
    def explode(rows):
        out = []
        for r in rows:
            out.append(dict(id=r["qhash"][:16] + ":think", prompt=r["prompt"], target=r["think"], mode="think"))
            out.append(dict(id=r["qhash"][:16] + ":short", prompt=r["prompt"], target=r["short"], mode="short"))
        random.Random(CFG["seed"]).shuffle(out); return out
    write_jsonl(SFT_TRAIN, explode(rows_train)); write_jsonl(SFT_EVAL, explode(rows_eval))
    write_jsonl(DATA / "sft_pairs_full.jsonl", kept)
    SFT_STATS.write_text(json.dumps({"source": CFG["sft_source"], "revision": sha, "rows": len(raw),
                                     "paired_instructions": len(paired), **stats,
                                     "max_target_tokens": CFG["sft_max_target_tokens"],
                                     "think_tokens_mean": float(np.mean([r["think_tokens"] for r in kept])),
                                     "think_tokens_median": float(np.median([r["think_tokens"] for r in kept]))}, indent=1))
    del raw, by_hash; gc.collect()
print(json.loads(SFT_STATS.read_text()) if SFT_STATS.exists() else "SFT data not built yet")

## 9–15) Stage A: LoRA SFT (same recipe as v1, now with both modes)
v1's correct parts are kept:
* bf16 model, LoRA r=16 / α=32 on all projections
* prompt-masked labels with `DataCollatorForSeq2Seq`
* gradient checkpointing
* `adamw_torch_fused`, cosine schedule, early stopping on eval loss

✅ FIX: Qwen's own pad token is kept (v1 overwrote it with EOS). Epochs and learning rate are tuned for the larger, both-mode data. Training resumes from checkpoints on Drive.

In [ ]:
# 9) Load Qwen2.5-Math-1.5B-Instruct in bf16 (skipped once the SFT adapter exists)
SFT_DIR = RUNS / "sft_v2"
RUN_SFT = STAGES["P4_sft"] and not (SFT_DIR / "adapter" / "adapter_config.json").exists()
if RUN_SFT:
    from transformers import AutoModelForCausalLM
    try:
        import flash_attn  # noqa: F401
        _attn = "flash_attention_2"
    except ImportError:
        _attn = "sdpa"
    tokenizer = AutoTokenizer.from_pretrained(CFG["policy_base"])
    model = AutoModelForCausalLM.from_pretrained(CFG["policy_base"], dtype=torch.bfloat16,
                                                 attn_implementation=_attn).cuda()
    print(f"loaded {CFG['policy_base']} (attn={_attn}), pad={tokenizer.pad_token!r}, eos={tokenizer.eos_token!r}")
else:
    print("SFT adapter already exists (or P4_sft is off): skipping 9)–15).")

In [ ]:
# 10) LoRA setup (v1 recipe: r=16, alpha=32, all attention + MLP projections)
if RUN_SFT:
    from peft import LoraConfig, get_peft_model
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=CFG["sft"]["lora_r"], lora_alpha=CFG["sft"]["lora_alpha"], lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

In [ ]:
# 11) Format: prompts were already built by common.build_prompt in 6–8 (same builder as GRPO + eval)
if RUN_SFT:
    sft_train_rows, sft_eval_rows = read_jsonl(SFT_TRAIN), read_jsonl(SFT_EVAL)
    print("train:", Counter(r["mode"] for r in sft_train_rows), "| eval:", Counter(r["mode"] for r in sft_eval_rows))
    ex = next(r for r in sft_train_rows if r["mode"] == "think")
    print(ex["prompt"] + ex["target"][:400] + " …")

In [ ]:
# 12) Tokenization: loss only on the assistant response (prompt masked with -100), dynamic padding
if RUN_SFT:
    from datasets import Dataset
    from transformers import DataCollatorForSeq2Seq
    END_IDS = tokenizer("<|im_end|>\n", add_special_tokens=False)["input_ids"]
    def encode(ex):
        p = tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"]
        t = tokenizer(ex["target"], add_special_tokens=False)["input_ids"] + END_IDS
        return {"input_ids": p + t, "attention_mask": [1] * (len(p) + len(t)), "labels": [-100] * len(p) + t}
    cols = ["id", "prompt", "target", "mode"]
    train_ds = Dataset.from_list(sft_train_rows).map(encode, remove_columns=cols)
    eval_ds = Dataset.from_list(sft_eval_rows).map(encode, remove_columns=cols)
    longest = max(max((len(x) for x in train_ds["input_ids"]), default=0),
                  max((len(x) for x in eval_ds["input_ids"]), default=0))
    assert longest <= CFG["sft"]["max_len"], "6–8) must drop over-long rows; nothing is truncated here"
    collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
    print(f"train={len(train_ds)} eval={len(eval_ds)} longest={longest} tokens")

In [ ]:
# 13) Gradient checkpointing (non-reentrant, as in v1)
if RUN_SFT:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

In [ ]:
# 14) TrainingArguments + Trainer (early stopping on eval loss; checkpoints on Drive for resume)
if RUN_SFT:
    from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
    sc = CFG["sft"]
    def make_sft_trainer(batch, accum):
        """Rebuilt with a smaller micro-batch after an OOM; batch*accum (the effective batch) never changes."""
        total = sc["max_steps"] if sc["max_steps"] > 0 else math.ceil(len(train_ds) / (batch * accum)) * sc["epochs"]
        args = TrainingArguments(**common.supported_kwargs(TrainingArguments, dict(
            output_dir=str(SFT_DIR / "checkpoints"), num_train_epochs=sc["epochs"], max_steps=sc["max_steps"],
            per_device_train_batch_size=batch, per_device_eval_batch_size=batch,
            gradient_accumulation_steps=accum, learning_rate=sc["lr"], lr_scheduler_type="cosine",
            warmup_steps=max(1, int(0.03 * total)), max_grad_norm=1.0, bf16=True, gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}, optim="adamw_torch_fused", group_by_length=True,
            eval_strategy="steps", save_strategy="steps", eval_steps=sc["eval_steps"], save_steps=sc["eval_steps"],
            save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
            logging_steps=5, report_to="tensorboard", logging_dir=str(SFT_DIR / "tb"), dataloader_num_workers=2,
            seed=CFG["seed"]), required=("learning_rate", "bf16", "eval_strategy")))
        return Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collator,
                       callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer = make_sft_trainer(sc["batch"], sc["grad_accum"])
    mem_report("before SFT")

In [ ]:
# 15) Train (resumes after a disconnect), save the SFT adapter, free the GPU
if RUN_SFT:
    _batch, _accum = sc["batch"], sc["grad_accum"]
    while True:   # adaptive: on CUDA OOM halve the micro-batch, double grad-accum, resume from the last checkpoint
        resume = bool(sorted((SFT_DIR / "checkpoints").glob("checkpoint-*")))
        try:
            out = trainer.train(resume_from_checkpoint=resume)
            break
        except torch.cuda.OutOfMemoryError:
            if _batch == 1:
                raise
            del trainer; gc.collect(); torch.cuda.empty_cache()
            _batch, _accum = _batch // 2, _accum * 2
            print(f"[mem] SFT out of memory → micro-batch {_batch} × grad-accum {_accum} (effective batch unchanged); resuming")
            mem_report()
            trainer = make_sft_trainer(_batch, _accum)
    print(out)
    model.save_pretrained(SFT_DIR / "adapter"); tokenizer.save_pretrained(SFT_DIR / "adapter")
    (SFT_DIR / "trainer_log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    mark_stage("SFT_train")
if (SFT_DIR / "trainer_log_history.json").exists():
    h = pd.DataFrame(json.loads((SFT_DIR / "trainer_log_history.json").read_text()))
    fig, ax = plt.subplots(figsize=(6, 3))
    if "loss" in h: ax.plot(h.dropna(subset=["loss"]).step, h.dropna(subset=["loss"]).loss, label="train loss", lw=2)
    if "eval_loss" in h: ax.plot(h.dropna(subset=["eval_loss"]).step, h.dropna(subset=["eval_loss"]).eval_loss, "o-", label="eval loss", lw=2)
    ax.set_xlabel("step"); ax.legend(); ax.set_title("SFT loss"); plt.tight_layout()
    fig.savefig(FIG / "fig0_sft_loss.png", dpi=300); plt.show()

## 15.1) Pre-RL check: SFT evaluation, gate G1, decision D1 (v1 9 "pre-RL baseline", now meaningful)
The SFT model is run three ways: **its own mode choice**, **forced `<think>`**, and **forced `<short>`**.
* **G1:** format ≥ 95%, think-rate between 20% and 80%, and MATH accuracy no worse than base − 5 points.
* **D1 (reported, no longer a switch):** does forced thinking beat forced short answers on MATH levels 4–5? The planned fallback (R1-Distill-1.5B) is not viable at our token budget: it scores 39% on MATH-500 at 2,048 tokens vs 74% for the base (Table 15), and its GRPO rollouts would be 4–7× longer.
* **S2 result (long think traces, now `sft_long`):** G1 and D1 failed. 52% of forced thinks on MATH hit the token limit, mostly repetition loops. The ones that finished were as accurate as short answers (90.0% vs 89.2%). S2b retrains on think traces of ≤1,500 tokens, and this cell prints both SFTs side by side.

In the pilot these checks run but are not meaningful (25 SFT steps).

In [ ]:
# 15.1) Evaluate SFT (own choice + forced modes), then G1 and D1
def mode_report(key, base_key="base"):
    dfs = [eval_model(key, ("single", "forced_think", "forced_short"))]
    t = acc_table(pd.concat(dfs)); show(t)
    s = t[(t.condition == "single")].set_index("bench")
    base_math = acc_table(eval_model(base_key, "single", difficulty=True)).set_index("bench").loc["math500", "acc"]
    g1 = dict(format_ok=s.loc["math500", "format_ok"] >= 0.95,
              think_rate_in_20_80=0.2 <= s.loc["math500", "think_rate"] <= 0.8,
              math_not_worse_than_base_minus_5=s.loc["math500", "acc"] >= base_math - 0.05)
    lv = pd.concat(dfs); lv = lv[(lv.bench == "math500") & (lv.kind == "greedy")]
    by = lv.groupby(["condition", "level"]).correct.mean().unstack(0)
    display(by.round(3))
    hard = lv[lv.level >= 4].groupby("condition").correct.mean()
    d1 = hard.get("forced_think", 0) > hard.get("forced_short", 0)
    tag = " (PILOT: 25 SFT steps; not meaningful)" if globals().get("PILOT") else ""
    print("G1:", g1, "→", "PASS" if all(g1.values()) else "FAIL", tag)
    print(f"D1 (diagnostic): MATH L4–5 forced_think {hard.get('forced_think', np.nan):.3f} vs forced_short "
          f"{hard.get('forced_short', np.nan):.3f} → "
          f"{'thinking helps on hard problems' if d1 else 'thinking does not beat short answers on L4–5 (reported)'}{tag}")
    ft, fs = (lv[lv.condition == c].set_index("pid") for c in ("forced_think", "forced_short"))
    fin = ft[~ft.truncated.astype(bool)]
    if len(fin) and len(fs):
        print(f"    forced_think on MATH: {len(fin) / len(ft):.0%} finish, {ft.repetition.mean():.0%} loop; finished thinks "
              f"{fin.correct.mean():.3f} vs forced_short {fs.reindex(fin.index).correct.mean():.3f} on the same {len(fin)} problems")
    return t, g1, d1
if (RUNS / "sft_v2" / "adapter" / "adapter_config.json").exists():
    SFT_T, G1, D1 = mode_report("sft_v2")
    if STAGES["P4_sft"]:
        mark_stage("P4_sft_eval")
    if all((EVAL / "preds" / f"sft_long__{c}.jsonl").exists() for c in ("single", "forced_think", "forced_short")):
        _long = acc_table(eval_model("sft_long", ("single", "forced_think", "forced_short")))
        _cmp = pd.concat([_long.assign(sft="long traces (≤3400)"),
                          SFT_T.assign(sft=f"budget-fit (≤{CFG['sft_max_target_tokens']})")])
        print("SFT ablation on MATH-500, long vs budget-fit think traces:")
        show(_cmp[_cmp.bench == "math500"][["sft", "condition", "acc", "format_ok", "think_rate", "truncated",
                                            "repetition", "mean_tokens"]])

# Part 2: Stage B, RL-augmented reasoning (GRPO)
v1's design is kept: the SFT checkpoint gives the initial weights, the policy **generates rollouts**, a **verifier** scores them, and **GRPO** updates the weights. The rewards are now verifiable and **mode-aware**: a correct answer earns more when it uses fewer tokens, so the model learns to think only when it pays off.

In [ ]:
# VRAM cleanup + 4) RL configuration (from 5)
gc.collect(); torch.cuda.empty_cache()
print(json.dumps(CFG["grpo"], indent=1))

## 5) RL training prompts
✅ FIX: v1 re-sliced Hybrid-OpenThoughts2 *without* the shuffle, so the leakage guard did nothing, and the correctness reward compared the last number of a reasoning trace. The RL prompts now come from **MATH-train (levels 1–5, stratified) + GSM8K-train**, with gold answers checked by math-verify. They are disjoint from the SFT questions and decontaminated. A separate held-out pool is reserved for DPO.

In [ ]:
# 5) RL prompts (MATH-train levels 1–5 + GSM8K-train) with gold answers; disjoint DPO prompt pool
RL_TRAIN, DPO_PROMPTS, RL_STATS = DATA / "rl_train.jsonl", DATA / "dpo_prompts.jsonl", DATA / "rl_stats.json"
if STAGES["P1_data"] and not RL_TRAIN.exists():
    from datasets import load_dataset
    sft_hashes = {r["qhash"] for r in read_jsonl(DATA / "sft_pairs_full.jsonl")}
    pool, stats = [], Counter()
    math_tr = load_dataset("DigitalLearningGmbH/MATH-lighteval", split="train")
    for i, r in enumerate(math_tr):
        boxes = common.boxed_answers(r["solution"])
        if not boxes:
            stats["math_no_box"] += 1; continue
        gold = boxes[-1]
        try:
            common.symbolic_gold(gold)
        except common.MV_ERRORS:                           # includes math-verify timeouts (BaseException)
            stats["math_gold_unparseable"] += 1; continue
        lvl = re.search(r"\d", str(r["level"]))
        pool.append(dict(id=f"math_train:{i}", question=r["problem"], gold=gold, task="math_train",
                         level=int(lvl.group()) if lvl else None, reference=r["solution"]))
    gsm_tr = load_dataset("openai/gsm8k", "main", split="train")
    for i, r in enumerate(gsm_tr):
        pool.append(dict(id=f"gsm8k_train:{i}", question=r["question"], gold=r["answer"].rsplit("####", 1)[1].strip(),
                         task="gsm8k", level=0, reference=r["answer"]))
    clean = []
    for r in pool:
        h = common.text_hash(r["question"])
        if h in sft_hashes: stats["overlaps_sft"] += 1; continue
        if contaminated(r["question"]): stats["contaminated"] += 1; continue
        r["prompt"] = common.build_prompt(TOK, r["question"], "policy")
        if len(TOK(r["prompt"], add_special_tokens=False)["input_ids"]) > CFG["rl_max_prompt_tokens"]:
            stats["prompt_too_long"] += 1; continue
        clean.append(r)
    rng = random.Random(CFG["seed"]); rng.shuffle(clean)
    gsm = [r for r in clean if r["task"] == "gsm8k"]; mth = [r for r in clean if r["task"] == "math_train"]
    n_g = int(CFG["rl_n_prompts"] * CFG["rl_gsm8k_frac"]); n_m = CFG["rl_n_prompts"] - n_g
    by_lvl = defaultdict(list)
    for r in mth: by_lvl[r["level"]].append(r)
    per = n_m // 5
    rl = gsm[:n_g] + [r for l in range(1, 6) for r in by_lvl[l][:per]]
    used = {r["id"] for r in rl}
    dpo = [r for r in clean if r["id"] not in used][:CFG["dpo_n_prompts"]]
    rng.shuffle(rl)
    write_jsonl(RL_TRAIN, rl); write_jsonl(DPO_PROMPTS, dpo)
    RL_STATS.write_text(json.dumps({**stats, "pool": len(pool), "clean": len(clean), "rl": len(rl), "dpo": len(dpo),
                                    "rl_by_level": Counter(str(r["level"]) for r in rl)}, indent=1))
print(json.loads(RL_STATS.read_text()) if RL_STATS.exists() else "RL data not built yet")
if STAGES["P1_data"]:
    mark_stage("P1_data")

## 6a) Rule-based verifier (format + repetition), now in `common.py`
The format rule is: exactly one mode, closed tags, and a final `\boxed{}`. An unclosed or truncated `<think>` is invalid.

✅ FIX (2026-10-09): the repetition detector is now a real **loop** detector. It flags a block of words repeated back-to-back at least 5 times (covering at least 50 words), or a tail that compresses like a loop. v1's rule ("any 4-word phrase seen 6 times") flagged 21% of the base model's normal answers and 51% of R1-Distill's (90% of them correct), because it measured length rather than loops. In GRPO that would have put a −0.5 penalty on about half of all long `<think>` answers. The new rule flags 0 of 8,928 correct answers in the S1 baselines.

In [ ]:
# 6a) Verifier demo (definitions in common.py; full self-check already passed in 5.4)
for t in ["<short>The answer is \\boxed{5}</short>", "<think>2 + 3 = 5</think> So the total is \\boxed{5}.",
          "<think>never closed ... \\boxed{5}", "I think the result is 12.", "<think>" + "the answer is " * 40 + "</think> \\boxed{1}"]:
    print(f"{t[:55]!r:60} mode={common.parse_policy_output(t)[0]:9} format_ok={common.format_ok(t)!s:5} "
          f"loop={common.has_repetition_loop(t)}")

## 6b–6d) The neural judge, now *measured* instead of trusted
In v1 a 4-bit DeepSeek-R1-Distill-1.5B judge produced both the reward and the benchmark scores. Here the judges are scored against **strict gold labels**, all on the same outputs:
* **Judges:** the v1 judge (1.5B, 512 tokens), the same judge with 2048 tokens (no cut-off), and a 7B judge.
* **Parsers:** each judge is scored with two parsers.
  * The **v1 parser** takes the first number after `</think>`. This is what v1 actually optimised.
  * The **corrected parser** takes the number that completes `SCORE:`, i.e. the judge's real verdict.

**Pilot finding:** the 1.5B judge writes its verdict first, then `</think>`, then re-solves the problem. The v1 parser read a number from that re-solution and clipped it to 1.0. So most of v1's "the judge says correct" was a **parser artifact** (FPR 95% → 17.5% with the corrected parser), and the judge gave no valid score 37.5% of the time.

We report precision, recall, **false-positive rate**, Cohen's κ, and the share of outputs with no valid score. 6d re-reads your v1 CSVs. R2 below optimises the judge's **actual** verdict (`judge_parser="fixed"`).

In [ ]:
# 6b) Run the judges (v1 judge + 2 controls) on gold-labelled base-model outputs; both parsers
# (does the failure come from truncation or from model size?)
JUDGE_CONFIGS = [   # key, label, model, max_new_tokens, output file  (v1 keeps its original file/job name = cache)
    ("v1",   "R1-Distill-1.5B · 512 tok (v1 judge)",   CFG["judge_model"], CFG["judge_v1_tokens"], "judge_offline.jsonl"),
    ("long", "R1-Distill-1.5B · 2048 tok (no cut-off)", CFG["judge_model"], CFG["judge_long_tokens"], "judge_offline__long.jsonl"),
    ("big",  "Qwen2.5-7B-Instruct · 512 tok (larger)",  CFG["judge_large"], CFG["judge_v1_tokens"], "judge_offline__big.jsonl"),
]
def judge_items():
    """Deterministic: all wrong (up to N) + the same number of correct base-model outputs, GSM8K + MATH-500."""
    src = pd.DataFrame(read_jsonl(EVAL / "preds" / "base__single.jsonl"))
    src = src[src.bench.isin(["gsm8k", "math500"]) & src.kind.isin(["greedy", "difficulty"])]
    wrong, right = src[~src.correct], src[src.correct]
    n = min(CFG["judge_items_per_class"], len(wrong), len(right))
    return pd.concat([wrong.sample(n, random_state=CFG["seed"]), right.sample(n, random_state=CFG["seed"])]).reset_index(drop=True)

def judge_scores(text):
    """Both parsers on the same judge output: v1's (first number after </think>, as v1 trained on)
    and the corrected one (the number that completes 'SCORE:', i.e. the judge's actual verdict)."""
    v1, fx = common.parse_judge_v1(text), common.parse_judge_fixed(text)
    d = common.JUDGE_DEFAULT_ON_ERROR
    return dict(judge_text=text, judge_raw=text[-400:], judge_v1=v1, judge_fixed=fx,
                judge_strict=common.parse_judge_strict(text),
                judge_score=d if v1 is None else v1, judge_score_fixed=d if fx is None else fx)

if STAGES["P3_judge"]:
    todo = [c for c in JUDGE_CONFIGS if not (EVAL / "preds" / c[4]).exists()]
    if todo:
        items = judge_items()
        by_pid = BENCH.set_index("pid")
        prompts = [common.judge_prompt(by_pid.loc[r.pid].question, by_pid.loc[r.pid].gold, r.text) for r in items.itertuples()]
        for key, label, model_name, max_new, fname in todo:
            reqs = [dict(id=str(i), prompt=p, n=1, temperature=0.0, max_tokens=max_new) for i, p in enumerate(prompts)]
            outs, _ = generate("judge_offline" if key == "v1" else f"judge_offline_{key}", model_name, reqs)
            jrows = []
            for o in outs:
                r = items.iloc[int(o["id"])]
                jrows.append(dict(judge=label, pid=r.pid, bench=r.bench, kind=r.kind, gold_correct=bool(r.correct),
                                  status=r.status, has_final_answer=r.final_answer is not None, cand_tokens=r.n_tokens,
                                  judge_truncated=o["finish_reason"] == "length", **judge_scores(o["text"])))
            write_jsonl(EVAL / "preds" / fname, jrows)
        mark_stage("P3_judge")

In [ ]:
# 6c) Judge reliability metrics
def judge_metrics(j, thr=0.7, col="judge_score"):
    pred, gold = j[col] >= thr, j.gold_correct.astype(bool)
    tp, fp = int((pred & gold).sum()), int((pred & ~gold).sum())
    tn, fn = int((~pred & ~gold).sum()), int((~pred & gold).sum())
    n = len(j); po = (tp + tn) / n
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n**2
    fpr_lo, fpr_hi = common.wilson(fp, fp + tn) if fp + tn else (np.nan, np.nan)
    return dict(n=n, TP=tp, FP=fp, TN=tn, FN=fn, precision=tp / max(1, tp + fp), recall=tp / max(1, tp + fn),
                FPR=fp / max(1, fp + tn), FPR_ci_low=fpr_lo, FPR_ci_high=fpr_hi, accuracy=po,
                kappa=(po - pe) / (1 - pe) if pe < 1 else np.nan,
                no_valid_score=(j["judge_v1" if col == "judge_score" else "judge_fixed"].isna().mean()),
                judge_truncated=j.judge_truncated.mean())

def load_judge(key, fname):
    """Judge rows; caches written before the corrected parser existed are re-scored from the saved raw
    vLLM outputs (eval/jobs/*.jsonl). No GPU is needed."""
    rows = read_jsonl(EVAL / "preds" / fname)
    if rows and "judge_fixed" not in rows[0]:
        job = EVAL / "jobs" / (("judge_offline" if key == "v1" else f"judge_offline_{key}") + ".jsonl")
        texts = {int(o["id"]): o["text"] for o in read_jsonl(job)}
        if len(texts) != len(rows):
            print(f"[judge] raw outputs for {key} not found ({job.name}); corrected-parser metrics skipped")
            return pd.DataFrame(rows)
        rows = [{**r, **judge_scores(texts[i])} for i, r in enumerate(rows)]
        write_jsonl(EVAL / "preds" / fname, rows)
    return pd.DataFrame(rows)

J = load_judge("v1", "judge_offline.jsonl")
if len(J):
    JUDGE_TABLE = pd.DataFrame([{"subset": "all", **judge_metrics(J)}] +
        [{"subset": f"bench={b}", **judge_metrics(x)} for b, x in J.groupby("bench")] +
        [{"subset": "no final answer in candidate", **judge_metrics(J[~J.has_final_answer])}] +
        [{"subset": f"candidate tokens {int(q.left)+1}–{int(q.right)}", **judge_metrics(x)}
         for q, x in J.groupby(pd.qcut(J.cand_tokens.astype(float), 4, duplicates="drop"), observed=True)])
    show(JUDGE_TABLE)
    JUDGE_TABLE.to_csv(TAB / "table3_judge_reliability.csv", index=False)

# Table 3b: judge controls on the SAME items × both parsers
# (v1 parser = what v1 optimised; corrected parser = the judge's actual verdict)
JC = []
for key, label, _, _, fname in JUDGE_CONFIGS:
    d = load_judge(key, fname)
    if len(d):
        for parser, col in (("v1 parser", "judge_score"), ("corrected parser", "judge_score_fixed")):
            if col not in d:
                continue
            JC.append({"judge": label, "parser": parser, "subset": "all", **judge_metrics(d, col=col)})
            if (~d.has_final_answer).any():
                JC.append({"judge": label, "parser": parser, "subset": "no final answer",
                           **judge_metrics(d[~d.has_final_answer], col=col)})
JUDGE_COMPARE = pd.DataFrame(JC)
if len(JUDGE_COMPARE):
    show(JUDGE_COMPARE); JUDGE_COMPARE.to_csv(TAB / "table3b_judge_controls.csv", index=False)
    JUDGE_COMPARE.round(3).to_latex(TAB / "table3b_judge_controls.tex", index=False, label="tab:judge_controls",
                                    caption="Judge controls on identical gold-labelled outputs: the v1 1.5B judge, the same "
                                            "judge without output truncation, and a 7B instruct judge, each scored with the v1 "
                                            "parser (first number after the reasoning) and the corrected parser (the judge's "
                                            "actual score).")
    fx = JUDGE_COMPARE[(JUDGE_COMPARE.subset == "all")]
    for r in fx.itertuples():
        print(f"{r.judge:45} {r.parser:17} FPR={r.FPR:.3f} [{r.FPR_ci_low:.2f},{r.FPR_ci_high:.2f}] "
              f"recall={r.recall:.3f} κ={r.kappa:+.3f} no-valid-score={r.no_valid_score:.2f}")

In [ ]:
# 6d) v1 case study: what the v1 judge accepted (v1 headline numbers came from this)
v1_dir = V1_BUNDLE / "experiment_data"
REFUSAL = re.compile(r"(can't help|cannot help|I'm here to help|I'm sorry|unable to (?:discuss|help)|political)", re.I)
case = []
for f in sorted(v1_dir.glob("*_evaluation_results.csv")) if v1_dir.exists() else []:
    d = pd.read_csv(f)
    acc = d[d["Judge Correct"] == True]
    for _, r in acc.iterrows():
        ans, gt = str(r["Model Answer"]), str(r["Ground Truth"])
        gt_num = common.rational(gt)
        nums = [common.rational(x) for x in re.findall(r"-?\d+(?:\.\d+)?(?:/\d+)?", ans)]
        case.append(dict(file=f.name, refusal=bool(REFUSAL.search(ans)),
                         numeric_gold_absent=gt_num is not None and gt_num not in nums, answer=ans[:100], gold=gt[:40]))
CASE = pd.DataFrame(case)
if len(CASE):
    print("v1 rows the judge accepted as correct:", len(CASE))
    print(CASE.groupby("file")[["refusal", "numeric_gold_absent"]].sum())
    show(CASE[CASE.refusal | CASE.numeric_gold_absent].head(12))
    CASE.to_csv(EXP / "v1_judge_case_study.csv", index=False)
else:
    print("v1 bundle not found on Drive; case study skipped.")

## 7–8) Initial weights = SFT checkpoint; reward functions
| Reward | R1 **ours** | R2 judge | R3 std. length |
|---|---|---|---|
| correctness: strict scorer 1/0 (truncated or malformed = 0) | ✓ | the 1.5B judge's score instead | ✓ |
| format: +0.1 valid / −0.5 invalid | ✓ | ✓ | ✓ |
| repetition loop: −0.5 | ✓ | ✓ | ✓ |
| ★ **difficulty-aware mode reward** (below), *only if correct* | ✓ | ✓ (judge decides "correct") | ✗ |
| standard length bonus: +0.2·(1 − tokens/2048), *only if correct* | ✗ | ✗ | ✓ |
| monitors (weight 0): strict accuracy, think / short / malformed rate | logged | logged | logged |

★ **Difficulty-aware mode reward (C2, new).** GRPO samples 8 answers per problem. Their pass rate *p* is a free, on-policy estimate of how hard the problem is **for the model right now**. No difficulty model and no extra rollouts are needed. A correct answer earns:
* **mode bonus** 0.1 · (*p* for `<short>`, 1 − *p* for `<think>`): be brief when the problem is easy for you, think when it is hard;
* **length bonus** 0.1 · (n_max − n)/(n_max − n_min) among the group's correct answers **in the same mode**: no wasted tokens within the chosen mode.

The total bonus budget (0.2) equals R3's standard length bonus, so **R1 vs R3 isolates how the bonus is allocated**. That comparison is the evidence for the new reward. Related ideas (ALP, DAST) scale a *length* penalty by solve rate. Here the signal drives the *think/short decision*.

In [ ]:
# 7) + 8) Check the initial weights exist; show the reward functions on example rollouts
assert (RUNS / "sft_v2" / "adapter" / "adapter_config.json").exists() or not any(
    STAGES.get(k) for k in ("P5_smoke", "P5_ours", "P5_balanced", "P5_judge", "P5_stdlen")), "Run Stage A (9–15) first."
ex = ["<short>\\boxed{5}</short>", "<think>" + "work " * 300 + "</think> \\boxed{5}", "<short>\\boxed{6}</short>",
      "<think>never closed \\boxed{5}"]
gold, task = ["5"] * 4, ["gsm8k"] * 4
ids = [[0] * len(TOK(t)["input_ids"]) for t in ex]
print(pd.DataFrame({"completion": [t[:40] for t in ex],
                    "correctness": common.correctness_reward(None, ex, gold, task),
                    "format": common.format_reward(ex), "repetition": common.repetition_reward(ex),
                    "difficulty_aware (R1)": np.round(common.adaptive_mode_reward(None, ex, gold, task, completion_ids=ids), 3),
                    "std_length (R3)": np.round(common.mode_length_reward(None, ex, gold, task, completion_ids=ids), 3)}))
print("one group, pass rate p = 0.5 → a correct <short> earns 0.1·p, a correct <think> 0.1·(1−p), + brevity within its mode")

## 10–11) GRPO training runs (resumable separate processes)
**Settings:** vLLM-colocated rollouts, T=1.0, **no repetition penalty** (✅ FIX), 8 generations × 8 prompts per step, lr 1e-5, β=0, a checkpoint every 25 steps, and a time limit per session.

**G2:** if the think-rate leaves the 5–95% band after step 60, the run stops and writes `G2_MODE_COLLAPSE.json`.

In the **pilot**, every run takes a few steps, which checks the APIs. In the **full** study every run trains 200 steps; R1, R2 and R3 must all use the same steps.

In [ ]:
# 10) GRPO config + curve helpers
def grpo_cfg(variant, out_name, **over):
    cfg = dict(base=CFG["policy_base"], init_adapter=str(RUNS / "sft_v2" / "adapter"), variant=variant,
               out_dir=str(RUNS / out_name), train_file=str(RL_TRAIN), seed=CFG["seed"],
               judge_model=CFG["judge_model"])
    cfg.update(CFG["grpo"]); cfg.update(over)          # over may change anything, including the seed
    return cfg

def grpo_fallbacks(cfg):
    """OOM fallbacks, tried in order. per_device_batch*grad_accum (the effective batch, and the 64 rollouts per
    step) never changes, so retries do not change the experiment. They only use less memory at a time."""
    pdb, ga = cfg["per_device_batch"], cfg["grad_accum"]
    out = []
    for k, extra in ((2, {}), (4, {"vllm_enable_sleep_mode": True})):   # sleep mode offloads vLLM during training
        if pdb // k < 1:
            break
        out.append(dict(per_device_batch=pdb // k, grad_accum=ga * k,
                        vllm_gpu_mem=round(cfg["vllm_gpu_mem"] * (0.85 if k == 2 else 0.7), 2),
                        judge_batch=max(4, cfg["judge_batch"] // k), **extra))
    return out

def run_grpo(cfg, name):
    """GRPO in a fresh process; on CUDA OOM: free memory, shrink the micro-batch/vLLM share, resume from checkpoint."""
    return run_adaptive("train_grpo.py", cfg, name, grpo_fallbacks(cfg))

def run_log(name):
    p = RUNS / name / "trainer_log_history.json"
    if p.exists():
        return pd.DataFrame(json.loads(p.read_text()))
    ck = sorted((RUNS / name / "checkpoints").glob("checkpoint-*/trainer_state.json"),
                key=lambda x: int(x.parent.name.split("-")[1]))
    return pd.DataFrame(json.loads(ck[-1].read_text())["log_history"]) if ck else pd.DataFrame()

def mode_accuracy(h, window=10):
    """Per-step accuracy of think and of short rollouts from the logged monitors (rolling ratio of sums)."""
    col = lambda k: next((c for c in h.columns if k in c and c.endswith("/mean")), None)
    tc, tr, sa = col("think_correct"), col("think_rate"), col("strict_accuracy")
    if not (tc and tr and sa):
        return {}
    r = lambda x: h[x].rolling(window, min_periods=1).sum()    # each step has the same 64 rollouts → sums of means
    think = r(tr)
    short = h[tr].rolling(window, min_periods=1).count() - think
    return {"think accuracy": r(tc) / think.where(think > 0),
            "short accuracy": (r(sa) - r(tc)) / short.where(short > 0)}

def quick_curves(name):
    h = run_log(name)
    if h.empty or "step" not in h:
        print("no logs yet for", name); return
    cols = [c for c in h.columns if c.startswith("rewards/") and c.endswith("/mean")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for c in cols:
        if "think_correct" in c:
            continue                                  # shown as per-mode accuracies below
        if any(k in c for k in ("strict_accuracy", "think_rate", "short_rate", "malformed_rate")):
            axes[1].plot(h.step, h[c].rolling(5, min_periods=1).mean(), lw=2, label=c.split("/")[1])
        else:
            axes[0].plot(h.step, h[c].rolling(5, min_periods=1).mean(), lw=2, label=c.split("/")[1])
    for lab, acc in mode_accuracy(h).items():
        axes[1].plot(h.step, acc, lw=1.5, ls="--", label=lab)
    for a, t in zip(axes, ("reward components", "monitors (rates)")):
        a.set_title(f"{name}: {t}"); a.set_xlabel("step"); a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

# One-time archive (S6): R1 and R3 trained with another advantage scaling (TRL's default
# group std, S3–S4) are kept as the "*_gstd" ablation: run folder (adapters, the step-60 adapter, logs), predictions,
# eval cache. R1 and R3 then retrain from scratch with CFG's scaling. R1b keeps its group-std run.
GSTD_RUNS = {"grpo_ours": ("grpo_ours_gstd", "P5_R1_ours"), "grpo_stdlen": ("grpo_stdlen_gstd", "P5_R3_stdlen")}
def archive_group_std():
    if PILOT:
        return
    for run, (arch, stage) in GSTD_RUNS.items():
        rc = RUNS / run / "resolved_grpo_config.json"
        if not rc.exists():
            continue
        used = json.loads(rc.read_text())["args"].get("scale_rewards", "group")
        if used == CFG["grpo"]["scale_rewards"]:
            continue
        assert not (RUNS / arch).exists(), f"{arch} already exists; archive {run} by hand"
        shutil.move(str(RUNS / run), str(RUNS / arch))
        for p in sorted((EVAL / "preds").glob(f"{run}__*.jsonl")):
            rows = read_jsonl(p)
            for r in rows:
                r["model"] = arch
            write_jsonl(EVAL / "preds" / p.name.replace(f"{run}__", f"{arch}__"), rows); p.unlink()
        (EVAL / "jobs" / arch).mkdir(parents=True, exist_ok=True)
        for p in sorted((EVAL / "jobs").glob(f"eval_{run}_*")):   # else the new run's eval would reuse these generations
            if re.match(rf"eval_{run}_(single|forced_think|forced_short)", p.name):   # not eval_grpo_ours_step60_*
                shutil.move(str(p), str(EVAL / "jobs" / arch / p.name))
        (RUNS / arch / "ARCHIVED.json").write_text(json.dumps({
            "utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "scale_rewards": used,
            "reason": "CoT collapse after step ≈130 (S5 greedy eval: R1 MATH 74.2 at step 60 → 50.4 at step 200)"}))
        print(f"archived {run} (scale_rewards={used!r}) as {arch}; {run} retrains with "
              f"scale_rewards={CFG['grpo']['scale_rewards']!r}")
archive_group_std()

In [ ]:
# 11) R1: GRPO ours (strict verifier + format + repetition + difficulty-aware mode reward)
def resume_after_g2_stop(run):
    """S3 stopped R1 at the G2 collapse (step 60). Since S4, G2 is reported, not enforced (user decision
    2026-10-10), so every GRPO run trains its full 200 steps. One-time: keep the stopped adapter as a record and
    let the run resume from its last checkpoint."""
    d = RUNS / run
    summ = d / "train_summary.json"
    if CFG["grpo"]["stop_on_collapse"] or not (d / "adapter" / "adapter_config.json").exists() or not summ.exists():
        return
    steps = json.loads(summ.read_text())["steps"]
    if steps >= CFG["grpo"]["max_steps"]:
        return
    keep = d / f"adapter_g2stop_step{steps}"
    assert not keep.exists(), keep
    shutil.move(str(d / "adapter"), str(keep))
    for f in ("trainer_log_history.json", "train_summary.json"):
        if (d / f).exists():
            shutil.copy(d / f, keep / f)
    print(f"{run}: G2-stopped adapter (step {steps}) kept as {keep.name}; training resumes to "
          f"{CFG['grpo']['max_steps']} steps from the last checkpoint")

def mode_summary(run, label):
    h = run_log(run)
    a = mode_accuracy(h) if not h.empty else {}
    if not a:
        return
    tr = [c for c in h.columns if "think_rate" in c and c.endswith("/mean")][0]
    pt = [c for c in h.columns if "policy_think_prob" in c and c.endswith("/mean")]
    print(f"{label} rollouts, last 10 steps: think-rate {h[tr].tail(10).mean():.3f}"
          + (f" (forced 50/50; the policy's own think probability {h[pt[0]].tail(10).mean():.3f})" if pt else "")
          + f" | think acc {a['think accuracy'].iloc[-1]:.3f} | short acc {a['short accuracy'].iloc[-1]:.3f}")

if STAGES["P5_ours"]:
    resume_after_g2_stop("grpo_ours")
    go, over = stage_gate("P5_ours")
    run_grpo(grpo_cfg("ours", "grpo_ours", **over), "grpo_ours")
    mark_stage("P5_R1_ours")
quick_curves("grpo_ours")
if (RUNS / "grpo_ours" / "G2_MODE_COLLAPSE.json").exists():
    print("G2 (reported, not enforced):", (RUNS / "grpo_ours" / "G2_MODE_COLLAPSE.json").read_text(),
          "→ the policy stopped thinking; see the per-mode accuracies and R1b (balanced) below.")
mode_summary("grpo_ours", "R1")

In [ ]:
# 11a) R1b: R1 with balanced think/short exploration (AdaptThink-style; fairness check after R1's G2 collapse)
# Same rewards, steps and settings as R1; only the rollouts differ: half of every group is forced to think and
# half to answer short (importance-sampled mode token), so thinking keeps being trained and compared per problem.
if STAGES.get("P5_balanced"):
    go, over = stage_gate("P5_balanced")
    if go:
        run_grpo(grpo_cfg("ours", "grpo_balanced", balanced=True, **over), "grpo_balanced")
        mark_stage("P5_R1b_balanced")
quick_curves("grpo_balanced")
if (RUNS / "grpo_balanced" / "G2_MODE_COLLAPSE.json").exists():
    print("G2 (reported):", (RUNS / "grpo_balanced" / "G2_MODE_COLLAPSE.json").read_text())
mode_summary("grpo_balanced", "R1b")

In [ ]:
# 11b) R2: GRPO with the 1.5B LLM judge as correctness reward (reward-hacking study)
if STAGES["P5_judge"]:
    go, over = stage_gate("P5_judge")
    if go:
        run_grpo(grpo_cfg("judge", "grpo_judge", vllm_gpu_mem=0.2,   # leave room for the in-process judge model
                       **{**({"max_steps": CFG["judge_run_steps"]} if CFG["judge_run_steps"] else {}), **over}), "grpo_judge")
        mark_stage("P5_R2_judge")
quick_curves("grpo_judge")

In [ ]:
# 11c) R3: standard-length ablation (isolates the new reward)
# (same total bonus), so R1 vs R3 isolates the new reward.
if STAGES["P5_stdlen"]:
    go, over = stage_gate("P5_stdlen")
    if go:
        run_grpo(grpo_cfg("std_length", "grpo_stdlen", **over), "grpo_stdlen")
        mark_stage("P5_R3_stdlen")
quick_curves("grpo_stdlen")

In [ ]:
# 11e) R1 with a second (S7) and a third (S9) training seed, for seed robustness (Table 14)
if STAGES.get("P5_seed2"):
    go, over = stage_gate("P5_seed2")
    if go:
        run_grpo(grpo_cfg("ours", "grpo_ours_s2", seed=CFG["seed"] + 1), "grpo_ours_s2")
        mark_stage("P5_R1_seed2")
quick_curves("grpo_ours_s2")
# Third seed (S9), so Table 14 reports mean ± sd over 3 seeds.
if STAGES.get("P5_seed3"):
    go, over = stage_gate("P5_seed3")
    if go:
        run_grpo(grpo_cfg("ours", "grpo_ours_s3", seed=CFG["seed"] + 2, **over), "grpo_ours_s3")
        mark_stage("P5_R1_seed3")
quick_curves("grpo_ours_s3")

In [ ]:
# 11f) Mechanism controls of the group-std CoT collapse (S8)
# Same SFT start, prompts, seed, steps and settings as R1; exactly one factor changes per run.
#   C1 group std, NO brevity bonus (correctness + format + repetition) → is the bonus necessary for the collapse?
#   C2 R1's bonus, no std scaling (scale_rewards="none", as in Dr. GRPO) → the other published fix
#   C3 group std, R1's bonus × 0.1 → prediction: the SAME collapse. In an all-correct group only the bonus varies,
#      and dividing by the group std turns any bonus, however small, into a unit-size advantage.
# Optional: cut early (after R1's seed 3: C3, then C2, then C1). The monitor all_correct_group logs the share of
# all-correct groups per step.
CONTROL_RUNS = [("P5_ctrl_nobonus", "grpo_nobonus_gstd", "P5_C1_nobonus_gstd", "no_length", dict(scale_rewards="group")),
                ("P5_ctrl_nostd", "grpo_ours_nostd", "P5_C2_ours_nostd", "ours", dict(scale_rewards="none")),
                ("P5_ctrl_b01", "grpo_ours_gstd_b01", "P5_C3_ours_gstd_b01", "ours",
                 dict(scale_rewards="group", bonus_scale=0.1))]
for stage, run, label, variant, extra in CONTROL_RUNS:
    if STAGES.get(stage):
        go, over = stage_gate(stage)
        if go:
            run_grpo(grpo_cfg(variant, run, **extra, **over), run)
            mark_stage(label)
    quick_curves(run)

## 11g) Second base model: does the behaviour depend on the math-specialised base? (S10–S11)
Reviewers will ask whether the findings are specific to Qwen2.5-Math-1.5B-Instruct. Here the **same recipe** runs on **Qwen2.5-1.5B-Instruct**, a general instruct model of the same size with the same vocabulary and chat template. It is **not** math-specialised (published GSM8K 73.2, MATH 55.2, vs 84.8 / 75.8).
* Same 3,000 verified SFT pairs, RL prompts, seed, rewards, settings and 200 GRPO steps. Only the base model changes.
* **S10:** SFT, then R1's recipe (difficulty-aware reward, batch-std advantages). **S11:** R1's recipe with TRL's group std, the collapse condition of C2f.
* Every checkpoint gets its greedy evaluation as soon as it exists: single, plus forced think/short for SFT and R1.

**Pre-registered predictions (2026-10-13, before any second-model run; verdicts in Table 19d):**
1. **The more accurate mode wins.** If the second model's short rollouts are more accurate than its think rollouts over steps 1–20, its rollout think-rate is below 5% over steps 61–100. If thinking is more accurate there, thinking survives (≥ 5%).
2. **Group std collapses the CoT, batch std does not.** The group-std run's mean completion length at steps 161–200 is below 0.6 × steps 61–100; R1's recipe (batch std) stays above.

In [ ]:
# 11g) Second base model (Qwen2.5-1.5B-Instruct): SFT, R1, R1 with group std, and their evaluation
# Does the recipe's behaviour depend on the math-specialised base? The same recipe on Qwen2.5-1.5B-Instruct, a general
# instruct model with the same size, tokenizer and chat template: the same 3,000 verified SFT pairs, RL prompts, seed,
# rewards and 200 GRPO steps.
#   m2_sft        LoRA SFT (train_sft.py = the 9–15 recipe)                                   S10
#   m2_grpo       R1's recipe (difficulty-aware reward, batch-std advantages)                  S10
#   m2_grpo_gstd  R1's recipe with TRL's group std (the condition of the CoT collapse, C2f)    S11
# Each checkpoint gets its greedy eval as soon as it exists (stage P7_m2_eval; forced think/short for SFT and R1).
# Pre-registered (2026-10-13, before any second-model run; verdicts in Table 19d):
#   (a) the more accurate mode wins: if m2_grpo's short rollouts are more accurate than its think rollouts over steps
#       1–20, its rollout think-rate is below 5% over steps 61–100; if think is more accurate there, it stays ≥ 5%
#   (b) m2_grpo_gstd collapses (mean length at steps 161–200 < 0.6 × steps 61–100); m2_grpo does not
M2_CONDS = {"m2_base": ("single",), "m2_sft": ("single", "forced_think", "forced_short"),
            "m2_grpo": ("single", "forced_think", "forced_short"), "m2_grpo_gstd": ("single",)}
M2_RUNS = [("P5_m2_ours", "m2_grpo", "P5_M2_R1", {}),
           ("P5_m2_gstd", "m2_grpo_gstd", "P5_M2_R1_gstd", {"scale_rewards": "group"})]

def m2_ready(key):
    s = MODELS[key]
    return s["lora"] is None or (Path(s["lora"]) / "adapter_config.json").exists()

def m2_cached(key):
    return all((EVAL / "preds" / f"{key}__{c}.jsonl").exists() for c in M2_CONDS[key])

def m2_eval(keys):
    """Greedy eval of the second model's finished checkpoints (cached per condition)."""
    todo = [k for k in keys if m2_ready(k) and not m2_cached(k)]
    if not (STAGES.get("P7_m2_eval") and todo):
        return
    for k in todo:
        eval_model(k, M2_CONDS[k])
    mark_stage("P7_m2_eval")

def m2_grpo_cfg(run, **over):
    return grpo_cfg("ours", run, base=CFG["policy_base2"], init_adapter=str(RUNS / "m2_sft" / "adapter"), **over)

if any(STAGES.get(k) for k in ("P4_m2_sft", "P5_m2_ours", "P5_m2_gstd", "P7_m2_eval")):
    TOK2 = AutoTokenizer.from_pretrained(CFG["policy_base2"])
    for _q in ("What is 2+3?", BENCH.question.iloc[0]):   # same template + vocabulary → the SFT/RL prompts carry over
        _p = common.build_prompt(TOK, _q, "policy")
        assert common.build_prompt(TOK2, _q, "policy") == _p and TOK2(_p)["input_ids"] == TOK(_p)["input_ids"], \
            "the second model renders the prompts differently; its SFT and RL data would need rebuilding"
# 1) SFT on the same verified pairs
if STAGES.get("P4_m2_sft"):
    go, _ = stage_gate("P4_m2_sft")
    if go:
        sc = CFG["sft"]
        run_adaptive("train_sft.py", dict(base=CFG["policy_base2"], out_dir=str(RUNS / "m2_sft"), seed=CFG["seed"],
                                          train_file=str(SFT_TRAIN), eval_file=str(SFT_EVAL), **sc), "m2_sft",
                     fallbacks=[dict(batch=max(1, sc["batch"] // k), grad_accum=sc["grad_accum"] * k) for k in (2, 4)])
        mark_stage("M2_SFT_train")
m2_eval(["m2_base", "m2_sft"])
if m2_cached("m2_base"):
    _t = acc_table(eval_model("m2_base", "single")).set_index("bench")
    for b, pub in CFG["m2_published"].items():
        print(f"G0 (2nd model) {b}: harness {_t.loc[b, 'acc']:.3f} vs published {pub:.3f} → Δ {_t.loc[b, 'acc'] - pub:+.3f}"
              " (reported, not a gate: the Qwen2.5 blog's own prompt and protocol)")
if m2_cached("m2_base") and m2_cached("m2_sft"):
    print("2nd model after SFT (G1/D1 as for the main model; reported, not enforced):")
    M2_SFT_T, M2_G1, M2_D1 = mode_report("m2_sft", base_key="m2_base")
# 2) GRPO with R1's recipe (S10), then with TRL's group std (S11)
for stage, run, label, extra in M2_RUNS:
    if STAGES.get(stage):
        go, over = stage_gate(stage)
        if go:
            assert m2_ready("m2_sft"), "the second model's SFT must be trained first (S10)"
            run_grpo(m2_grpo_cfg(run, **extra, **over), run)
            mark_stage(label)
    m2_eval([run])
    if (RUNS / run).exists():
        quick_curves(run)

## 14) DPO alternative (ablation), done properly
✅ FIX: v1's DPO trained for **5 steps**, and its comparison bar used the SFT accuracy labelled as "GRPO". Here:
* **Data:** sample 4 answers per prompt from SFT on **held-out** prompts and score them strictly. **Chosen** = the shortest correct, well-formatted answer. **Rejected** = an incorrect, malformed or looping one.
* **Training:** the SFT adapter is merged first, so the DPO reference model *is* the SFT policy. Then a full epoch.

In [ ]:
# 14) DPO: build verifier pairs (vLLM), then train inline (reference = merged SFT)
PAIRS = DATA / "dpo_pairs.jsonl"
if STAGES["P6_dpo"] and stage_gate("P6_dpo")[0]:
    if not PAIRS.exists():
        prompts = read_jsonl(DPO_PROMPTS)
        reqs = [dict(id=str(i), prompt=r["prompt"], n=CFG["dpo"]["samples"], temperature=1.0, top_p=1.0,
                     max_tokens=CFG["grpo"]["max_completion_length"], seed=CFG["seed"]) for i, r in enumerate(prompts)]
        outs, _ = generate("dpo_sampling", CFG["policy_base"], reqs, lora=RUNS / "sft_v2" / "adapter")
        groups = defaultdict(list)
        for o in outs:
            groups[int(o["id"])].append(o)
        pairs, rng = [], random.Random(CFG["seed"])
        for i, cands in groups.items():
            r = prompts[i]
            good, bad = [], []
            for o in cands:
                t = o["text"]
                ok = common.score_text(t, r["gold"], r["task"])["correct"] and common.format_ok(t) \
                     and not common.has_repetition_loop(t) and o["finish_reason"] != "length"
                (good if ok else bad).append((o["n_tokens"], t))
            if good and bad:
                chosen = min(good)[1]
                for _, rej in rng.sample(bad, min(2, len(bad))):
                    pairs.append(dict(prompt=r["prompt"], chosen=chosen, rejected=rej, id=r["id"]))
        write_jsonl(PAIRS, pairs)
    print("DPO pairs:", len(read_jsonl(PAIRS)))
    DPO_DIR = RUNS / "dpo_v2"
    if not (DPO_DIR / "merged" / "config.json").exists():
        from datasets import Dataset
        from peft import LoraConfig, PeftModel
        from transformers import AutoModelForCausalLM
        from trl import DPOConfig, DPOTrainer
        dc = CFG["dpo"]
        pairs = read_jsonl(PAIRS)
        if len(pairs) < 8:
            print(f"Skipping DPO training: only {len(pairs)} pairs (SFT always right, always wrong, or never well-formatted).")
        else:
            ds = Dataset.from_list([{"prompt": p["prompt"], "chosen": p["chosen"] + "<|im_end|>",
                                     "rejected": p["rejected"] + "<|im_end|>"} for p in pairs])
            def build_dpo_trainer(batch, accum):
                """Fresh SFT-merged reference + new LoRA each time (DPOTrainer wraps the model in place,
                so after an OOM it must be rebuilt, not reused). batch*accum stays constant."""
                _base = AutoModelForCausalLM.from_pretrained(CFG["policy_base"], dtype=torch.bfloat16,
                                                             attn_implementation="sdpa")
                _sft = PeftModel.from_pretrained(_base, RUNS / "sft_v2" / "adapter").merge_and_unload()
                _sft.config.use_cache = False
                total = math.ceil(len(pairs) / (batch * accum)) * dc["epochs"]
                dargs = DPOConfig(**common.supported_kwargs(DPOConfig, dict(
                    output_dir=str(DPO_DIR / "checkpoints"), beta=dc["beta"], learning_rate=dc["lr"],
                    num_train_epochs=dc["epochs"], per_device_train_batch_size=batch,
                    gradient_accumulation_steps=accum, max_length=dc["max_length"],
                    max_prompt_length=dc["max_prompt_length"], lr_scheduler_type="cosine",
                    warmup_steps=max(1, int(0.05 * total)), bf16=True,
                    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}, logging_steps=5,
                    save_steps=50, save_total_limit=1, report_to="tensorboard", logging_dir=str(DPO_DIR / "tb"),
                    seed=CFG["seed"]), required=("beta", "learning_rate")))
                return DPOTrainer(model=_sft, args=dargs, train_dataset=ds, processing_class=TOK,
                                  peft_config=LoraConfig(r=dc["lora_r"], lora_alpha=dc["lora_alpha"], lora_dropout=0.05,
                                                         bias="none", task_type="CAUSAL_LM",
                                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                                         "gate_proj", "up_proj", "down_proj"]))
            try:
                _b, _a = dc["batch"], dc["grad_accum"]
                dpo_trainer = build_dpo_trainer(_b, _a)
                mem_report("before DPO")
                while True:   # adaptive: on CUDA OOM rebuild with half the micro-batch, double grad-accum, resume
                    try:
                        dpo_trainer.train(resume_from_checkpoint=bool(sorted((DPO_DIR / "checkpoints").glob("checkpoint-*"))))
                        break
                    except torch.cuda.OutOfMemoryError:
                        if _b == 1:
                            raise
                        del dpo_trainer; gc.collect(); torch.cuda.empty_cache()
                        _b, _a = _b // 2, _a * 2
                        print(f"[mem] DPO out of memory → micro-batch {_b} × grad-accum {_a} (effective batch unchanged)")
                        dpo_trainer = build_dpo_trainer(_b, _a)
                (DPO_DIR / "trainer_log_history.json").write_text(json.dumps(dpo_trainer.state.log_history, indent=1))
                merged = dpo_trainer.model.merge_and_unload()
                merged.save_pretrained(DPO_DIR / "merged", safe_serialization=True); TOK.save_pretrained(DPO_DIR / "merged")
            except Exception as e:
                print("DPO ablation failed; continuing without it:", repr(e)[:500])
            finally:
                free_gpu("after DPO")   # always release the kernel's GPU memory before the vLLM evaluations
    mark_stage("DPO")

## 13) Post-RL evaluation: every model, every benchmark, same items (replaces v1 13, 31–39.5)
Models evaluated:
* the base model
* the R1-Distill reference
* v1 GRPO
* SFT-v2
* **GRPO ours**
* GRPO judge
* GRPO std. length (R3)
* DPO
* GRPO ours, seeds 2 and 3 (seed 3 trains in S9)
* the S8 mechanism controls C1–C3 (group std without the bonus, no std scaling, group std with the bonus ×0.1)
* the second base model (Qwen2.5-1.5B-Instruct: base, SFT, R1, R1 group-std; evaluated in 11g as soon as each exists)
* **Thinkless-1.5B** and **AdaptThink-1.5B** (prior work; evaluated in 5.8 with their published checkpoints)

Each model is evaluated on **GSM8K, MATH-500 and AIME-2026**. SFT-v2 and GRPO-ours also get forced-think and forced-short runs. All conditions for one model share a single model load. A missing checkpoint is reported and skipped, never filled with a made-up number.

In [ ]:
# 13) All models × all benchmarks (cached; one vLLM load per model)
FINAL_PLAN = [("base", "single"), ("r1distill", "single"), ("v1_grpo", "single"), ("sft_v2", "single"),
              ("sft_v2", "forced_think"), ("sft_v2", "forced_short"), ("grpo_ours", "single"),
              ("grpo_ours", "forced_think"), ("grpo_ours", "forced_short"), ("grpo_judge", "single"),
              ("grpo_stdlen", "single"), ("dpo_v2", "single"), ("grpo_ours_s2", "single"), ("grpo_ours_s3", "single"),
              ("grpo_balanced", "single"), ("grpo_balanced", "forced_think"), ("grpo_balanced", "forced_short"),
              ("grpo_ours_step60", "single"), ("grpo_ours_gstd", "single"), ("grpo_ours_gstd", "forced_think"),
              ("grpo_ours_gstd", "forced_short"), ("grpo_stdlen_gstd", "single"),
              ("grpo_nobonus_gstd", "single"), ("grpo_ours_nostd", "single"), ("grpo_ours_gstd_b01", "single"),
              ("thinkless", "single"), ("adaptthink", "single"),
              # second base model (S10–S11): evaluated in 11g as soon as each checkpoint exists; read from cache here
              ("m2_base", "single"), ("m2_sft", "single"), ("m2_sft", "forced_think"), ("m2_sft", "forced_short"),
              ("m2_grpo", "single"), ("m2_grpo", "forced_think"), ("m2_grpo", "forced_short"), ("m2_grpo_gstd", "single")]
def available(key):
    s = MODELS[key]
    if key in EXTERNAL:          # prior-work checkpoints are evaluated once, in 5.8; never re-tried here
        return (EVAL / "preds" / f"{key}__single.jsonl").exists()
    if key == "dpo_v2":
        return (RUNS / "dpo_v2" / "merged" / "config.json").exists()
    return s["lora"] is None or (Path(s["lora"]) / "adapter_config.json").exists()
plan = defaultdict(list)
for key, cond in FINAL_PLAN:
    plan[key].append(cond)
EVAL_NOW = STAGES["P7_final"] or STAGES.get("P7_eval")   # S5: early eval of the trained models (before R2)
for key, conds in plan.items():
    if not available(key):
        print(f"skip {key}: checkpoint not available"); continue
    have = [c for c in conds if (EVAL / "preds" / f"{key}__{c}.jsonl").exists()]
    if EVAL_NOW or have:
        eval_model(key, conds if EVAL_NOW else have, difficulty=(key == "base"))
if STAGES.get("P7_eval") and not STAGES["P7_final"]:
    mark_stage("P7_eval_early")
ALL = pd.concat([pd.DataFrame(read_jsonl(p)) for p in sorted((EVAL / "preds").glob("*__*.jsonl"))
                 if p.name.split("__")[0] in MODELS], ignore_index=True)   # model predictions only (not judge_offline__*, multi__*, mpv*)
_stale = ~ALL.pid.isin(set(BENCH.pid))       # cached predictions for items no longer frozen (e.g. old ARC rows)
if _stale.any():
    print(f"ignoring {_stale.sum()} cached predictions for items not in BENCH:", ALL[_stale].bench.value_counts().to_dict())
    ALL = ALL[~_stale].reset_index(drop=True)
ALL["repetition"] = ALL["text"].map(common.has_repetition_loop)   # loop detector v2 (2026-10-09) on cached rows too
ALL["chose_think"] = [common.chose_think(m, t, MODELS[k]["style"] == "r1")     # mode decision incl. cut-off thinking
                      for k, m, t in zip(ALL.model, ALL["mode"], ALL["text"])]
ALL.drop(columns=["text"]).to_parquet(EXP / "all_predictions_scored.parquet")
print(ALL.groupby(["model", "condition"]).size())

## 12b) The v1 multi-reasoning workflow, kept as an honest ablation (C4)
The workflow is v1 cell 35: initial → 3 sequential reflections → synthesis. The DeepSeek rewriter is removed, so only the policy's own answer is scored. It runs batched in vLLM, and we report accuracy, tokens, and paired **rescues** (wrong → right) and **damages** (right → wrong). v1's strict evaluation found no gain at 4× the cost.

In [ ]:
# 12b) Multi-reasoning workflow (v1 cell 35 logic, batched, policy-only)
REFLECTION_PERSPECTIVES = [
    "Could this be solved a completely different way? Explore an alternative approach.",
    "What are the edge cases or failure modes of the current solution?",
    "How would the outcome change if we reframe the problem from first principles?"]
def mr_answer_text(raw, fmt):
    mode, region, think = common.classify(raw, fmt)
    return region if region else raw.strip()
MPV_BENCHES = ["gsm8k", "math500", "aime2026"]          # the three datasets of 12b/12c
def case_items(bench, n):
    """Fixed, hash-ordered subset (identical for every model/method → paired comparisons)."""
    x = BENCH[BENCH.bench == bench].copy()
    x["_order"] = x.pid.map(lambda p: common.text_hash("case:" + p))
    return x.sort_values("_order").head(n).drop(columns="_order")

def run_multi(key):
    out_file = EVAL / "preds" / f"multi__{key}.jsonl"
    if out_file.exists():
        return pd.DataFrame(read_jsonl(out_file))
    spec = MODELS[key]
    items = pd.concat([case_items(b, CFG["mpv_n"]) for b in MPV_BENCHES])
    def prompt_for(user):
        return common.build_prompt(tok_for(spec), user, spec["style"])
    states = {r.pid: {"q": r.question, "gold": r.gold, "bench": r.bench, "stages": []} for r in items.itertuples()}
    plan = [("initial", 600)] + [(f"reflection_{i+1}", 600) for i in range(3)] + [("synthesis", 1024)]
    for si, (stage, cap) in enumerate(plan):
        reqs = []
        for pid, s in states.items():
            if stage == "initial":
                user = s["q"]
            elif stage.startswith("reflection"):
                prior = mr_answer_text(s["stages"][-1], spec["fmt"])
                user = (f"Original problem: {s['q']}\n\nA previous solution concluded:\n{prior}\n\n"
                        f"Now reconsider from a different angle: {REFLECTION_PERSPECTIVES[si-1]}\n"
                        "Provide your response in the appropriate format.")
            else:
                block = "\n\n---\n\n".join(f"Perspective {i+1}:\n{r}" for i, r in enumerate(s["stages"]))
                user = (f"Problem: {s['q']}\n\nBelow are multiple reasoning perspectives on this problem:\n\n{block}\n\n"
                        "Synthesize these perspectives and provide the final correct solution.")
            reqs.append(dict(id=pid, prompt=prompt_for(user), n=1, temperature=0.0, max_tokens=cap))
        outs, meta = generate(f"multi_{key}_{stage}", spec["model"], reqs, lora=spec["lora"])
        for o in outs:
            states[o["id"]]["stages"].append(o["text"])
            states[o["id"]].setdefault("tokens", 0)
            states[o["id"]]["tokens"] += o["n_tokens"]
    rows = []
    for pid, s in states.items():
        init = common.score_text(s["stages"][0], s["gold"], s["bench"], spec["fmt"])["correct"]
        final = common.score_text(s["stages"][-1], s["gold"], s["bench"], spec["fmt"])["correct"]
        oracle = any(common.score_text(t, s["gold"], s["bench"], spec["fmt"])["correct"] for t in s["stages"][:-1])
        rows.append(dict(model=key, pid=pid, bench=s["bench"], initial_correct=init, final_correct=final,
                         candidate_oracle=oracle, total_tokens=s["tokens"], calls=len(s["stages"])))
    write_jsonl(out_file, rows)
    return pd.DataFrame(rows)
MULTI = []
for key in CFG["multi_models"]:
    if available(key) and (STAGES["P7_final"] or (EVAL / "preds" / f"multi__{key}.jsonl").exists()):
        MULTI.append(run_multi(key))
MULTI = pd.concat(MULTI) if MULTI else pd.DataFrame()
if STAGES["P7_final"]:
    mark_stage("P7_final_eval")

## 12c) ★ Multi-perspective reasoning with **memory** and **validation** (MPV), plus **adaptive MPV**
The model solves each problem in **several different ways**, **validates** every attempt, and writes **one final best solution** from a memory of all of them. ✅ FIX vs v1: v1 chained its reflections, so one early mistake propagated into every later pass. The strict evaluation measured no gain from that. MPV removes the chaining and adds explicit validation and selection.

| Stage | What happens | Calls / problem |
|---|---|---|
| 1. Attempts | **Independent** attempts from 4 perspectives (direct · alternative method · first principles · solve-and-verify), T=0.7. 8 are sampled (perspectives cycled). MPV uses the first 4; the other 4 provide the **compute-matched majority@8** baseline. | 4 (+4) |
| 2. Validation | **Independent re-derivation:** the model is told attempt k's answer, re-solves the problem itself and boxes its own answer. Verdict = CORRECT if they match (an explicit `VERDICT:` line is honoured). The math-specialised model follows "solve" far better than "judge" instructions (pilot). | 4 |
| 3. Selection | Equivalent answers are grouped; score = votes + passed validations (**no gold answer used**) | 0 |
| 4. Synthesis | A memory of all attempts, answers and verdicts comes first and the problem **last**; the model writes the final solution. If it gives no final answer, MPV returns the selected answer (rate reported). | 1 |

**★ Adaptive MPV (adaptive depth *across* attempts, mirroring `<think>`/`<short>` *within* an answer).** Compute is spent only when needed:
* **Early-stop MPV:** if attempts 1 and 2 agree, answer immediately (2 calls). Otherwise run full MPV (9 calls).
* **Mode-routed MPV:** if the model answered the single pass in `<short>` mode (it judged the problem easy), keep that answer. If it chose `<think>`, escalate to MPV.

These policies are computed from the **same saved attempts**, so their accuracy and token costs are exact, not estimated.

**Ablations on the same problems:** single pass · first attempt · majority@4 · **majority@8 (compute-matched)** · selection only · full MPV · adaptive MPV · mode-routed MPV · v1 sequential (12b) · oracle@4 (an upper bound, not a method).

**Data:** GSM8K, MATH-500 and AIME-2026, with 100 problems each (AIME has 30). The first 30 per dataset form the requested per-problem solvability tables.

In [ ]:
# 12c) MPV runner: 3 batched vLLM stages per model (attempts → validation → synthesis), cached per model
MPV_MODELS = [k for k in CFG["mpv_models"] if available(k)]
MPV_TAG = "mpv3"   # v3: re-derivation validation, problem-last synthesis + fallback (older caches ignored, never mixed)
def run_mpv(key):
    out_file = EVAL / "preds" / f"{MPV_TAG}__{key}.jsonl"
    if out_file.exists():
        return pd.DataFrame(read_jsonl(out_file))
    spec, K, KV = MODELS[key], len(common.PERSPECTIVES), CFG["mpv_vote_k"]
    items = pd.concat([case_items(b, CFG["mpv_n"]) for b in MPV_BENCHES])
    P = lambda user: common.build_prompt(tok_for(spec), user, spec["style"])
    # 1) KV independent attempts (perspective k mod K, distinct seeds); the first K feed MPV, all KV feed majority@KV
    reqs = [dict(id=f"{r.pid}|{k}", prompt=P(common.mpv_attempt_question(r.question, k % K)), n=1,
                 temperature=CFG["mpv_temperature"], top_p=0.95, max_tokens=CFG["mpv_max_tokens"], seed=CFG["seed"] + k)
            for r in items.itertuples() for k in range(KV)]
    A = {o["id"]: o for o in generate(f"{MPV_TAG}_{key}_attempts", spec["model"], reqs, lora=spec["lora"])[0]}
    ANS = {i: common.extract_final(o["text"], spec["fmt"])[1] for i, o in A.items()}
    # 2) self-validation of the first K attempts by INDEPENDENT RE-DERIVATION (told the candidate's answer)
    reqs = [dict(id=f"{r.pid}|{k}", n=1, temperature=0.0, max_tokens=CFG["mpv_verify_tokens"],
                 prompt=P(common.VALIDATE_TEMPLATE.format(q=r.question, ans=ANS[f"{r.pid}|{k}"] or "unclear",
                                                          sol=A[f"{r.pid}|{k}"]["text"][-1200:])))
            for r in items.itertuples() for k in range(K)]
    V = {o["id"]: o for o in generate(f"{MPV_TAG}_{key}_validate", spec["model"], reqs, lora=spec["lora"])[0]}
    # 3) selection (no gold) + 4) synthesis prompt with the memory of all attempts
    state, reqs = {}, []
    for r in items.itertuples():
        texts = [A[f"{r.pid}|{k}"]["text"] for k in range(KV)]
        answers = [ANS[f"{r.pid}|{k}"] for k in range(KV)]
        correct = [bool(common.score_text(t, r.gold, r.bench, spec["fmt"])["correct"]) for t in texts]
        vv = [common.validation_verdict(V[f"{r.pid}|{k}"]["text"], answers[k], spec["fmt"]) for k in range(K)]
        verdicts, verdict_how = [v for v, _ in vv], [h for _, h in vv]
        sel, maj, groups = common.mpv_select(answers[:K], verdicts)
        _, maj_all, _ = common.mpv_select(answers, [None] * KV)
        memory = "\n\n".join(
            f"Attempt {k+1} ({common.PERSPECTIVES[k][0]}): final answer = {answers[k] if answers[k] is not None else 'none'}; "
            f"self-check = {verdicts[k] or 'unclear'}\nKey steps (excerpt): …{texts[k][-300:]}" for k in range(K))
        if sel is not None:
            g = next(g for g in groups if sel in g["members"])
            n_ok = sum(verdicts[i] == "CORRECT" for i in g["members"])
            selection = (f"The answer {answers[sel]} is supported by {len(g['members'])} of {K} attempts and passed "
                         f"{n_ok} self-checks.")
        else:
            selection = "No attempt produced a clear final answer."
        reqs.append(dict(id=r.pid, n=1, temperature=0.0, max_tokens=CFG["mpv_max_tokens"],
                         prompt=P(common.SYNTH_TEMPLATE.format(q=r.question, k=K, memory=memory, selection=selection))))
        state[r.pid] = dict(r=r, answers=answers, verdicts=verdicts, verdict_how=verdict_how, correct=correct,
                            sel=sel, maj=maj, maj_all=maj_all,
                            att_tok=[A[f"{r.pid}|{k}"]["n_tokens"] for k in range(KV)],
                            val_tok=[V[f"{r.pid}|{k}"]["n_tokens"] for k in range(K)])
    rows = []
    for o in generate(f"{MPV_TAG}_{key}_synthesis", spec["model"], reqs, lora=spec["lora"])[0]:
        s = state[o["id"]]; r = s["r"]; c = s["correct"]
        fin = common.score_text(o["text"], r.gold, r.bench, spec["fmt"])
        fallback = fin["final_answer"] is None and s["sel"] is not None   # synthesis gave no answer → selected answer
        mpv_ok = bool(c[s["sel"]]) if fallback else bool(fin["correct"])
        judged = [(v, x) for v, x in zip(s["verdicts"], c[:K]) if v is not None]
        rows.append(dict(model=key, pid=r.pid, bench=r.bench, level=r.level, gold=r.gold,
                         attempt_answers_all=s["answers"], attempt_correct_all=c, attempt_tokens_list=s["att_tok"],
                         validate_tokens_list=s["val_tok"], verdicts=s["verdicts"],
                         attempt_answers=s["answers"][:K], attempt_correct=c[:K],
                         first_attempt_correct=c[0],
                         majority_correct=bool(s["maj"] is not None and c[s["maj"]]),
                         majority_all_correct=bool(s["maj_all"] is not None and c[s["maj_all"]]),
                         select_correct=bool(s["sel"] is not None and c[s["sel"]]),
                         mpv_correct=mpv_ok, mpv_answer=(s["answers"][s["sel"]] if fallback else fin["final_answer"]),
                         mpv_mode=fin["mode"], synth_fallback=fallback, synth_correct=bool(fin["correct"]),
                         verdict_how=s["verdict_how"],
                         oracle_correct=any(c[:K]),
                         validator_accuracy=(np.mean([(v == "CORRECT") == x for v, x in judged]) if judged else np.nan),
                         verdict_parse_rate=len(judged) / K,
                         attempt_tokens=sum(s["att_tok"][:K]), validate_tokens=sum(s["val_tok"]), synth_tokens=o["n_tokens"],
                         mpv_total_tokens=sum(s["att_tok"][:K]) + sum(s["val_tok"]) + o["n_tokens"], mpv_calls=2 * K + 1,
                         majority_all_tokens=sum(s["att_tok"]), synthesis_text=o["text"]))
    write_jsonl(out_file, rows)
    return pd.DataFrame(rows)

MPV = []
MPV_EARLY = bool(STAGES.get("P7_mpv")) and not STAGES["P7_final"]   # S6: MPV early on the models that exist now
for key in MPV_MODELS:
    if (STAGES["P7_final"] or (MPV_EARLY and key in CFG["mpv_early_models"])
            or (EVAL / "preds" / f"{MPV_TAG}__{key}.jsonl").exists()):
        MPV.append(run_mpv(key))
MPV = pd.concat(MPV, ignore_index=True) if MPV else pd.DataFrame()
if STAGES["P7_final"] and len(MPV):
    mark_stage("MPV")
elif MPV_EARLY:
    mark_stage("MPV_early")             # S7 reuses these caches
print(MPV.groupby(["model", "bench"])[["first_attempt_correct", "majority_correct", "majority_all_correct",
                                       "select_correct", "mpv_correct", "oracle_correct"]].mean().round(3)
      if len(MPV) else "MPV not run yet")

## 12d) Table 3c: the R2 judge on the policies' own outputs (S9)
R2 trained against the 1.5B judge for 200 steps without reward hacking, although Table 3 measured recall ≈59% and FPR ≈15% on base-model outputs. Here the **same judge, prompt, corrected parser and 16-token verdict as R2's reward** grades every greedy GSM8K + MATH-500 answer of the base model, R1 (verifier reward) and R2 (judge reward) against the strict gold labels. This shows how reliable the judge is on the distribution it actually rewarded.
* The classes keep their natural balance; Table 3 sampled 50/50 wrong/right.
* The judge sees the gold *answer* as its reference, as in Table 3. R2's training prompts carried the full reference solution.
* `judge_reward_mean` is R2's correctness reward: the judge score, or 0.5 when there is no valid score.

In [ ]:
# 12d) Table 3c: R2's judge (same prompt, corrected parser, 16-token verdict) on the policies' own greedy outputs
ONPOLICY_JUDGE = ["base", "grpo_ours", "grpo_judge"]
ONPOLICY_LABEL = f"R1-Distill-1.5B judge, {CFG['grpo']['judge_max_new_tokens']} tokens, corrected parser (R2's reward)"
def onpolicy_file(key):
    return EVAL / "preds" / f"judge_onpolicy__{key}.jsonl"   # not a MODELS key, so never read into ALL

todo = [k for k in ONPOLICY_JUDGE
        if not onpolicy_file(k).exists() and (EVAL / "preds" / f"{k}__single.jsonl").exists()]
if STAGES.get("P3_judge_onpolicy") and todo:
    by_pid = BENCH.set_index("pid")
    items = {}
    for k in todo:
        d = pd.DataFrame(read_jsonl(EVAL / "preds" / f"{k}__single.jsonl"))
        items[k] = d[(d.kind == "greedy") & d.bench.isin(["gsm8k", "math500"])].reset_index(drop=True)
    reqs = [dict(id=f"{k}|{i}", prompt=common.judge_prompt(by_pid.loc[r.pid].question, by_pid.loc[r.pid].gold, r.text),
                 n=1, temperature=0.0, max_tokens=CFG["grpo"]["judge_max_new_tokens"])
            for k, d in items.items() for i, r in enumerate(d.itertuples())]
    outs, _ = generate("judge_onpolicy_" + "+".join(todo), CFG["judge_model"], reqs)   # one judge load for all policies
    jrows = defaultdict(list)
    for o in outs:
        k, i = o["id"].split("|")
        r = items[k].iloc[int(i)]
        jrows[k].append(dict(judge=ONPOLICY_LABEL, policy=k, pid=r.pid, bench=r.bench, gold_correct=bool(r.correct),
                             status=r.status, has_final_answer=r.final_answer is not None, cand_tokens=r.n_tokens,
                             judge_truncated=o["finish_reason"] == "length", **judge_scores(o["text"])))
    for k in todo:
        write_jsonl(onpolicy_file(k), jrows[k])
    mark_stage("P3_judge_onpolicy")
missing = [k for k in ONPOLICY_JUDGE if not onpolicy_file(k).exists()]
if missing:
    print("Table 3c: no judge verdicts yet for", missing, "(stage P3_judge_onpolicy, S9; needs their greedy predictions)")

JO = [pd.DataFrame(read_jsonl(onpolicy_file(k))) for k in ONPOLICY_JUDGE if onpolicy_file(k).exists()]
if JO:
    JO = pd.concat(JO, ignore_index=True)
    rows = []
    for k in [k for k in ONPOLICY_JUDGE if k in set(JO.policy)]:
        x = JO[JO.policy == k]
        for subset, y in [("all", x)] + [(f"bench={b}", y) for b, y in x.groupby("bench")]:
            m = judge_metrics(y, col="judge_score_fixed")
            rows.append(dict(policy=MODELS[k]["label"], subset=subset, n=m["n"], strict_acc=y.gold_correct.mean(),
                             judge_accept=(y.judge_score_fixed >= 0.7).mean(),
                             judge_reward_mean=y.judge_score_fixed.mean(),
                             reward_minus_strict=y.judge_score_fixed.mean() - y.gold_correct.mean(),
                             n_wrong=m["FP"] + m["TN"], **{c: m[c] for c in ("precision", "recall", "FPR", "FPR_ci_low",
                                                                              "FPR_ci_high", "kappa", "no_valid_score")}))
    T3C = pd.DataFrame(rows)
    show(T3C); T3C.to_csv(TAB / "table3c_judge_onpolicy.csv", index=False)
    T3C.round(3).to_latex(TAB / "table3c_judge_onpolicy.tex", index=False, label="tab:judge_onpolicy",
                          caption="R2's judge reward (R1-Distill-1.5B, corrected parser, 16-token verdict, score "
                                  "$\\geq$ 0.7 = correct) on every greedy GSM8K and MATH-500 answer of the base model, R1 "
                                  "and R2, against strict gold labels. FPR with Wilson 95\\% CI; natural class balance.")
    if len(globals().get("JUDGE_COMPARE", [])):
        ref = JUDGE_COMPARE[(JUDGE_COMPARE.subset == "all") & (JUDGE_COMPARE.parser == "corrected parser")]
        for r in ref.itertuples():
            print(f"Table 3 (offline, base outputs, 50/50, {r.judge}): FPR {r.FPR:.3f}, recall {r.recall:.3f}, "
                  f"no valid score {r.no_valid_score:.2f}")
    for r in T3C[T3C.subset == "all"].itertuples():
        print(f"Table 3c {r.policy:40} n={r.n} strict {r.strict_acc:.3f} | judge reward {r.judge_reward_mean:.3f} "
              f"({r.reward_minus_strict:+.3f}) | FPR {r.FPR:.3f} [{r.FPR_ci_low:.2f},{r.FPR_ci_high:.2f}] of {r.n_wrong} "
              f"wrong | recall {r.recall:.3f} | κ {r.kappa:+.2f} | no valid score {r.no_valid_score:.2f}")

## 12e) Majority vote on the full benchmarks (S9)
MPV ran on 100 + 100 + 30 problems. That showed a +5 to +10 point gain from voting, but with too few problems to reach significance. Here the base model and R1 each answer **every** GSM8K and MATH-500 problem **8 times independently**:
* same prompt as the evaluation, T = 0.7, no repetition penalty;
* strict scorer on every sample.

AIME reuses the 16 samples from 13). Table 18 (40.4) turns these samples into:
* majority@k for k = 1…8, with tied votes getting fractional credit;
* a **breadth vs depth** comparison at matched tokens: a vote over k short answers against one long think answer (the model's own forced think, and AdaptThink / Thinkless / R1-Distill), on the same problems.

In [ ]:
# 12e) k independent samples per problem on the FULL GSM8K + MATH-500 for the vote models (S9)
def vote_file(key):
    return EVAL / "preds" / f"vote{CFG['vote_k']}__{key}.jsonl"   # not a MODELS key, so never read into ALL

def run_votes(key):
    spec = MODELS[key]
    items = BENCH[BENCH.bench.isin(["gsm8k", "math500"])]
    reqs = [dict(id=f"{r.pid}|vote", prompt=model_prompt(spec, r.question), n=CFG["vote_k"],
                 temperature=CFG["mpv_temperature"], top_p=0.95, max_tokens=spec.get("max_tokens", CFG["eval_max_tokens"]),
                 seed=CFG["seed"] + 7) for r in items.itertuples()]
    outs, _ = generate(f"vote{CFG['vote_k']}_{key}", spec["model"], reqs, lora=spec["lora"],
                       max_model_len=spec.get("max_model_len", CFG["context"]))
    by_pid = BENCH.set_index("pid")
    rows = []
    for o in outs:
        pid = o["id"].split("|")[0]
        b = by_pid.loc[pid]
        s = common.score_text(o["text"], b.gold, b.bench, spec["fmt"])
        rows.append(dict(model=key, pid=pid, bench=b.bench, level=b.level, sample=o["sample"], correct=bool(s["correct"]),
                         final_answer=s["final_answer"], mode=common.classify(o["text"], spec["fmt"])[0],
                         n_tokens=o["n_tokens"], truncated=o["finish_reason"] == "length"))
    write_jsonl(vote_file(key), rows)

todo = [k for k in CFG["vote_models"] if available(k) and not vote_file(k).exists()]
if STAGES.get("P7_votes") and todo:
    for k in todo:
        run_votes(k)
    mark_stage("P7_votes")
for k in CFG["vote_models"]:
    if vote_file(k).exists():
        v = pd.DataFrame(read_jsonl(vote_file(k)))
        print(f"{k}: {v.pid.nunique()} problems × {int(v.groupby('pid').size().max())} samples | sample accuracy "
              + ", ".join(f"{b} {x.correct.mean():.3f}" for b, x in v.groupby("bench"))
              + f" | mean tokens {v.n_tokens.mean():.0f}")
    else:
        print(f"{k}: no full-benchmark votes yet (stage P7_votes, S9)")

## 12) Adaptive inference: one pass, the model picks its own mode (+ 32 latency, 35 context switching)
These are your v1 demo questions. Each one gets a **single generation** with no reflection loop and no second model. The printout shows the chosen mode, the token count and the latency. Plain transformers is used here, so the latency is an honest single-request number.

In [ ]:
# 12) + 32) + 35) Single-pass demo, single-request latency, context-switching test (transformers, in-kernel)
DEMO_KEY = "grpo_ours" if available("grpo_ours") else "sft_v2"
if STAGES["P10_demo"] and available(DEMO_KEY):
    free_gpu("before demo")
    try:
        from transformers import AutoModelForCausalLM
        from peft import PeftModel
        _m = AutoModelForCausalLM.from_pretrained(CFG["policy_base"], dtype=torch.bfloat16, attn_implementation="sdpa").cuda()
        _m = PeftModel.from_pretrained(_m, str(MODELS[DEMO_KEY]["lora"])).merge_and_unload().eval()

        @torch.inference_mode()
        def ask(prompt_text, max_new=CFG["demo_max_new_tokens"]):
            enc = TOK(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")
            torch.cuda.synchronize(); t0 = time.perf_counter()
            out = _m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=TOK.pad_token_id)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            gen = out[0, enc["input_ids"].shape[1]:]
            return TOK.decode(gen, skip_special_tokens=True), int(gen.shape[0]), dt

        V1_DEMO = [
            "A number is randomly selected from the $[0,1]$ interval. What is the probability that the digit 5 appears among the first $n$ digits of the selected number in its decimal form?",
            "Lois has 40 books. She gives a fourth of her books to her nephew. From her remaining books, she donates a third of her books to the library. Then she purchases x new books from the book store. Lois now has 23 books. What is the value of unknown variable x?",
            "What is the value of x if 2x + 5 = 15?",
            "What is the value of x if 2x + 5 = 15? show me step by step solution",
            "Q: Sammy wanted to go to where the people were. Where might he go?\nOptions:\n(a) race track\n(b) populated areas\n(c) desert\n(d) apartment\n(e) roadblock",
            "Why is the sky blue?"]
        for q in V1_DEMO[:CFG["demo_n"]]:
            text, n, dt = ask(common.build_prompt(TOK, q, "policy"))
            mode, ans, _ = common.extract_final(text, "policy")
            print(f"\n=== mode={mode} | {n} tokens | {dt:.1f}s | answer={ans}\nQ: {q[:90]}\n{text[:600]}")

        # 32) single-request latency on GSM8K items (honest serving numbers, not batched throughput)
        lat = []
        for q in BENCH[BENCH.bench == "gsm8k"].question.head(CFG["latency_n"]):
            text, n, dt = ask(common.build_prompt(TOK, q, "policy"))
            lat.append(dict(tokens=n, seconds=dt, tok_per_s=n / dt, mode=common.parse_policy_output(text)[0]))
        LAT = pd.DataFrame(lat); show(LAT.describe().round(2)); LAT.to_csv(TAB / "table_latency_single_request.csv", index=False)

        # 35) context switching: similar vs distant domains, with and without conversation memory
        similar = ["What is the derivative of x^2?", "Solve for x: 3x + 7 = 22.",
                   "What is the probability of rolling a 6 on a fair six-sided die?",
                   "If a triangle has base 4 and height 5, what is its area?", "Calculate the integral of 2x with respect to x."]
        distant = ["Solve for x: 2x - 4 = 10.", "What is Newton's second law of motion?", "How does photosynthesis work in plants?",
                   "Who was the first president of the United States?", "Write a Python function to reverse a string."]
        ctx_rows = []
        for name, qs, memory in (("similar+memory", similar, True), ("distant+memory", distant, True), ("distant+no-memory", distant, False)):
            msgs = [{"role": "system", "content": common.POLICY_SYSTEM}]
            for q in qs[:CFG["ctx_n"]]:
                turn = (msgs if memory else msgs[:1]) + [{"role": "user", "content": q}]
                text, n, dt = ask(TOK.apply_chat_template(turn, tokenize=False, add_generation_prompt=True), 512)
                if memory:
                    msgs = turn + [{"role": "assistant", "content": text}]
                ctx_rows.append(dict(test=name, query=q, mode=common.parse_policy_output(text)[0], tokens=n,
                                     seconds=round(dt, 1), response=text[:200]))
        CTX = pd.DataFrame(ctx_rows); show(CTX[["test", "query", "mode", "tokens", "seconds"]])
        CTX.to_csv(EXP / "context_switching.csv", index=False)

    except torch.cuda.OutOfMemoryError:
        print("[mem] demo skipped: out of memory (it is qualitative only; results are unaffected)")
    finally:
        free_gpu("after demo")      # drops _m and releases the GPU even if the demo failed
    mark_stage("demo_latency_context")

## 15–16) Reasoning comparison vs DeepSeek-R1-Distill-1.5B (replaces v1's similarity cells)
✅ FIX: v1 compared reasoning by MiniLM *semantic similarity* to DeepSeek's text. That measures wording, not correctness. Here we compare on the **same benchmark items**: strict accuracy, who solves what, final-answer agreement, and how many tokens each model spends.

In [ ]:
# 15–16) Same-item comparison: GRPO-ours (or SFT) vs R1-Distill reference
CMP_KEY = "grpo_ours" if "grpo_ours" in set(ALL.model) else "sft_v2"
rows = []
for b in ("gsm8k", "math500", "aime2026"):
    pick = lambda k: ALL[(ALL.model == k) & (ALL.condition == "single") & (ALL.kind == "greedy") & (ALL.bench == b)].set_index("pid")
    a, d = pick(CMP_KEY), pick("r1distill")
    ids = a.index.intersection(d.index)
    if not len(ids):
        continue
    a, d = a.loc[ids], d.loc[ids]
    def same(x, y):
        if x is None or y is None: return False
        rx, ry = common.rational(x), common.rational(y)
        return (rx == ry) if (rx is not None and ry is not None) else str(x).strip() == str(y).strip()
    rows.append(dict(bench=b, n=len(ids), ours=CMP_KEY, ours_acc=a.correct.mean(), r1_acc=d.correct.mean(),
                     both_correct=int((a.correct & d.correct).sum()), only_ours=int((a.correct & ~d.correct).sum()),
                     only_r1=int((~a.correct & d.correct).sum()),
                     final_answer_agreement=np.mean([same(x, y) for x, y in zip(a.final_answer, d.final_answer)]),
                     ours_mean_tokens=a.n_tokens.mean(), r1_mean_tokens=d.n_tokens.mean(),
                     ours_think_rate=a.chose_think.mean(), r1_truncated=d.truncated.mean()))
T9 = pd.DataFrame(rows)
if len(T9):
    show(T9); T9.to_csv(TAB / "table9_vs_r1distill.csv", index=False)

## 39.1 / 40.1 / 40.2) Statistics and paper tables (replace v1 Table 1, McNemar and comparison cells)
* **Table 1:** accuracy ± Wilson 95% CI, tokens, think-rate, format, repetition, truncation.
* **Table 1b:** AIME greedy, avg@k, pass@k, maj@k (k = 16 for the main models, 4 otherwise; a tied vote gets fractional credit, so maj@k does not depend on sample order).
* **Table 2:** paired Δ vs GRPO-ours, with bootstrap CI, exact McNemar and Holm correction.
* **Adaptivity:** think-rate by level and by empirical difficulty, over/under-thinking, routing vs the oracle, token savings, and an automatic check of success criteria (a)–(d).
* **Table 4:** energy and CO2 of the study (codecarbon).
* **Tables 3, 5–8:** judge reliability, data, hyperparameters, multi-reasoning, scorer agreement.

Everything is computed from saved predictions, with no default values.

In [ ]:
# 39.1) Table 1 (main results) + Table 1b (AIME sampling metrics)
LABEL = {k: v["label"] for k, v in MODELS.items()}
T1 = acc_table(ALL)
T1["model_label"] = T1.model.map(LABEL)
T1.to_csv(TAB / "table1_main_results.csv", index=False)
def fmt_ci(r):
    return f"{100*r.acc:.1f} [{100*r.ci_low:.1f}, {100*r.ci_high:.1f}]"
pivot = T1.assign(cell=T1.apply(fmt_ci, axis=1)).pivot_table(index=["model_label", "condition"], columns="bench",
                                                             values="cell", aggfunc="first")
display(pivot)
pivot.to_latex(TAB / "table1_main_results.tex", caption="Strict accuracy (\\%) with Wilson 95\\% CI.",
               label="tab:main")

aime = ALL[(ALL.bench == "aime2026") & (ALL.kind == "sampled")]
rows = []
for m, x in aime.groupby("model"):
    per = x.groupby("pid")
    k = int(per.size().max())                # 16 for the main models, 4 for the rest (aime_k)
    maj_ok = []
    for pid, g in per:
        # votes: every correct answer counts as one (equivalent) answer; a tie at the top gives fractional credit
        # (= expected accuracy under random tie-breaking), so maj@k never depends on sample order
        votes = Counter("<correct>" if c else str(a).strip() for a, c in zip(g.final_answer, g.correct)
                        if isinstance(a, str) and a.strip())
        top = max(votes.values()) if votes else 0
        tied = [a for a, v in votes.items() if v == top]
        maj_ok.append(("<correct>" in tied) / len(tied) if tied else 0.0)
    n = per.ngroups
    rows.append(dict(model=LABEL[m], n_problems=n, greedy=ALL[(ALL.model == m) & (ALL.bench == "aime2026") &
                     (ALL.kind == "greedy") & (ALL.condition == "single")].correct.mean(),
                     avg_at_k=x.correct.mean(), pass_at_k=per.correct.any().mean(), maj_at_k=np.mean(maj_ok), k=k))
T1B = pd.DataFrame(rows); show(T1B); T1B.to_csv(TAB / "table1b_aime.csv", index=False)

In [ ]:
# 40.1) Table 2: paired comparisons vs GRPO-ours (bootstrap Δ, exact McNemar, Holm)
def paired(a_key, a_cond, b_key, b_cond, bench):
    g = lambda k, c: ALL[(ALL.model == k) & (ALL.condition == c) & (ALL.bench == bench) & (ALL.kind == "greedy")] \
        .set_index("pid").correct
    a, b = g(a_key, a_cond), g(b_key, b_cond)
    common_ids = a.index.intersection(b.index)
    if len(common_ids) == 0:
        return None
    a, b = a.loc[common_ids].astype(float), b.loc[common_ids].astype(float)
    d, lo, hi = common.paired_bootstrap(a.values, b.values, seed=CFG["seed"])
    a_only, b_only = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    return dict(bench=bench, n=len(common_ids), delta=d, ci_low=lo, ci_high=hi, a_only=a_only, b_only=b_only,
                p_mcnemar=common.mcnemar_exact(a_only, b_only))
COMPARATORS = [("base", "single"), ("sft_v2", "single"), ("v1_grpo", "single"), ("grpo_stdlen", "single"),
               ("grpo_judge", "single"), ("dpo_v2", "single"), ("grpo_ours", "forced_think"),
               ("grpo_ours", "forced_short"), ("r1distill", "single"), ("grpo_ours_s2", "single"),
               ("grpo_balanced", "single"), ("grpo_ours_step60", "single"), ("grpo_ours_gstd", "single"),
               ("grpo_stdlen_gstd", "single"), ("thinkless", "single"), ("adaptthink", "single")]
rows = []
if "grpo_ours" in set(ALL.model):
    for k, c in COMPARATORS:
        for bench in ("gsm8k", "math500"):
            r = paired("grpo_ours", "single", k, c, bench)
            if r:
                rows.append(dict(comparison=f"ours − {LABEL[k]} [{c}]", **r))
T2 = pd.DataFrame(rows)
if len(T2):
    T2["p_holm"] = common.holm(T2.p_mcnemar.tolist())
    show(T2); T2.to_csv(TAB / "table2_paired_comparisons.csv", index=False)
    T2.round(4).to_latex(TAB / "table2_paired_comparisons.tex", index=False, label="tab:paired",
                         caption="Paired accuracy differences (GRPO-ours minus comparator), bootstrap 95\\% CI, exact McNemar with Holm correction.")

In [ ]:
# 40.2) Adaptivity analysis: think-rate vs difficulty, over/under-thinking, routing, criteria (a)–(d)
def adaptivity(key="grpo_ours"):
    S = ALL[(ALL.model == key) & (ALL.condition == "single") & (ALL.kind == "greedy")]
    if S.empty:
        print("no predictions for", key); return None
    m = S[S.bench == "math500"]
    lvl = m.groupby("level")[["chose_think", "correct", "n_tokens"]].apply(lambda x: pd.Series(dict(n=len(x), think_rate=x.chose_think.mean(),
                                   ci=common.wilson(int(x.chose_think.sum()), len(x)), acc=x.correct.mean(),
                                   mean_tokens=x.n_tokens.mean())))
    display(lvl)
    # empirical difficulty from the BASE model's sample4 pass rate
    diff = ALL[(ALL.model == "base") & (ALL.kind == "difficulty")].groupby("pid").correct.mean().rename("base_pass")
    md = m.set_index("pid").join(diff, how="inner")
    md["difficulty_bin"] = pd.cut(md.base_pass, [-0.01, 0.0, 0.25, 0.5, 0.75, 1.0],
                                  labels=["0/4", "1/4", "2/4", "3/4", "4/4"])
    by_diff = md.groupby("difficulty_bin", observed=True)[["chose_think", "correct"]].apply(lambda x: pd.Series(dict(
        n=len(x), think_rate=x.chose_think.mean(), acc=x.correct.mean())))
    display(by_diff)
    # routing vs oracle using forced modes on identical items
    ft = ALL[(ALL.model == key) & (ALL.condition == "forced_think") & (ALL.kind == "greedy")].set_index("pid")
    fs = ALL[(ALL.model == key) & (ALL.condition == "forced_short") & (ALL.kind == "greedy")].set_index("pid")
    res = {}
    if len(ft) and len(fs):
        j = S.set_index("pid").join(ft[["correct", "n_tokens"]].add_prefix("ft_")).join(fs[["correct", "n_tokens"]].add_prefix("fs_"))
        j = j[j.bench.isin(["gsm8k", "math500"])]
        over = (j.chose_think & j.fs_correct).mean()
        under = ((j["mode"] == "short") & ~j.correct & j.ft_correct).mean()
        oracle_mode = np.where(j.fs_correct, "short", np.where(j.ft_correct, "think", "either"))
        routing_acc = np.mean([(o == "either") or (o == md_) for o, md_ in zip(oracle_mode, j["mode"])])
        tok_saving = 1 - j.n_tokens.mean() / j.ft_n_tokens.mean()
        res = dict(overthinking_rate=over, underthinking_rate=under, routing_accuracy_vs_oracle=routing_acc,
                   token_saving_vs_forced_think=tok_saving, acc_single=j.correct.mean(),
                   acc_forced_think=j.ft_correct.mean(), acc_forced_short=j.fs_correct.mean(),
                   oracle_acc=(j.ft_correct | j.fs_correct).mean())
        print(json.dumps({k: round(float(v), 4) for k, v in res.items()}, indent=1))
    # success criteria
    t1 = T1[(T1.condition == "single")].set_index(["model", "bench"])
    crit = {}
    tr = lvl.think_rate.values
    l1 = m[m.level == 1]; l5 = m[m.level == 5]
    from math import sqrt
    p1, p5 = l1.chose_think.mean(), l5.chose_think.mean()
    se = sqrt(p1 * (1 - p1) / max(1, len(l1)) + p5 * (1 - p5) / max(1, len(l5)))
    crit["a_think_rate_rises_L1_to_L5"] = bool(p5 - p1 > 1.96 * se and np.corrcoef(lvl.index, tr)[0, 1] > 0.8)
    ok_b = True
    for b in ("gsm8k", "math500"):
        ours = t1.loc[(key, b)]
        best = max(t1.loc[("base", b)].acc if ("base", b) in t1.index else 0,
                   T1[(T1.model == key) & (T1.condition == "forced_short") & (T1.bench == b)].acc.max())
        ok_b &= ours.ci_high >= best
    crit["b_acc_ge_max(base, forced_short)_within_CI"] = bool(ok_b)
    if res:
        crit["c_tokens_30pct_below_forced_think_noninferior"] = bool(res["token_saving_vs_forced_think"] >= 0.30 and
                                                                     res["acc_single"] >= res["acc_forced_think"] - 0.02)
    sm = S[S.bench.isin(["gsm8k", "math500"])]
    crit["d_format_ge_95_and_repetition_le_2"] = bool(sm.format_ok.mean() >= 0.95 and sm.repetition.mean() <= 0.02)
    print("SUCCESS CRITERIA" + (" (PILOT: not meaningful)" if globals().get("PILOT") else "") + ":",
          json.dumps(crit, indent=1))
    return dict(level=lvl, difficulty=by_diff, routing=res, criteria=crit)
ADAPT = adaptivity("grpo_ours")
if ADAPT:
    ADAPT["level"].to_csv(TAB / "adaptivity_by_level.csv"); ADAPT["difficulty"].to_csv(TAB / "adaptivity_by_difficulty.csv")
    (TAB / "adaptivity_routing_and_criteria.json").write_text(json.dumps(
        {"routing": ADAPT["routing"], "criteria": ADAPT["criteria"]}, indent=1, default=float))

In [ ]:
# 40.3) Tables 4–8: energy/CO2 · data/decontamination · hyperparameters · multi-reasoning · scorer agreement
em = ROOT / "emissions.csv"
if em.exists():
    e = pd.read_csv(em)
    T4 = pd.DataFrame([dict(energy_kwh=float(e.energy_consumed.sum()), co2_kg=float(e.emissions.sum()),
                            source="codecarbon (GPU + CPU + RAM), all completed sessions")])
    show(T4); T4.to_csv(TAB / "table4_energy_carbon.csv", index=False)
    print("(this session's energy is written when the tracker stops; cell 42 rewrites Table 4 with it)")

T5 = pd.DataFrame([{"set": "SFT", **json.loads(SFT_STATS.read_text())}, {"set": "RL/DPO", **json.loads(RL_STATS.read_text())}]) \
     if SFT_STATS.exists() and RL_STATS.exists() else pd.DataFrame()
show(T5); T5.to_csv(TAB / "table5_data_decontamination.csv", index=False)

hp = {"SFT": CFG["sft"], "GRPO": {k: v for k, v in CFG["grpo"].items() if k != "max_hours"},   # no session time limit
      "DPO": CFG["dpo"],
      "rewards": {"correctness": "strict 1/0", "format": "+0.1/-0.5", "repetition": -0.5,
                  "difficulty_aware_mode (R1, R2)": f"{common.MODE_BETA}*(p if short else 1-p) + "
                                                     f"{common.MODE_LEN_ALPHA}*within-mode relative length, if correct",
                  "standard_length (R3)": f"{common.LENGTH_ALPHA}*(1-tokens/{common.L_MAX}) if correct"},
      "eval": {"decoding": "greedy, no repetition penalty", "max_tokens": CFG["eval_max_tokens"],
               "aime": f"{CFG['aime_samples_main']} samples (main models: {CFG['aime_main_models']}), "
                       f"{CFG['aime_samples']} otherwise, T={CFG['aime_temperature']}",
               "prior_work": {k: MODELS[k]["model"] for k in EXTERNAL}}}
for r in ("grpo_ours", "grpo_balanced", "grpo_judge", "grpo_stdlen", "grpo_ours_gstd", "grpo_stdlen_gstd", "grpo_ours_s2",
          "grpo_ours_s3", "grpo_nobonus_gstd", "grpo_ours_nostd", "grpo_ours_gstd_b01", "m2_grpo", "m2_grpo_gstd"):
    p = RUNS / r / "resolved_grpo_config.json"
    if p.exists():
        rc = json.loads(p.read_text())
        hp[f"{r} (resolved)"] = {k: v for k, v in rc["args"].items()
                                 if k in ("learning_rate", "beta", "loss_type", "num_generations", "max_steps",
                                          "max_completion_length", "temperature", "scale_rewards")}
        hp[f"{r} (resolved)"].update(variant=rc.get("variant"), bonus_scale=rc.get("bonus_scale", 1.0))
(TAB / "table6_hyperparameters.json").write_text(json.dumps(hp, indent=1, default=_json_default))
print(json.dumps(hp, indent=1, default=_json_default)[:2500])

if len(MULTI):
    rows = []
    for (m, b), x in MULTI.groupby(["model", "bench"]):
        resc = int((~x.initial_correct & x.final_correct).sum()); dam = int((x.initial_correct & ~x.final_correct).sum())
        single = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy") & ALL.pid.isin(x.pid)]
        rows.append(dict(model=LABEL[m], bench=b, n=len(x), single_pass_acc=single.correct.mean(),
                         single_pass_tokens=single.n_tokens.mean(), initial600_acc=x.initial_correct.mean(),
                         multi_acc=x.final_correct.mean(), multi_tokens=x.total_tokens.mean(), rescued=resc,
                         damaged=dam, p_mcnemar=common.mcnemar_exact(resc, dam),
                         candidate_oracle=x.candidate_oracle.mean()))
    T7 = pd.DataFrame(rows); show(T7); T7.to_csv(TAB / "table7_multi_reasoning.csv", index=False)

g = ALL[(ALL.kind == "greedy") & ALL.second_scorer.notna()]
T8 = g.groupby(["model", "condition", "bench"])[["correct", "second_scorer", "mode", "self_checks"]].apply(lambda x: pd.Series(dict(
    n=len(x), strict_acc=x.correct.mean(), second_scorer_acc=x.second_scorer.astype(bool).mean(),
    agreement=(x.correct == x.second_scorer.astype(bool)).mean(),
    self_check_markers_think=x[x["mode"] == "think"].self_checks.mean(),
    acc_think=x[x["mode"] == "think"].correct.mean(), acc_short=x[x["mode"] == "short"].correct.mean()))).reset_index()
show(T8); T8.to_csv(TAB / "table8_scorer_agreement_traces.csv", index=False)

## 40.4) The paper's central questions, answered from the data
1. **Does GRPO improve on SFT? Is GRPO better than DPO?** Table 12 (training-method ablation): accuracy ± CI for every stage, and **paired Δ vs SFT** with exact McNemar and Holm correction, on the full benchmarks.
2. **How many problems does each model solve?** Table 11: per-problem results on **30 problems** from each of GSM8K, MATH-500 and AIME-2026 ("solved / evaluated"), plus a per-problem CSV for the appendix.
3. **Does multi-perspective reasoning with memory and validation help, beyond simply spending more compute?** Table 10: MPV against single pass, majority@4, **compute-matched majority@8**, selection only, the v1 sequential workflow and the oracle, on the same problems.
4. **Can compute be spent adaptively?** Table 13 + Fig 13: accuracy vs tokens for single pass, majority voting, MPV, **early-stop MPV** and **mode-routed MPV**.
5. **At which token budget does a thinking model overtake our short-answer policy?** Table 15b: paired test of GRPO-ours vs AdaptThink, Thinkless and R1-Distill at every budget (Holm).
6. **Does thinking pay off, and does RL change that?** Table 16 + Fig 17: forced-think vs forced-short on the same problems, by MATH level, for each training stage (SFT → GRPO variants), with paired tests.
7. **Why did group-std GRPO compress its reasoning?** Table 17 + Fig 16: training-log statistics (accuracy, length, entropy, grad-norm, share of all-correct groups) at steps 61–100 vs 161–200, and greedy accuracy, for R1/R3 and the S8 controls (C1 group std without the bonus, C2 no std scaling, C3 bonus ×0.1). Table 17b: paired tests (Holm).
8. **Do several independent attempts beat one, and does breadth beat depth?** Table 18a: single pass vs any-correct@4, majority@4/@8 and the early-stop vote, on the same problems (full GSM8K + MATH-500 for base and R1 from 12e; the MPV subset otherwise). Table 18b + Fig 18: a vote over k short answers vs ONE think answer (forced think, AdaptThink, Thinkless, R1-Distill) at matched tokens.
9. **Does it replicate on a second base model?** Table 19 + Fig 19 (S10–S11): the same recipe on the general Qwen2.5-1.5B-Instruct. Accuracy per stage for both base models, paired tests, think − short by MATH level, and the training dynamics with the verdicts of the two predictions pre-registered in 11g (the more accurate mode wins; group std collapses the CoT, batch std does not).

In [ ]:
# 40.4) Table 10 (MPV), Table 11 (30-sample solvability), Table 12 (training-method ablation), Table 13 (adaptive MPV)
STAGE_MODELS = [m for m in ["base", "sft_long", "sft_v2", "dpo_v2", "grpo_stdlen_gstd", "grpo_stdlen", "grpo_judge", "grpo_ours_step60", "grpo_ours_gstd", "grpo_ours", "grpo_balanced", "grpo_ours_s2", "grpo_ours_s3"]
                if m in set(ALL.model)]
def single_correct(model, bench):
    return ALL[(ALL.model == model) & (ALL.condition == "single") & (ALL.kind == "greedy") &
               (ALL.bench == bench)].set_index("pid").correct.astype(bool)

def paired_test(a, b):
    """a, b: aligned bool arrays (same problems). Returns Δ, CI, wins, losses, exact McNemar p."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    d, lo, hi = common.paired_bootstrap(a, b, seed=CFG["seed"])
    w, l = int((a & ~b).sum()), int((~a & b).sum())
    return d, lo, hi, w, l, common.mcnemar_exact(w, l)

# ---- Table 12: training-method ablation (full benchmarks; paired vs SFT) ----------------------------
rows = []
for b in ("gsm8k", "math500", "aime2026"):
    ref = single_correct("sft_v2", b)
    for m in STAGE_MODELS:
        x = single_correct(m, b)
        if x.empty:
            continue
        lo, hi = common.wilson(int(x.sum()), len(x))
        row = dict(bench=b, model=LABEL[m], n=len(x), acc=x.mean(), ci_low=lo, ci_high=hi)
        if m != "sft_v2" and len(ref):
            ids = x.index.intersection(ref.index)
            d, dlo, dhi, w, l, p = paired_test(x.loc[ids].values, ref.loc[ids].values)
            row.update(delta_vs_sft=d, delta_ci_low=dlo, delta_ci_high=dhi, wins_vs_sft=w, losses_vs_sft=l, p_mcnemar=p)
        rows.append(row)
T12 = pd.DataFrame(rows)
if len(T12) and "p_mcnemar" in T12:
    mask = T12.p_mcnemar.notna()
    T12.loc[mask, "p_holm"] = common.holm(T12.loc[mask, "p_mcnemar"].tolist())
    show(T12); T12.to_csv(TAB / "table12_training_method_ablation.csv", index=False)
    T12.round(4).to_latex(TAB / "table12_training_method_ablation.tex", index=False, label="tab:stage_ablation",
                          caption="Training-method ablation. Strict accuracy with Wilson 95\\% CI; paired difference vs SFT "
                                  "(bootstrap 95\\% CI), wins/losses vs SFT, exact McNemar with Holm correction.")
    for b in ("gsm8k", "math500", "aime2026"):
        g = T12[(T12.bench == b) & (T12.model == LABEL.get("grpo_ours"))]
        if len(g) and "delta_vs_sft" in g and g.delta_vs_sft.notna().any():
            r = g.iloc[0]
            verdict = ("significant improvement" if r.p_holm < 0.05 and r.delta_vs_sft > 0 else
                       "significant degradation" if r.p_holm < 0.05 else "no significant difference")
            print(f"GRPO vs SFT on {b}: Δ={100*r.delta_vs_sft:+.1f} pts [{100*r.delta_ci_low:+.1f}, {100*r.delta_ci_high:+.1f}], "
                  f"Holm p={r.p_holm:.3g} → {verdict}")

# ---- Table 14: seed robustness of R1 (seed 42 = R1; seed 43 = the "auto" seed 2; seed 44 = seed 3, S9) ------------
# Per seed: greedy accuracy; across seeds: mean ± sd. Seed-averaged vs SFT and vs base: per-item mean correctness over
# the seeds minus the reference, paired bootstrap 95% CI (no McNemar: the seed average is not 0/1).
SEED_RUNS = {m: i for i, m in enumerate(("grpo_ours", "grpo_ours_s2", "grpo_ours_s3"), start=1) if m in set(ALL.model)}
if "grpo_ours" in SEED_RUNS and len(SEED_RUNS) >= 2:
    rows = []
    for b in ("gsm8k", "math500", "aime2026"):
        xs = {m: single_correct(m, b) for m in SEED_RUNS}
        if any(x.empty for x in xs.values()):
            continue
        ids = xs["grpo_ours"].index
        for x in xs.values():
            ids = ids.intersection(x.index)
        accs = np.array([xs[m].loc[ids].mean() for m in SEED_RUNS])
        row = dict(bench=b, n=len(ids), n_seeds=len(SEED_RUNS), **{f"seed{i}": a for i, a in zip(SEED_RUNS.values(), accs)},
                   mean=accs.mean(), sd=accs.std(ddof=1))
        for m, i in list(SEED_RUNS.items())[1:]:
            row[f"seed1_vs_seed{i}_p"] = paired_test(xs["grpo_ours"].loc[ids].values, xs[m].loc[ids].values)[5]
        avg = np.mean([xs[m].loc[ids].astype(float).values for m in SEED_RUNS], axis=0)
        for ref_key, tag in (("sft_v2", "sft"), ("base", "base")):
            ref = single_correct(ref_key, b).reindex(ids)
            if ref.notna().all() and len(ref):
                d, lo, hi = common.paired_bootstrap(avg, ref.astype(float).values, seed=CFG["seed"])
                row.update({f"mean_minus_{tag}": d, f"mean_minus_{tag}_ci_low": lo, f"mean_minus_{tag}_ci_high": hi})
        rows.append(row)
    T14 = pd.DataFrame(rows); show(T14); T14.to_csv(TAB / "table14_seed_robustness.csv", index=False)
    T14.round(4).to_latex(TAB / "table14_seed_robustness.tex", index=False, label="tab:seeds",
                          caption="Seed robustness of GRPO (R1): greedy accuracy per training seed, mean ± sd over "
                                  "seeds, seed 1 vs each other seed (paired exact McNemar p), and the seed-averaged "
                                  "accuracy minus SFT and minus the base model (paired bootstrap 95\\% CI).")

# ---- Table 10: MPV vs baselines (incl. compute-matched majority@8) on the same problems ----------------
if len(MPV):
    rows = []
    for (m, b), x in MPV.groupby(["model", "bench"]):
        sc = single_correct(m, b).reindex(x.pid)
        seq = MULTI[(MULTI.model == m) & (MULTI.bench == b)].set_index("pid").final_correct if len(MULTI) else pd.Series(dtype=bool)
        mpv = x.mpv_correct.astype(bool).values
        ok = sc.notna().values
        d, dlo, dhi, w, l, p = paired_test(mpv[ok], sc[ok].astype(bool).values) if ok.any() else (np.nan,) * 6
        d8, _, _, w8, l8, p8 = paired_test(mpv, x.majority_all_correct.astype(bool).values)
        single_tok = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy") & ALL.pid.isin(x.pid)].n_tokens.mean()
        rows.append(dict(model=LABEL[m], bench=b, n=len(x), single=sc.mean(), first_attempt=x.first_attempt_correct.mean(),
                         majority_at_4=x.majority_correct.mean(), majority_at_8=x.majority_all_correct.mean(),
                         selection=x.select_correct.mean(), MPV=x.mpv_correct.mean(),
                         v1_sequential=seq.reindex(x.pid).mean() if len(seq) else np.nan, oracle_at_4=x.oracle_correct.mean(),
                         MPV_minus_single=d, ci_low=dlo, ci_high=dhi, p_vs_single=p,
                         MPV_minus_majority8=d8, p_vs_majority8=p8,
                         validator_accuracy=x.validator_accuracy.mean(), verdict_rate=x.verdict_parse_rate.mean(),
                         synth_fallback_rate=x.synth_fallback.mean() if "synth_fallback" in x else np.nan,
                         tokens_single=single_tok,
                         tokens_MPV=x.mpv_total_tokens.mean(), tokens_majority8=x.majority_all_tokens.mean(),
                         calls_MPV=x.mpv_calls.mean()))
    T10 = pd.DataFrame(rows)
    T10["p_holm_vs_single"] = common.holm(T10.p_vs_single.fillna(1.0).tolist())
    T10["p_holm_vs_majority8"] = common.holm(T10.p_vs_majority8.fillna(1.0).tolist())
    show(T10); T10.to_csv(TAB / "table10_mpv.csv", index=False)
    T10.round(4).to_latex(TAB / "table10_mpv.tex", index=False, label="tab:mpv",
                          caption="Multi-perspective reasoning with memory and validation (MPV) vs single pass, majority "
                                  "voting over 4 and over 8 attempts (compute-matched: about 9 calls), selection without synthesis, "
                                  "the v1 sequential workflow and oracle@4; paired differences with exact McNemar (Holm).")

# ---- Table 13: adaptive test-time compute (exact, from the same saved attempts) ------------------------
def adaptive_policies(x, single_rows):
    """Per problem: (correct, tokens, calls) for every policy. Mode routing needs a tagged single pass."""
    out = defaultdict(list)
    for r in x.itertuples():
        a, c, t, vt = r.attempt_answers_all, r.attempt_correct_all, r.attempt_tokens_list, r.validate_tokens_list
        K = len(vt)
        full = sum(t[:K]) + sum(vt) + r.synth_tokens
        s = single_rows.loc[r.pid] if r.pid in single_rows.index else None
        agree2 = common.answers_equivalent(a[0], a[1])
        if s is not None:
            out["single pass"].append((bool(s.correct), s.n_tokens, 1))
        out["majority@4"].append((r.majority_correct, sum(t[:K]), K))
        out[f"majority@{len(t)}"].append((r.majority_all_correct, sum(t), len(t)))
        out["MPV (full)"].append((r.mpv_correct, full, 2 * K + 1))
        out["early-stop vote"].append((c[0], t[0] + t[1], 2) if agree2 else (r.majority_correct, sum(t[:K]), K))
        out["adaptive MPV (early-stop)"].append((c[0], t[0] + t[1], 2) if agree2 else (r.mpv_correct, full, 2 * K + 1))
        if s is not None and s["mode"] in ("think", "short"):
            out["mode-routed MPV"].append((bool(s.correct), s.n_tokens, 1) if s["mode"] == "short"
                                          else (r.mpv_correct, s.n_tokens + full, 2 * K + 2))
    return out

if len(MPV) and "attempt_tokens_list" in MPV:
    rows = []
    for (m, b), x in MPV.groupby(["model", "bench"]):
        S = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy") & (ALL.bench == b)].set_index("pid")
        pol = adaptive_policies(x, S)
        mpv_full = np.array([v[0] for v in pol["MPV (full)"]], bool)
        for name, vals in pol.items():
            corr = np.array([v[0] for v in vals], bool)
            lo, hi = common.wilson(int(corr.sum()), len(corr))
            row = dict(model=LABEL[m], bench=b, policy=name, n=len(corr), acc=corr.mean(), ci_low=lo, ci_high=hi,
                       mean_tokens=np.mean([v[1] for v in vals]), mean_calls=np.mean([v[2] for v in vals]))
            if name.startswith("adaptive") and len(corr) == len(mpv_full):   # non-inferiority evidence vs full MPV
                d, dlo, dhi, w, l, p = paired_test(corr, mpv_full)
                row.update(delta_vs_full_MPV=d, delta_ci_low=dlo, delta_ci_high=dhi, p_vs_full_MPV=p,
                           token_saving_vs_full_MPV=1 - row["mean_tokens"] / np.mean([v[1] for v in pol["MPV (full)"]]))
            rows.append(row)
    T13 = pd.DataFrame(rows)
    show(T13); T13.to_csv(TAB / "table13_adaptive_test_time_compute.csv", index=False)
    T13.round(4).to_latex(TAB / "table13_adaptive_test_time_compute.tex", index=False, label="tab:adaptive_mpv",
                          caption="Adaptive test-time compute. Accuracy (Wilson 95\\% CI), mean generated tokens and model "
                                  "calls per problem, computed exactly from the same saved attempts.")
elif len(MPV):
    print("Table 13 needs the v2 MPV schema (per-attempt tokens); rerun 12c.")

# ---- Table 11: 30-sample solvability per dataset ("how many of them are correct") ------------------
rows, per_problem = [], []
for b in MPV_BENCHES:
    ids = case_items(b, CFG["case_n"]).pid
    row = dict(dataset=b, problems=len(ids))
    for m in STAGE_MODELS:
        x = single_correct(m, b).reindex(ids)
        if x.notna().any():
            row[f"{LABEL[m]} (single)"] = f"{int(x.dropna().astype(bool).sum())}/{int(x.notna().sum())}"
    for m in (MPV.model.unique() if len(MPV) else []):
        y = MPV[(MPV.model == m) & (MPV.bench == b)].set_index("pid").mpv_correct.reindex(ids)
        if y.notna().any():
            row[f"{LABEL[m]} (MPV)"] = f"{int(y.dropna().astype(bool).sum())}/{int(y.notna().sum())}"
    rows.append(row)
    by_pid = BENCH.set_index("pid")
    for pid in ids:
        rec = dict(dataset=b, pid=pid, question=by_pid.loc[pid, "question"][:160], gold=by_pid.loc[pid, "gold"])
        for m in STAGE_MODELS:
            v = single_correct(m, b).get(pid)
            rec[LABEL[m]] = None if v is None else bool(v)
        for m in (MPV.model.unique() if len(MPV) else []):
            z = MPV[(MPV.model == m) & (MPV.pid == pid)]
            if len(z):
                rec[f"{LABEL[m]} MPV"] = bool(z.mpv_correct.iloc[0])
        per_problem.append(rec)
T11 = pd.DataFrame(rows); show(T11)
T11.to_csv(TAB / "table11_solvability_30.csv", index=False)
T11.to_latex(TAB / "table11_solvability_30.tex", index=False, label="tab:solved30",
             caption=f"Problems solved / problems evaluated, on a fixed hash-ordered subset of {CFG['case_n']} per dataset; single pass and MPV.")
pd.DataFrame(per_problem).to_csv(EXP / "per_problem_30_sample_results.csv", index=False)

# ---- Table 15: accuracy under fixed token budgets (N3; exact for greedy decoding, no extra GPU) ---------------
# Greedy generation with max_tokens = B produces exactly the first B tokens of the uncapped output. So accuracy
# at budget B = the saved output truncated to B tokens (re-tokenised with the model's own tokenizer), re-scored
# by the strict scorer. Budgets above a model's own generation cap are not reported (they cannot be computed).
BUDGETS = [256, 512, 1024, 2048, 3584, 8192]
BUDGET_MODELS = [m for m in ["base", "r1distill", "v1_grpo", "sft_long", "sft_v2", "grpo_ours_step60", "grpo_ours_gstd", "grpo_ours", "grpo_balanced", "grpo_ours_s2", "grpo_ours_s3",
                             "grpo_judge", "grpo_stdlen_gstd", "grpo_stdlen", "dpo_v2", "thinkless", "adaptthink"] if m in set(ALL.model)]
gold_of = BENCH.set_index("pid").gold
BUDGET_OK = {}                      # (model, bench, budget) → per-item correctness, for the paired Table 15b
rows = []
for m in BUDGET_MODELS:
    spec = MODELS[m]
    try:
        tok = tok_for(spec)
    except Exception as e:
        print(f"Table 15: skipping {m} (tokenizer unavailable: {type(e).__name__})"); continue
    cap = spec.get("max_tokens", CFG["eval_max_tokens"])
    x = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy")]
    ids = {}
    for b, xb in x.groupby("bench"):
        for B in [B for B in BUDGETS if B <= cap]:
            ok, used = [], []
            for r in xb.itertuples():
                if r.n_tokens <= B:
                    ok.append(bool(r.correct)); used.append(r.n_tokens); continue
                if r.pid not in ids:
                    ids[r.pid] = tok(r.text, add_special_tokens=False)["input_ids"]
                cut = tok.decode(ids[r.pid][:B])
                ok.append(bool(common.score_text(cut, gold_of[r.pid], b, spec["fmt"])["correct"])); used.append(B)
            BUDGET_OK[(m, b, B)] = pd.Series(ok, index=xb.pid.values, dtype=bool)
            k, n = int(sum(ok)), len(ok)
            lo, hi = common.wilson(k, n)
            rows.append(dict(model=m, bench=b, budget=B, n=n, acc=k / n, ci_low=lo, ci_high=hi,
                             mean_tokens_used=float(np.mean(used)), share_over_budget=float(np.mean(xb.n_tokens > B))))
T15 = pd.DataFrame(rows)
if len(T15):
    show(T15[T15.bench == "math500"])
    T15.to_csv(TAB / "table15_budget_curves.csv", index=False)
    piv = T15.assign(cell=lambda d: (d.acc * 100).round(1).astype(str)).pivot_table(
        index=["bench", "model"], columns="budget", values="cell", aggfunc="first")
    piv.to_latex(TAB / "table15_budget_curves.tex", label="tab:budgets",
                 caption="Strict accuracy (\\%) when generation is capped at B tokens (exact for greedy decoding: the "
                         "saved outputs truncated to B tokens and re-scored).")

# ---- Table 15b: budget crossover (paired, same items, same budget): ours vs the thinking models ----------------
OURS_BUDGET = "grpo_ours"
BENCH_NAME_T = {"gsm8k": "GSM8K", "math500": "MATH-500", "aime2026": "AIME-2026"}
rows = []
for e in [m for m in ("adaptthink", "thinkless", "r1distill") if m in BUDGET_MODELS]:
    for b in ("gsm8k", "math500"):
        for B in BUDGETS:
            a, o = BUDGET_OK.get((OURS_BUDGET, b, B)), BUDGET_OK.get((e, b, B))
            if a is None or o is None:
                continue                     # budgets above our generation cap cannot be computed for us
            ids = a.index.intersection(o.index)
            d, lo, hi, w, l, p = paired_test(a.loc[ids].values, o.loc[ids].values)
            rows.append(dict(bench=b, budget=B, comparator=LABEL[e], n=len(ids), ours=a.loc[ids].mean(),
                             theirs=o.loc[ids].mean(), delta=d, ci_low=lo, ci_high=hi, ours_only=w, theirs_only=l,
                             p_mcnemar=p))
T15B = pd.DataFrame(rows)
if len(T15B):
    T15B["p_holm"] = common.holm(T15B.p_mcnemar.tolist())
    T15B["verdict"] = np.where(T15B.p_holm >= 0.05, "n.s.", np.where(T15B.delta > 0, "ours better", "theirs better"))
    show(T15B); T15B.to_csv(TAB / "table15b_budget_crossover.csv", index=False)
    T15B.round(4).to_latex(TAB / "table15b_budget_crossover.tex", index=False, label="tab:crossover",
                           caption=f"Budget crossover: {LABEL[OURS_BUDGET]} minus each thinking model at the same token "
                                   "budget (same items; paired bootstrap 95\\% CI, exact McNemar, Holm over all rows).")
    _cap = MODELS[OURS_BUDGET].get("max_tokens", CFG["eval_max_tokens"])
    for (e, b), x in T15B.groupby(["comparator", "bench"]):
        ours_b, theirs_b = x[x.verdict == "ours better"].budget, x[x.verdict == "theirs better"].budget
        print(f"{BENCH_NAME_T.get(b, b)} vs {e}: ours significantly better at budgets {sorted(ours_b.tolist()) or 'none'}; "
              f"{e} significantly better at {sorted(theirs_b.tolist()) or 'none'} (budgets ≤ our cap {_cap})")

# ---- Table 16: does thinking pay off? forced-think vs forced-short on the same problems, by level and stage -------
FORCED_MODELS = [m for m in ("sft_long", "sft_v2", "grpo_ours_gstd", "grpo_balanced", "grpo_ours")
                 if {"forced_think", "forced_short"} <= set(ALL[ALL.model == m].condition)]
def forced_rows(m, c, b):
    return ALL[(ALL.model == m) & (ALL.condition == c) & (ALL.kind == "greedy") & (ALL.bench == b)].set_index("pid")
rows = []
for m in FORCED_MODELS:
    for b in ("gsm8k", "math500"):
        Tk, Sh = forced_rows(m, "forced_think", b), forced_rows(m, "forced_short", b)
        ids = Tk.index.intersection(Sh.index)
        Tk, Sh = Tk.loc[ids], Sh.loc[ids]
        subsets = [("all", ids)]
        if b == "math500":
            subsets += [(f"L{int(lv)}", Tk.index[Tk.level == lv]) for lv in sorted(Tk.level.dropna().unique())]
        for g, gi in subsets:
            t, sh = Tk.loc[gi].correct.astype(bool).values, Sh.loc[gi].correct.astype(bool).values
            if not len(t):
                continue
            d, lo, hi, w, l, p = paired_test(t, sh)
            rows.append(dict(model=LABEL[m], key=m, bench=b, subset=g, n=len(gi), acc_think=t.mean(), acc_short=sh.mean(),
                             think_minus_short=d, ci_low=lo, ci_high=hi, think_only=w, short_only=l, p_mcnemar=p,
                             oracle_either=(t | sh).mean(), tokens_think=Tk.loc[gi].n_tokens.mean(),
                             tokens_short=Sh.loc[gi].n_tokens.mean(),
                             truncated_think=Tk.loc[gi].truncated.astype(bool).mean()))
T16 = pd.DataFrame(rows)
if len(T16):
    T16["p_holm"] = common.holm(T16.p_mcnemar.tolist())
    show(T16.drop(columns=["key"])); T16.to_csv(TAB / "table16_think_vs_short_by_level.csv", index=False)
    T16.drop(columns=["key"]).round(4).to_latex(
        TAB / "table16_think_vs_short_by_level.tex", index=False, label="tab:think_vs_short",
        caption="Does thinking pay off? Forced-think minus forced-short accuracy on the same problems, by MATH level "
                "and training stage (paired bootstrap 95\\% CI, exact McNemar, Holm).")
    for m, x in T16[(T16.bench == "math500") & (T16.subset != "all")].groupby("model", sort=False):
        sig = x[x.p_holm < 0.05]
        print(f"{m}: think − short by MATH level = {', '.join(f'{r.subset} {100 * r.think_minus_short:+.1f}' for r in x.itertuples())}"
              f" | think significantly better on {int((sig.think_minus_short > 0).sum())}, worse on "
              f"{int((sig.think_minus_short < 0).sum())} of {len(x)} levels (Holm)")

# ---- Table 17: mechanism of the group-std CoT collapse (training logs + greedy accuracy; S8 controls) ------------
# Pre-registered (2026-10-12, before the controls ran): a run "collapses" if its mean completion length at steps
# 161–200 is below 0.6 × its length at steps 61–100. Predictions: C1 (group std, no bonus) does not collapse;
# C3 (group std, bonus ×0.1) collapses like R1-gstd; C2 (no std scaling) does not collapse.
MECH_RUNS = ["grpo_ours_gstd", "grpo_stdlen_gstd", "grpo_nobonus_gstd", "grpo_ours_gstd_b01", "grpo_ours_nostd",
             "grpo_ours", "grpo_stdlen", "grpo_ours_s2", "grpo_ours_s3", "grpo_balanced", "grpo_judge"]
def _window(h, lo, hi):
    w = h[(h.step >= lo) & (h.step <= hi)]
    def get(k, mean_col=True):
        c = next((c for c in h.columns if k in c and (c.endswith("/mean") or not mean_col)), None)
        return float(w[c].mean()) if c is not None and w[c].notna().any() else np.nan
    return dict(acc=get("strict_accuracy"), length=get("completions/mean_length", False),
                entropy=get("entropy", False), grad_norm=get("grad_norm", False),
                all_correct_groups=get("all_correct_group"), zero_std_groups=get("frac_reward_zero_std", False))
rows = []
for r in MECH_RUNS:
    h = run_log(r)
    if h.empty or "step" not in h:
        continue
    rc = RUNS / r / "resolved_grpo_config.json"
    rc = json.loads(rc.read_text()) if rc.exists() else {}
    row = dict(run=r, model=LABEL.get(r, r), scale_rewards=rc.get("args", {}).get("scale_rewards"),
               variant=rc.get("variant"), bonus_scale=rc.get("bonus_scale", 1.0), steps=int(h.step.max()))
    for tag, (lo, hi) in (("mid", (61, 100)), ("late", (161, 200))):
        row.update({f"{k}_{tag}": v for k, v in _window(h, lo, hi).items()})
    ratio = row["length_late"] / row["length_mid"] if row["length_mid"] else np.nan
    row["length_ratio_late_mid"] = ratio
    row["collapsed"] = bool(ratio < 0.6) if ratio == ratio else None
    for b in ("gsm8k", "math500"):
        x = single_correct(r, b)
        row[f"greedy_{b}"] = x.mean() if len(x) else np.nan
    rows.append(row)
T17 = pd.DataFrame(rows)
if len(T17):
    show(T17); T17.to_csv(TAB / "table17_mechanism_controls.csv", index=False)
    T17[["model", "scale_rewards", "bonus_scale", "acc_mid", "acc_late", "length_mid", "length_late", "entropy_late",
         "grad_norm_late", "all_correct_groups_mid", "length_ratio_late_mid", "collapsed", "greedy_gsm8k",
         "greedy_math500"]].round(3).to_latex(
        TAB / "table17_mechanism_controls.tex", index=False, label="tab:mechanism",
        caption="Group-std CoT collapse: rollout accuracy, completion length, entropy and gradient norm at GRPO steps "
                "61--100 (mid) and 161--200 (late), share of all-correct groups, and greedy accuracy. "
                "Collapse (pre-registered): late length below 0.6 times mid length.")
    pairs = [("grpo_ours", "grpo_ours_gstd"), ("grpo_stdlen", "grpo_stdlen_gstd")]
    pairs += [(c, ref) for c in ("grpo_nobonus_gstd", "grpo_ours_nostd", "grpo_ours_gstd_b01")
              for ref in ("grpo_ours_gstd", "grpo_ours")]
    rows = []
    for a_, b_ in pairs:
        for b in ("gsm8k", "math500"):
            x, y = single_correct(a_, b), single_correct(b_, b)
            if x.empty or y.empty:
                continue
            ids = x.index.intersection(y.index)
            d, lo, hi, w, l, p = paired_test(x.loc[ids].values, y.loc[ids].values)
            rows.append(dict(comparison=f"{LABEL[a_]} − {LABEL[b_]}", bench=b, n=len(ids), delta=d, ci_low=lo,
                             ci_high=hi, a_only=w, b_only=l, p_mcnemar=p))
    T17B = pd.DataFrame(rows)
    if len(T17B):
        T17B["p_holm"] = common.holm(T17B.p_mcnemar.tolist())
        show(T17B); T17B.to_csv(TAB / "table17b_mechanism_paired.csv", index=False)
    for r in T17.itertuples():
        if r.collapsed is not None:
            print(f"{r.model}: length {r.length_mid:.0f} → {r.length_late:.0f} tokens (×{r.length_ratio_late_mid:.2f}), "
                  f"rollout accuracy {r.acc_mid:.2f} → {r.acc_late:.2f} → {'COLLAPSED' if r.collapsed else 'stable'}")

# ---- Table 18a/18b: multiple independent attempts --------------------------------------
# Pool = independent samples per problem: the full-benchmark votes of 12e (AIME: the first k of 13's samples) if they
# exist, else the MPV attempts (100/100/30 subset, perspective prompts). A vote over the first k samples gets
# fractional credit on a tied top (as Table 1b). Credits are not 0/1, so p-values come from a paired sign-flip
# permutation test, with a paired bootstrap CI; each table is its own Holm family.
def _akey(a):
    r = common.rational(a)
    return str(r) if r is not None else str(a).strip()

def vote_credit(answers, corrects):
    votes = Counter("<correct>" if c else _akey(a) for a, c in zip(answers, corrects)
                    if c or (isinstance(a, str) and a.strip()))
    if not votes:
        return 0.0
    top = max(votes.values())
    tied = [a for a, v in votes.items() if v == top]
    return ("<correct>" in tied) / len(tied)

def vote_pool(key):
    K = CFG["vote_k"]
    f = EVAL / "preds" / f"vote{K}__{key}.jsonl"
    cols = ["pid", "bench", "sample", "correct", "answer", "n_tokens"]
    if f.exists():
        v = pd.DataFrame(read_jsonl(f))
        a = ALL[(ALL.model == key) & (ALL.bench == "aime2026") & (ALL.kind == "sampled") & (ALL["sample"] < K)]
        v = pd.concat([v, a], ignore_index=True).rename(columns={"final_answer": "answer"})
        return v[cols], f"full benchmarks, {K} samples"
    m = MPV[MPV.model == key] if len(MPV) and "attempt_tokens_list" in MPV else pd.DataFrame()
    if len(m):
        v = pd.DataFrame([dict(pid=r.pid, bench=r.bench, sample=k, correct=bool(c), answer=a, n_tokens=t)
                          for r in m.itertuples() for k, (a, c, t) in
                          enumerate(zip(r.attempt_answers_all, r.attempt_correct_all, r.attempt_tokens_list))])
        return v[cols], f"MPV subset, {int(v.groupby('pid').size().min())} perspective attempts"
    return None, None

def vote_items(pool):
    """Per problem: answers, correctness and tokens of its samples, in sample order (all problems get the same k)."""
    g = {pid: x.sort_values("sample") for pid, x in pool.groupby("pid")}
    K = min(len(x) for x in g.values())
    return {pid: (list(x.answer)[:K], list(x.correct.astype(bool))[:K], list(x.n_tokens)[:K]) for pid, x in g.items()}, K

def perm_p(d, draws=4000):
    d = np.asarray(d, float)
    if not np.any(d):
        return 1.0
    rng, obs, hits = np.random.default_rng(CFG["seed"]), abs(d.mean()), 0
    for _ in range(draws // 1000):
        s = rng.choice([-1.0, 1.0], size=(1000, len(d)))
        hits += int((np.abs((s * d).mean(axis=1)) >= obs - 1e-12).sum())
    return (hits + 1) / (draws + 1)

def greedy_rows(key, cond, bench):
    return ALL[(ALL.model == key) & (ALL.condition == cond) & (ALL.kind == "greedy") & (ALL.bench == bench)].set_index("pid")

VOTE_MODELS_ALL = [m for m in dict.fromkeys(list(CFG["vote_models"]) + list(CFG["mpv_models"])) if m in MODELS]
VOTE_CURVES, rows_a, rows_b = {}, [], []
DEPTH_THINKERS = [m for m in ("adaptthink", "thinkless", "r1distill") if m in set(ALL.model)]
for key in VOTE_MODELS_ALL:
    pool, source = vote_pool(key)
    if pool is None:
        continue
    for b in ("gsm8k", "math500", "aime2026"):
        if not (pool.bench == b).any():
            continue
        items, K = vote_items(pool[pool.bench == b])
        ids = pd.Index(sorted(items))
        credit = {k: np.array([vote_credit(items[i][0][:k], items[i][1][:k]) for i in ids]) for k in range(1, K + 1)}
        tokens = {k: np.array([sum(items[i][2][:k]) for i in ids], float) for k in range(1, K + 1)}
        VOTE_CURVES[(key, b)] = dict(source=source, n=len(ids), k=list(range(1, K + 1)),
                                     acc=[credit[k].mean() for k in credit], tokens=[tokens[k].mean() for k in tokens])
        # 18a: one attempt vs several (claims 1-3 of the paper plan, C4c)
        single = greedy_rows(key, "single", b).reindex(ids)
        k4 = min(4, K)
        es = [(float(items[i][1][0]), items[i][2][0] + items[i][2][1]) if common.answers_equivalent(items[i][0][0], items[i][0][1])
              else (vote_credit(items[i][0][:k4], items[i][1][:k4]), sum(items[i][2][:k4])) for i in ids]
        es_c, es_t = np.array([c for c, _ in es]), np.array([t for _, t in es], float)
        ok = single.correct.notna().values
        s_c = single.correct.astype(float).values
        row = dict(model=LABEL[key], vote_source=source, bench=b, n=len(ids), single_greedy=np.nanmean(s_c),
                   single_tokens=single.n_tokens.mean(), first_sample=credit[1].mean(),
                   any_correct_at_4=np.mean([any(items[i][1][:k4]) for i in ids]),
                   any_correct_at_K=np.mean([any(items[i][1]) for i in ids]),
                   majority_at_4=credit[k4].mean(), majority_at_K=credit[K].mean(), K=K,
                   tokens_majority_at_4=tokens[k4].mean(), tokens_majority_at_K=tokens[K].mean(),
                   early_stop_vote=es_c.mean(), tokens_early_stop=es_t.mean(),
                   early_stop_token_saving_vs_maj4=1 - es_t.mean() / tokens[k4].mean())
        for name, c in ((f"maj{K}_minus_single", credit[K]), ("maj4_minus_single", credit[k4])):
            if ok.any():
                d, lo, hi = common.paired_bootstrap(c[ok], s_c[ok], seed=CFG["seed"])
                row.update({name: d, f"{name}_ci_low": lo, f"{name}_ci_high": hi, f"{name}_p": perm_p(c[ok] - s_c[ok])})
        d, lo, hi = common.paired_bootstrap(es_c, credit[k4], seed=CFG["seed"])
        row.update(early_stop_minus_maj4=d, early_stop_minus_maj4_ci_low=lo, early_stop_minus_maj4_ci_high=hi)
        rows_a.append(row)
        # 18b: breadth (vote over k short samples) vs depth (ONE think answer) at matched tokens, same problems
        for dkey, dcond in [(key, "forced_think")] + [(m, "single") for m in DEPTH_THINKERS]:
            dep = greedy_rows(dkey, dcond, b)
            cid = ids.intersection(dep.index)
            if len(cid) < 10:
                continue
            pos = ids.get_indexer(cid)
            t_depth = dep.loc[cid].n_tokens.mean()
            fit = [k for k in range(1, K + 1) if tokens[k][pos].mean() <= t_depth]
            k_star = max(fit) if fit else 1          # breadth never gets more tokens than depth (unless even k=1 does)
            v, dok = credit[k_star][pos], dep.loc[cid].correct.astype(float).values
            d, lo, hi = common.paired_bootstrap(v, dok, seed=CFG["seed"])
            rows_b.append(dict(model=LABEL[key], vote_source=source, bench=b, depth=f"{LABEL[dkey]} [{dcond}]",
                               vote_key=key, depth_key=dkey, depth_cond=dcond,
                               n=len(cid), depth_acc=dok.mean(), depth_tokens=t_depth, k_matched=k_star,
                               vote_acc=v.mean(), vote_tokens=tokens[k_star][pos].mean(), vote_minus_depth=d,
                               ci_low=lo, ci_high=hi, p_perm=perm_p(v - dok), vote_uses_more_tokens=not fit,
                               vote_acc_at_K=credit[K][pos].mean(), vote_tokens_at_K=tokens[K][pos].mean()))
T18A, T18B = pd.DataFrame(rows_a), pd.DataFrame(rows_b)
if len(T18A):
    pcols = [c for c in T18A.columns if c.endswith("_minus_single_p")]
    flat = [(i, c) for c in pcols for i in T18A.index if pd.notna(T18A.at[i, c])]
    for (i, c), ph in zip(flat, common.holm([T18A.at[i, c] for i, c in flat]) if flat else []):
        T18A.at[i, c + "_holm"] = ph
    show(T18A); T18A.to_csv(TAB / "table18a_multiple_attempts.csv", index=False)
    T18A.round(4).to_latex(TAB / "table18a_multiple_attempts.tex", index=False, label="tab:attempts",
                           caption="One attempt vs several independent attempts on the same problems: greedy single "
                                   "pass, one sample, any-correct@4/@k, majority@4/@k (fractional credit on ties) and the "
                                   "early-stop vote (stop when the first two attempts agree, else majority@4), with mean "
                                   "tokens per problem. Paired bootstrap 95\\% CI, sign-flip permutation p, Holm.")
    for r in T18A.itertuples():
        print(f"{r.model:32} {r.bench:8} n={r.n:4} single {r.single_greedy:.3f} | any-correct@4 {r.any_correct_at_4:.3f} | "
              f"maj@4 {r.majority_at_4:.3f} | maj@{r.K} {r.majority_at_K:.3f} | early-stop {r.early_stop_vote:.3f} "
              f"(−{100 * r.early_stop_token_saving_vs_maj4:.0f}% tokens vs maj@4) [{r.vote_source}]")
if len(T18B):
    T18B["p_holm"] = common.holm(T18B.p_perm.tolist())
    show(T18B); T18B.to_csv(TAB / "table18b_breadth_vs_depth.csv", index=False)
    T18B.round(4).to_latex(TAB / "table18b_breadth_vs_depth.tex", index=False, label="tab:breadth_depth",
                           caption="Breadth vs depth at matched tokens: a majority vote over k independent short answers "
                                   "(k = the largest number whose mean tokens do not exceed the think answer's) against "
                                   "one think answer (the same model's forced think, or a thinking model), on the same "
                                   "problems. Paired bootstrap 95\\% CI, sign-flip permutation p, Holm.")
    for r in T18B.itertuples():
        print(f"{r.model:32} {r.bench:8} vote@{r.k_matched} {r.vote_acc:.3f} ({r.vote_tokens:.0f} tok) vs "
              f"{r.depth:45} {r.depth_acc:.3f} ({r.depth_tokens:.0f} tok): Δ {100 * r.vote_minus_depth:+.1f} "
              f"[{100 * r.ci_low:+.1f}, {100 * r.ci_high:+.1f}], Holm p={r.p_holm:.3g}")

# ---- Table 19: the second base model (S10–S11) --------------------------------------------
# Does the recipe's behaviour replicate on Qwen2.5-1.5B-Instruct (general, not math-specialised; same size, vocabulary
# and chat template; same SFT data, RL prompts, seed, rewards and steps)? 19a: accuracy per stage for both base models.
# 19b: paired tests within the second model. 19c: think − short by MATH level (Table 16's method). 19d: training
# dynamics and the verdicts of the two predictions pre-registered in 11g. Each table is its own Holm family, so
# Tables 2, 12, 16 and 17 are unchanged.
M2_STAGES = [("base", "base", "m2_base"), ("SFT", "sft_v2", "m2_sft"), ("GRPO R1 (batch std)", "grpo_ours", "m2_grpo"),
             ("GRPO R1 (group std)", "grpo_ours_gstd", "m2_grpo_gstd")]
BASE_NAME = {"main": "Qwen2.5-Math-1.5B-Instruct", "m2": "Qwen2.5-1.5B-Instruct"}
if any(k in set(ALL.model) for _, _, k in M2_STAGES):
    def cond_correct(model, cond, bench):
        return ALL[(ALL.model == model) & (ALL.condition == cond) & (ALL.kind == "greedy") &
                   (ALL.bench == bench)].set_index("pid").correct.astype(bool)
    rows = []
    for stage, k1, k2 in M2_STAGES:
        for fam, k in (("main", k1), ("m2", k2)):
            for r in T1[T1.model == k].itertuples():
                rows.append(dict(stage=stage, base_model=BASE_NAME[fam], key=k, condition=r.condition, bench=r.bench,
                                 n=r.n, acc=r.acc, ci_low=r.ci_low, ci_high=r.ci_high, mean_tokens=r.mean_tokens,
                                 think_rate=r.think_rate, format_ok=r.format_ok, repetition=r.repetition,
                                 truncated=r.truncated))
    T19A = pd.DataFrame(rows)
    show(T19A); T19A.to_csv(TAB / "table19a_second_model.csv", index=False)
    _s = T19A[T19A.condition == "single"].copy()
    _s["cell"] = [f"{100 * a:.1f} [{100 * lo:.1f}, {100 * hi:.1f}]" for a, lo, hi in zip(_s.acc, _s.ci_low, _s.ci_high)]
    _s.pivot_table(index=["stage", "base_model"], columns="bench", values="cell", aggfunc="first").to_latex(
        TAB / "table19a_second_model.tex", label="tab:second_model",
        caption="Second base model: strict accuracy (\\%, Wilson 95\\% CI) after each stage of the same recipe, for the "
                "math-specialised Qwen2.5-Math-1.5B-Instruct and the general Qwen2.5-1.5B-Instruct.")
    tests = [("m2_sft", "single", "m2_base", "single", "SFT − base"),
             ("m2_grpo", "single", "m2_sft", "single", "GRPO R1 − SFT"),
             ("m2_grpo", "single", "m2_base", "single", "GRPO R1 − base"),
             ("m2_grpo", "single", "m2_grpo_gstd", "single", "batch std − group std (R1's recipe)"),
             ("m2_sft", "forced_think", "m2_sft", "forced_short", "SFT: forced think − forced short"),
             ("m2_grpo", "forced_think", "m2_grpo", "forced_short", "GRPO R1: forced think − forced short")]
    rows = []
    for a_, ac, b_, bc, name in tests:
        for b in ("gsm8k", "math500"):
            x, y = cond_correct(a_, ac, b), cond_correct(b_, bc, b)
            ids = x.index.intersection(y.index)
            if not len(ids):
                continue
            d, lo, hi, w, l, pv = paired_test(x.loc[ids].values, y.loc[ids].values)
            rows.append(dict(comparison=name, bench=b, n=len(ids), a_acc=x.loc[ids].mean(), b_acc=y.loc[ids].mean(),
                             delta=d, ci_low=lo, ci_high=hi, a_only=w, b_only=l, p_mcnemar=pv))
    T19B = pd.DataFrame(rows)
    if len(T19B):
        T19B["p_holm"] = common.holm(T19B.p_mcnemar.tolist())
        show(T19B); T19B.to_csv(TAB / "table19b_second_model_paired.csv", index=False)
        T19B.round(4).to_latex(TAB / "table19b_second_model_paired.tex", index=False, label="tab:second_model_paired",
                               caption="Second base model: paired differences on the same problems (bootstrap 95\\% "
                                       "CI, exact McNemar, Holm over this table).")
    rows = []
    for m in ("m2_sft", "m2_grpo"):
        Tk, Sh = forced_rows(m, "forced_think", "math500"), forced_rows(m, "forced_short", "math500")
        ids = Tk.index.intersection(Sh.index)
        if not len(ids):
            continue
        Tk, Sh = Tk.loc[ids], Sh.loc[ids]
        for g, gi in [("all", ids)] + [(f"L{int(lv)}", Tk.index[Tk.level == lv]) for lv in sorted(Tk.level.dropna().unique())]:
            t, sh = Tk.loc[gi].correct.astype(bool).values, Sh.loc[gi].correct.astype(bool).values
            d, lo, hi, w, l, pv = paired_test(t, sh)
            rows.append(dict(model=LABEL[m], key=m, bench="math500", subset=g, n=len(gi), acc_think=t.mean(),
                             acc_short=sh.mean(), think_minus_short=d, ci_low=lo, ci_high=hi, think_only=w, short_only=l,
                             p_mcnemar=pv, tokens_think=Tk.loc[gi].n_tokens.mean(), tokens_short=Sh.loc[gi].n_tokens.mean(),
                             truncated_think=Tk.loc[gi].truncated.astype(bool).mean()))
    T19C = pd.DataFrame(rows)
    if len(T19C):
        T19C["p_holm"] = common.holm(T19C.p_mcnemar.tolist())
        show(T19C.drop(columns=["key"])); T19C.to_csv(TAB / "table19c_second_model_think_vs_short.csv", index=False)
    def early_mode_acc(h, lo=1, hi=20):
        """Rollout accuracy of think and of short answers over steps lo–hi (ratio of sums: 64 rollouts per step)."""
        w = h[(h.step >= lo) & (h.step <= hi)]
        col = lambda k: next((c for c in h.columns if k in c and c.endswith("/mean")), None)
        tc, tr, sa = col("think_correct"), col("think_rate"), col("strict_accuracy")
        if w.empty or not (tc and tr and sa):
            return np.nan, np.nan
        think, n = w[tr].sum(), len(w)
        return (w[tc].sum() / think if think > 0 else np.nan,
                (w[sa].sum() - w[tc].sum()) / (n - think) if n - think > 0 else np.nan)
    rows = []
    for stage, k1, k2 in M2_STAGES[2:]:
        for fam, run in (("main", k1), ("m2", k2)):
            h = run_log(run)
            if h.empty or "step" not in h:
                continue
            trc = next((c for c in h.columns if "think_rate" in c and c.endswith("/mean")), None)
            row = dict(stage=stage, base_model=BASE_NAME[fam], run=run, steps=int(h.step.max()))
            for lo, hi in ((1, 20), (21, 60), (61, 100), (161, 200)):
                w = h[(h.step >= lo) & (h.step <= hi)]
                row[f"think_rate_{lo}_{hi}"] = float(w[trc].mean()) if trc and len(w) else np.nan
            row["think_acc_1_20"], row["short_acc_1_20"] = early_mode_acc(h)
            roll = h.set_index("step")[trc].rolling(10, min_periods=10).mean() if trc else pd.Series(dtype=float)
            below = roll[roll < 0.05]
            row["first_step_think_below_5pct"] = int(below.index[0]) if len(below) else np.nan   # end of a 10-step window
            for tag, (lo, hi) in (("mid", (61, 100)), ("late", (161, 200))):
                row.update({f"{k}_{tag}": v for k, v in _window(h, lo, hi).items()})
            ratio = row["length_late"] / row["length_mid"] if row["length_mid"] else np.nan
            row["length_ratio_late_mid"] = ratio
            row["collapsed"] = bool(ratio < 0.6) if ratio == ratio else None
            rows.append(row)
    T19D = pd.DataFrame(rows)
    VERDICTS_19 = {}
    if len(T19D):
        show(T19D); T19D.to_csv(TAB / "table19d_second_model_dynamics.csv", index=False)
        d2 = T19D.set_index("run")
        if "m2_grpo" in d2.index:
            r = d2.loc["m2_grpo"]
            if pd.notna(r.think_acc_1_20) and pd.notna(r.short_acc_1_20) and pd.notna(r.think_rate_61_100):
                short_wins = bool(r.short_acc_1_20 > r.think_acc_1_20)
                VERDICTS_19["a_more_accurate_mode_wins"] = dict(
                    think_acc_steps_1_20=r.think_acc_1_20, short_acc_steps_1_20=r.short_acc_1_20,
                    think_rate_steps_61_100=r.think_rate_61_100,
                    prediction="think-rate < 5% over steps 61–100" if short_wins else "think-rate ≥ 5% over steps 61–100",
                    held=bool((r.think_rate_61_100 < 0.05) == short_wins))
            else:
                VERDICTS_19["a_more_accurate_mode_wins"] = dict(held=None, note="not testable: no think (or no short) "
                                                                "rollouts over steps 1–20, or fewer than 100 steps")
        if {"m2_grpo", "m2_grpo_gstd"} <= set(d2.index):
            cg, cb = d2.loc["m2_grpo_gstd", "collapsed"], d2.loc["m2_grpo", "collapsed"]
            VERDICTS_19["b_group_std_collapses_batch_std_does_not"] = dict(
                ratio_group_std=d2.loc["m2_grpo_gstd", "length_ratio_late_mid"],
                ratio_batch_std=d2.loc["m2_grpo", "length_ratio_late_mid"],
                all_correct_groups_mid_group_std=d2.loc["m2_grpo_gstd", "all_correct_groups_mid"],
                held=None if cg is None or cb is None else bool(cg and not cb))
        (TAB / "table19d_preregistered_verdicts.json").write_text(json.dumps(VERDICTS_19, indent=1, default=float))
        for r in T19D.itertuples():
            verdict = "COLLAPSED" if r.collapsed else ("stable" if r.collapsed is not None else "n/a")
            print(f"{r.base_model:28} {r.stage:22} think-rate {r.think_rate_1_20:.2f} (steps 1–20) → "
                  f"{r.think_rate_61_100:.2f} (61–100) | early rollout accuracy think {r.think_acc_1_20:.2f} vs short "
                  f"{r.short_acc_1_20:.2f} | length ×{r.length_ratio_late_mid:.2f} → {verdict}")
        for k, v in VERDICTS_19.items():
            state = "HELD" if v.get("held") else ("DID NOT HOLD" if v.get("held") is False else "not decided yet")
            shown = {kk: round(float(vv), 3) if isinstance(vv, (float, np.floating)) else vv for kk, vv in v.items()}
            print(f"pre-registered {k}: {state} | {json.dumps(shown, ensure_ascii=False, default=str)}")
else:
    print("Table 19: no second-model predictions yet (S10–S11)")

## 39.0) Figures (300-DPI PNG + vector PDF; one fixed color per model across all figures)

In [ ]:
# 39.0) Figure style
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948",
           "#7a7a7a", "#8c564b"]
MODEL_ORDER = ["base", "r1distill", "v1_grpo", "sft_v2", "grpo_ours", "grpo_judge", "grpo_stdlen", "dpo_v2",
               "thinkless", "adaptthink"]
COLOR = dict(zip(MODEL_ORDER, PALETTE))
COLOR["grpo_ours_s2"] = COLOR["grpo_ours"]          # same method, second seed → same colour (8-colour palette)
COLOR["grpo_ours_s3"] = COLOR["grpo_ours"]          # and the third seed (S9)
COLOR["sft_long"] = "#f5cf73"                        # S2 long-trace SFT ablation: a light tint of SFT's amber
COLOR["grpo_balanced"] = "#0f9fb3"                   # R1b (balanced exploration): teal
COLOR["grpo_ours_step60"] = "#f3b8d0"                # R1 at step 60: a light tint of R1's pink
COLOR["grpo_ours_gstd"] = "#a8466f"                  # R1, group-std (collapsed): a dark shade of R1's pink
COLOR["grpo_stdlen_gstd"] = "#958ad3"                # R3, group-std (collapsed): a light tint of R3's violet
COLOR["grpo_nobonus_gstd"] = "#5f5e5a"               # control C1 (group std, no bonus): dark grey
COLOR["grpo_ours_nostd"] = "#8c564b"                 # control C2 (no std scaling): brown
COLOR["grpo_ours_gstd_b01"] = "#6b1f3f"              # control C3 (group std, bonus ×0.1): the darkest shade of R1's pink
COLOR["m2_base"] = "#5d6d7e"                         # second base model (S10–S11), its own hue family: slate,
COLOR["m2_sft"] = "#9a8c00"                          # olive,
COLOR["m2_grpo"] = "#00a6a6"                         # teal (R1's recipe)
COLOR["m2_grpo_gstd"] = "#005f5f"                    # and a dark shade of it for the group-std run
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "legend.frameon": False})
def save(fig, name):
    if not any(ax.has_data() for ax in fig.axes):        # e.g. GRPO curves before any GRPO run
        plt.close(fig); print(f"{name}: no data yet (it fills in once the models it shows are trained)"); return
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight"); fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show()
BENCH_NAME = {"gsm8k": "GSM8K", "math500": "MATH-500", "aime2026": "AIME-2026"}
def bench_title(b):   # n comes from the frozen benchmark actually evaluated (pilot subsets included)
    return f"{BENCH_NAME[b]} (n={int((BENCH.bench == b).sum())})"

In [ ]:
# Fig 1: accuracy with 95% CI per benchmark (single-pass models)
t = T1[T1.condition == "single"]
models = [m for m in MODEL_ORDER if m in set(t.model)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=False)
for ax, b in zip(axes, BENCH_NAME):
    x = t[t.bench == b].set_index("model").reindex(models).dropna(subset=["acc"])
    ax.bar(range(len(x)), x.acc * 100, color=[COLOR[m] for m in x.index], width=0.7, edgecolor="white", linewidth=2)
    ax.errorbar(range(len(x)), x.acc * 100, yerr=[(x.acc - x.ci_low) * 100, (x.ci_high - x.acc) * 100],
                fmt="none", ecolor="#52514e", capsize=3, lw=1)
    ax.set_xticks(range(len(x)), [LABEL[m] for m in x.index], rotation=55, ha="right", fontsize=8)
    ax.set_title(bench_title(b)); ax.set_ylabel("strict accuracy (%)"); ax.grid(axis="x", visible=False)
save(fig, "fig1_accuracy_ci")

In [ ]:
# Fig 2: accuracy vs mean generated tokens (Pareto view), GSM8K and MATH-500
PARETO_MODELS = ["base", "r1distill", "sft_v2", "grpo_ours"]          # <=4 hues keeps a scatter readable
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, b in zip(axes, ("gsm8k", "math500")):
    x = T1[(T1.bench == b) & T1.model.isin(PARETO_MODELS)]
    for r in x.itertuples():
        marker = {"single": "o", "forced_think": "^", "forced_short": "v"}[r.condition]
        ax.errorbar(r.mean_tokens, r.acc * 100, yerr=[[max(0.0, (r.acc - r.ci_low) * 100)], [max(0.0, (r.ci_high - r.acc) * 100)]],
                    fmt=marker, color=COLOR[r.model], ms=8, mec="white", mew=1.5, elinewidth=1, capsize=2)
    ax.set_xscale("log"); ax.set_xlabel("mean generated tokens (log)"); ax.set_ylabel("strict accuracy (%)")
    ax.set_title(bench_title(b))
handles = [plt.Line2D([], [], marker="o", ls="", color=COLOR[m], label=LABEL[m]) for m in PARETO_MODELS if m in models] + \
          [plt.Line2D([], [], marker=k, ls="", color="#52514e", label=v) for k, v in
           (("o", "self-selected mode"), ("^", "forced think"), ("v", "forced short"))]
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
save(fig, "fig2_accuracy_vs_tokens")

In [ ]:
# Fig 3 (core adaptivity figure): think-rate by MATH level, and by empirical difficulty
# think-rate = the model CHOSE to think (incl. thinking cut off by the token limit); prior work dashed for comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for m in ("sft_v2", "grpo_ours", "grpo_balanced", "grpo_stdlen", "grpo_judge", "dpo_v2", "thinkless", "adaptthink"):
    s = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy") & (ALL.bench == "math500")]
    if s.empty: continue
    g = s.groupby("level")["chose_think"].mean()
    n = s.groupby("level").size()
    ci = [common.wilson(int(round(p * k)), k) for p, k in zip(g, n)]
    main = m in ("sft_v2", "grpo_ours")
    prior = m in EXTERNAL
    yerr = [[100 * (p - c[0]) for p, c in zip(g, ci)], [100 * (c[1] - p) for p, c in zip(g, ci)]] if main else None
    axes[0].errorbar(g.index, g * 100, yerr=yerr, color=COLOR[m], marker="s" if prior else "o", ms=6 if main else 4,
                     lw=2 if main else 1.2, ls="--" if prior else "-", alpha=1 if main else 0.8, capsize=2, label=LABEL[m])
    diff = ALL[(ALL.model == "base") & (ALL.kind == "difficulty")].groupby("pid").correct.mean()
    sd = s.set_index("pid").join(diff.rename("base_pass"), how="inner")
    if len(sd):
        h = sd.groupby("base_pass")["chose_think"].mean()
        axes[1].plot(1 - h.index, h * 100, color=COLOR[m], marker="s" if prior else "o", ms=6 if main else 4,
                     lw=2 if main else 1.2, ls="--" if prior else "-", label=LABEL[m])
axes[0].set_ylim(-3, 103); axes[1].set_ylim(-3, 103)
axes[0].set_xlabel("MATH level (1 = easiest)"); axes[0].set_ylabel("think-mode rate (%)")
axes[0].set_title("Think-rate by annotated difficulty"); axes[0].set_xticks(range(1, 6))
axes[1].set_xlabel("empirical difficulty = 1 − base pass rate (4 samples)"); axes[1].set_ylabel("think-mode rate (%)")
axes[1].set_title("Think-rate by empirical difficulty")
if axes[0].has_data():
    axes[0].legend(fontsize=8)
save(fig, "fig3_think_rate_by_difficulty")

In [ ]:
# Fig 4: token-length distributions by chosen mode (GRPO-ours, MATH-500) and by level
s = ALL[(ALL.model == "grpo_ours") & (ALL.condition == "single") & (ALL.kind == "greedy") & (ALL.bench == "math500")]
if len(s):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    data = [s[s["mode"] == md].n_tokens.values for md in ("short", "think")]
    present = [(lab, d) for lab, d in zip(["<short>", "<think>"], data) if len(d)]
    if present:   # robust when a model never uses one of the modes (seen in the pilot)
        parts = axes[0].violinplot([d for _, d in present], showmedians=True)
        for pc, c in zip(parts["bodies"], (PALETTE[0], PALETTE[1])):
            pc.set_facecolor(c); pc.set_alpha(0.6)
        axes[0].set_xticks(range(1, len(present) + 1), [lab for lab, _ in present])
    axes[0].set_ylabel("generated tokens")
    axes[0].set_title("Length by chosen mode (GRPO-ours, MATH-500)")
    s.assign(level=s.level.astype(int)).boxplot(column="n_tokens", by="level", ax=axes[1], grid=False)
    axes[1].set_title("Length by MATH level"); axes[1].set_xlabel("level"); axes[1].set_ylabel("generated tokens")
    fig.suptitle("")
    save(fig, "fig4_length_by_mode_and_level")

In [ ]:
# Fig 5: GRPO training curves (R1 vs R2 vs R3): strict accuracy, think-rate, completion length, accuracy by mode
fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
for run, mkey in (("grpo_ours", "grpo_ours"), ("grpo_balanced", "grpo_balanced"), ("grpo_judge", "grpo_judge"),
                  ("grpo_stdlen", "grpo_stdlen"), ("grpo_ours_gstd", "grpo_ours_gstd"),
                  ("grpo_stdlen_gstd", "grpo_stdlen_gstd")):
    h = run_log(run)
    if h.empty: continue
    sm = lambda c: h[c].rolling(10, min_periods=1).mean()
    acc_c = next((c for c in h.columns if "strict_accuracy" in c and c.endswith("/mean")), None)
    thk_c = next((c for c in h.columns if "think_rate" in c and c.endswith("/mean")), None)
    thk_c = next((c for c in h.columns if "policy_think_prob" in c and c.endswith("/mean")), thk_c)  # R1b: own choice
    len_c = next((c for c in h.columns if "mean_length" in c and "completions" in c), None)
    for ax, c in zip(axes, (acc_c, thk_c, len_c)):
        if c: ax.plot(h.step, sm(c), color=COLOR[mkey], lw=2, ls="--" if mkey.endswith("_gstd") else "-",
                      label=LABEL[mkey])
    for (lab, acc), ls in zip(mode_accuracy(h).items(), ("-", ":")):     # does RL make thinking pay off?
        axes[3].plot(h.step, acc, color=COLOR[mkey], lw=2, ls=ls, label=f"{LABEL[mkey]}: {lab.split()[0]}")
for ax, t in zip(axes, ("strict accuracy (rollouts)", "think-mode rate (R1b: policy's own P(think))", "mean completion length",
                        "accuracy by chosen mode (— think, ··· short)")):
    ax.set_title(t); ax.set_xlabel("GRPO step")
if axes[3].has_data():
    axes[3].legend(fontsize=7)
if axes[0].has_data():
    axes[0].legend(fontsize=8)
save(fig, "fig5_grpo_training_curves")

In [ ]:
# Fig 6 (C3, online): R2 judge reward vs strict accuracy during training (reward hacking if they diverge)
h = run_log("grpo_judge")
if not h.empty:
    jc = next(c for c in h.columns if "judge_correctness" in c and c.endswith("/mean"))
    ac = next(c for c in h.columns if "strict_accuracy" in c and c.endswith("/mean"))
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.plot(h.step, h[jc].rolling(10, min_periods=1).mean(), color=PALETTE[1], lw=2, label="judge reward (optimised)")
    ax.plot(h.step, h[ac].rolling(10, min_periods=1).mean(), color=PALETTE[0], lw=2, label="strict accuracy (not optimised)")
    ax.set_xlabel("GRPO step"); ax.set_ylabel("rollout mean (same 0–1 scale)"); ax.legend()
    ax.set_title("R2: judge-rewarded GRPO, judge score vs verified correctness")
    save(fig, "fig6_judge_reward_hacking")

In [ ]:
# Fig 7 (C3, offline): judge confusion matrix + FPR by candidate length / no-answer subset
if len(J):
    met = judge_metrics(J)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    cm = np.array([[met["TN"], met["FP"]], [met["FN"], met["TP"]]])
    axes[0].imshow(cm, cmap="Blues")
    for (i, j_), v in np.ndenumerate(cm):
        axes[0].text(j_, i, str(v), ha="center", va="center", color="#0b0b0b" if v < cm.max() * 0.6 else "white")
    axes[0].set_xticks([0, 1], ["judge: wrong", "judge: correct"]); axes[0].set_yticks([0, 1], ["gold: wrong", "gold: correct"])
    axes[0].grid(False); axes[0].set_title(f"1.5B judge, v1 parser, vs strict gold (κ={met['kappa']:.2f})")
    sub = JUDGE_TABLE[JUDGE_TABLE.subset.str.startswith(("candidate tokens", "no final"))]
    axes[1].barh(range(len(sub)), sub.FPR * 100, color=PALETTE[7], height=0.6)
    axes[1].errorbar(sub.FPR * 100, range(len(sub)), xerr=[(sub.FPR - sub.FPR_ci_low) * 100, (sub.FPR_ci_high - sub.FPR) * 100],
                     fmt="none", ecolor="#52514e", capsize=3)
    axes[1].set_yticks(range(len(sub)), sub.subset, fontsize=8); axes[1].set_xlabel("false-positive rate (%)")
    axes[1].set_title("Judge accepts wrong answers")
    save(fig, "fig7_judge_reliability")

In [ ]:
# Fig 8 (C4): multi-reasoning, accuracy and tokens vs single pass, with rescues/damages
if len(MULTI) and "T7" in globals():
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    lab = T7.model.str.split(" (", regex=False).str[0] + " · " + T7.bench
    xx = np.arange(len(T7))
    axes[0].bar(xx - 0.2, T7.single_pass_acc * 100, 0.38, color=PALETTE[0], label="single pass")
    axes[0].bar(xx + 0.2, T7.multi_acc * 100, 0.38, color=PALETTE[1], label="5-call multi-reasoning")
    axes[0].set_xticks(xx, lab, rotation=20, ha="right", fontsize=8); axes[0].set_ylabel("strict accuracy (%)")
    axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8); axes[0].set_title("Accuracy", pad=26)
    axes[1].bar(xx - 0.2, T7.rescued, 0.38, color=PALETTE[2], label="rescued (initial wrong → final right)")
    axes[1].bar(xx + 0.2, T7.damaged, 0.38, color=PALETTE[7], label="damaged (initial right → final wrong)")
    axes[1].set_xticks(xx, lab, rotation=20, ha="right", fontsize=8); axes[1].set_ylabel("problems")
    axes[1].legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8); axes[1].set_title("Paired rescues vs damages", pad=26)
    save(fig, "fig8_multi_reasoning")

In [ ]:
# Fig 9: stage comparison (base → SFT → GRPO/DPO variants) on MATH-500 by level
fig, ax = plt.subplots(figsize=(8, 3.8))
for m in ("base", "sft_v2", "grpo_ours_step60", "grpo_ours_gstd", "grpo_ours", "grpo_balanced", "grpo_stdlen_gstd",
          "grpo_stdlen", "grpo_judge", "dpo_v2"):
    s = ALL[(ALL.model == m) & (ALL.condition == "single") & (ALL.kind == "greedy") & (ALL.bench == "math500")]
    if s.empty: continue
    g = s.groupby("level").correct.mean()
    ax.plot(g.index, g * 100, marker="o", ms=6, lw=2, color=COLOR[m], label=LABEL[m])
ax.set_xlabel("MATH level"); ax.set_ylabel("strict accuracy (%)"); ax.set_xticks(range(1, 6))
ax.set_title("MATH-500 accuracy by level across training stages"); ax.legend(fontsize=8)
save(fig, "fig9_stage_comparison_by_level")

In [ ]:
# Fig 11: training-method ablation (Base → SFT → DPO / GRPO variants) on the three datasets
if len(T12):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    order = [m for m in ["base", "sft_long", "sft_v2", "dpo_v2", "grpo_stdlen_gstd", "grpo_stdlen", "grpo_judge", "grpo_ours_step60", "grpo_ours_gstd", "grpo_ours", "grpo_balanced", "grpo_ours_s2", "grpo_ours_s3"]
             if m in STAGE_MODELS]
    for ax, b in zip(axes, ("gsm8k", "math500", "aime2026")):
        x = T12[T12.bench == b].set_index("model").reindex([LABEL[m] for m in order]).dropna(subset=["acc"])
        cols = [COLOR[m] for m in order if LABEL[m] in x.index]
        ax.bar(range(len(x)), x.acc * 100, color=cols, width=0.7, edgecolor="white", linewidth=2)
        ax.errorbar(range(len(x)), x.acc * 100, yerr=[np.maximum(0, (x.acc - x.ci_low) * 100), np.maximum(0, (x.ci_high - x.acc) * 100)],
                    fmt="none", ecolor="#52514e", capsize=3, lw=1)
        ax.set_xticks(range(len(x)), x.index, rotation=40, ha="right", fontsize=8)
        ax.set_title(bench_title(b)); ax.set_ylabel("strict accuracy (%)"); ax.grid(axis="x", visible=False)
    save(fig, "fig11_training_method_ablation")

In [ ]:
# Fig 12: MPV vs single pass / majority@4 / compute-matched majority@8 / selection / oracle (same problems)
if len(MPV) and "T10" in globals():
    conds = [("single", "single pass"), ("majority_at_4", "majority@4"), ("majority_at_8", "majority@8"),
             ("selection", "selection"), ("MPV", "MPV (ours)"), ("oracle_at_4", "oracle@4")]
    fig, axes = plt.subplots(1, len(MPV_BENCHES), figsize=(15, 3.8))
    for ax, b in zip(np.atleast_1d(axes), MPV_BENCHES):
        x = T10[T10.bench == b]
        w = 0.8 / max(1, len(x))
        for j, r in enumerate(x.itertuples()):
            key = next(k for k, v in LABEL.items() if v == r.model)
            ax.bar(np.arange(len(conds)) + j * w, [getattr(r, c) * 100 for c, _ in conds], w, color=COLOR[key],
                   edgecolor="white", linewidth=1, label=r.model)
        ax.set_xticks(np.arange(len(conds)) + w * (len(x) - 1) / 2, [l for _, l in conds], rotation=25, ha="right", fontsize=8)
        ax.set_title(bench_title(b)); ax.set_ylabel("strict accuracy (%)"); ax.grid(axis="x", visible=False)
    np.atleast_1d(axes)[0].legend(fontsize=8)
    save(fig, "fig12_mpv_vs_baselines")

In [ ]:
# Fig 13: adaptive test-time compute: accuracy vs tokens per problem (policies on the same problems)
if "T13" in globals() and len(T13):
    focus = LABEL["grpo_ours"] if LABEL["grpo_ours"] in set(T13.model) else T13.model.iloc[0]
    POLICY_STYLE = {"single pass": ("o", PALETTE[0]), "majority@4": ("s", PALETTE[1]), "early-stop vote": ("v", PALETTE[2]),
                    "MPV (full)": ("D", PALETTE[6]), "adaptive MPV (early-stop)": ("*", PALETTE[4]),
                    "mode-routed MPV": ("P", PALETTE[5])}
    fig, axes = plt.subplots(1, len(MPV_BENCHES), figsize=(15, 3.8))
    for ax, b in zip(np.atleast_1d(axes), MPV_BENCHES):
        x = T13[(T13.model == focus) & (T13.bench == b)]
        for r in x.itertuples():
            mk, col = POLICY_STYLE.get(r.policy, ("X", PALETTE[3]))      # majority@8 and others: fixed fallback style
            ax.errorbar(r.mean_tokens, r.acc * 100, yerr=[[max(0, (r.acc - r.ci_low) * 100)], [max(0, (r.ci_high - r.acc) * 100)]],
                        fmt=mk, color=col, ms=9 if mk == "*" else 7, mec="white", mew=1, capsize=2, label=r.policy)
        ax.set_xscale("log"); ax.set_xlabel("mean generated tokens per problem (log)"); ax.set_ylabel("strict accuracy (%)")
        from matplotlib.ticker import FuncFormatter, NullFormatter
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}")); ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_title(f"{bench_title(b)} · {focus}")
    h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
    fig.legend(h, l, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    save(fig, "fig13_adaptive_test_time_compute")

In [ ]:
# Fig 14: judge controls × parser: is the failure the judge, the truncation, or the v1 parser? (same items)
if "JUDGE_COMPARE" in globals() and len(JUDGE_COMPARE):
    allr = JUDGE_COMPARE[JUDGE_COMPARE.subset == "all"]
    judges = list(dict.fromkeys(allr.judge))
    parsers = [("v1 parser", PALETTE[7]), ("corrected parser", PALETTE[0])]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    y = np.arange(len(judges)); h = 0.38
    for j, (pname, col) in enumerate(parsers):
        x = allr[allr.parser == pname].set_index("judge").reindex(judges)
        axes[0].barh(y + (j - 0.5) * h, x.FPR * 100, h, color=col, label=pname)
        axes[0].errorbar(x.FPR * 100, y + (j - 0.5) * h, xerr=[np.maximum(0, (x.FPR - x.FPR_ci_low) * 100),
                         np.maximum(0, (x.FPR_ci_high - x.FPR) * 100)], fmt="none", ecolor="#52514e", capsize=2)
        axes[1].barh(y + (j - 0.5) * h, x.kappa, h, color=col, label=pname)
    axes[0].set_yticks(y, judges, fontsize=8); axes[0].set_xlabel("false-positive rate (%): wrong answers accepted")
    axes[0].set_title("Wrong answers accepted"); axes[0].legend(fontsize=8)
    axes[1].set_yticks(y, [""] * len(judges)); axes[1].set_xlabel("Cohen's κ vs strict gold")
    axes[1].axvline(0, color="#52514e", lw=0.8); axes[1].set_title("Agreement with ground truth")
    save(fig, "fig14_judge_controls")

In [ ]:
# Fig 15: accuracy vs token budget (who stays accurate when compute is capped?), from Table 15
if "T15" in globals() and len(T15):
    benches = [b for b in ("gsm8k", "math500", "aime2026") if b in set(T15.bench)]
    fig, axes = plt.subplots(1, len(benches), figsize=(5 * len(benches), 3.8), squeeze=False)
    for ax, b in zip(axes[0], benches):
        for m in [m for m in BUDGET_MODELS if m in set(T15.model)]:
            x = T15[(T15.model == m) & (T15.bench == b)].sort_values("budget")
            main = m in ("grpo_ours", "thinkless", "adaptthink", "sft_v2")
            ax.plot(x.budget, x.acc * 100, marker="o", ms=4 if main else 3, lw=2.2 if main else 1.1,
                    color=COLOR.get(m, PALETTE[3]), alpha=1 if main else 0.7, label=LABEL[m])
            if main:
                ax.fill_between(x.budget, x.ci_low * 100, x.ci_high * 100, color=COLOR.get(m, PALETTE[3]), alpha=0.12)
        ax.set_xscale("log", base=2); ax.set_xticks(BUDGETS, [str(B) for B in BUDGETS])
        ax.set_xlabel("token budget (max generated tokens)"); ax.set_ylabel("strict accuracy (%)")
        ax.set_title(bench_title(b)); ax.grid(alpha=.3)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    save(fig, "fig15_budget_curves")

In [ ]:
# Fig 16: mechanism of the group-std CoT collapse: R1 (batch std) vs R1 group-std vs the S8 controls (training logs)
MECH_FIG = [("grpo_ours_gstd", "--"), ("grpo_ours_gstd_b01", "--"), ("grpo_nobonus_gstd", "--"), ("grpo_ours_nostd", "-."),
            ("grpo_ours", "-"), ("grpo_ours_s2", ":"), ("grpo_ours_s3", ":")]   # seeds 2, 3 = R1's recipe (batch std)
fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
for run, ls in MECH_FIG:
    h = run_log(run)
    if h.empty or "step" not in h:
        continue
    sm = lambda c: h[c].rolling(10, min_periods=1).mean()
    cols = (next((c for c in h.columns if "strict_accuracy" in c and c.endswith("/mean")), None),
            "completions/mean_length" if "completions/mean_length" in h else None,
            "entropy" if "entropy" in h else None,
            next((c for c in h.columns if "all_correct_group" in c and c.endswith("/mean")), None))
    for ax, c in zip(axes, cols):
        if c:
            ax.plot(h.step, sm(c), color=COLOR[run], lw=2, ls=ls, label=LABEL[run])
for ax, t in zip(axes, ("strict accuracy (rollouts)", "mean completion length (tokens)", "policy entropy",
                        "share of all-correct groups")):
    ax.set_title(t); ax.set_xlabel("GRPO step")
if axes[0].has_data():
    axes[0].legend(fontsize=7)
save(fig, "fig16_mechanism_controls")

In [ ]:
# Fig 17: does thinking pay off? forced-think minus forced-short accuracy by MATH level, per training stage (Table 16)
if "T16" in globals() and len(T16):
    x = T16[(T16.bench == "math500") & (T16.subset != "all")]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for j, (key, g) in enumerate(x.groupby("key", sort=False)):
        lv = g.subset.str[1:].astype(int).values + (j - 2) * 0.06          # small dodge so the CIs do not overlap
        ax.errorbar(lv, g.think_minus_short * 100, yerr=[np.maximum(0, (g.think_minus_short - g.ci_low) * 100),
                    np.maximum(0, (g.ci_high - g.think_minus_short) * 100)], fmt="o", color=COLOR.get(key, PALETTE[3]),
                    ls="--" if key.endswith("_gstd") else "-",   # group-std R1: its SHORT mode collapsed (Table 17)
                    lw=1.8, ms=5, capsize=2, label=LABEL[key])
    ax.axhline(0, color="#52514e", lw=0.9)
    ax.set_xticks([1, 2, 3, 4, 5], [f"level {i}" for i in range(1, 6)])
    ax.set_ylabel("think − short accuracy (points)"); ax.set_title("MATH-500: does thinking pay off? (same problems)")
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    save(fig, "fig17_think_vs_short_by_level")

In [ ]:
# Fig 18: breadth vs depth (Table 18b): accuracy vs mean tokens per problem. Lines = majority vote over the first
# k = 1..8 independent samples; markers = ONE answer (x greedy single pass, ^ forced think, ■ thinking models)
if "VOTE_CURVES" in globals() and VOTE_CURVES:
    curves = {k: v for k, v in VOTE_CURVES.items() if v["source"].startswith("full")} or VOTE_CURVES
    benches = [b for b in ("gsm8k", "math500") if any(k[1] == b for k in curves)]
    if benches:
        fig, axes = plt.subplots(1, len(benches), figsize=(6.4 * len(benches), 4.0), squeeze=False)
        for ax, b in zip(axes[0], benches):
            srcs = set()
            for (m, bb), c in curves.items():
                if bb != b:
                    continue
                srcs.add(c["source"])
                ax.plot(c["tokens"], np.array(c["acc"]) * 100, "-o", color=COLOR.get(m, PALETTE[8]), lw=1.8, ms=3.5,
                        label=f"{LABEL[m]}: vote over k samples")
                ax.annotate(f"k={c['k'][-1]}", (c["tokens"][-1], c["acc"][-1] * 100), fontsize=7,
                            xytext=(3, -9), textcoords="offset points")
                a = T18A[(T18A.bench == b) & (T18A.model == LABEL[m])] if len(T18A) else pd.DataFrame()
                if len(a) and pd.notna(a.iloc[0].single_greedy):
                    ax.plot(a.iloc[0].single_tokens, a.iloc[0].single_greedy * 100, "x", color=COLOR.get(m, PALETTE[8]),
                            ms=8, mew=2, label=f"{LABEL[m]}: one greedy answer")
            x = T18B[(T18B.bench == b) & T18B.vote_source.isin(srcs)] if len(T18B) else pd.DataFrame()
            for (dk, dc), g in (x.groupby(["depth_key", "depth_cond"], sort=False) if len(x) else []):
                r = g.iloc[0]            # the same problems for every vote model of this source
                ax.plot(r.depth_tokens, r.depth_acc * 100, "^" if dc == "forced_think" else "s", ms=8,
                        color=COLOR.get(dk, PALETTE[8]), mec="#222222", mew=0.6, ls="none",
                        label=f"{LABEL[dk]}: one think answer" if dc == "forced_think" else f"{LABEL[dk]}")
            ax.set_xscale("log"); ax.set_xlabel("mean tokens per problem (log)"); ax.set_ylabel("accuracy (%)")
            ax.set_title(f"{b}: breadth vs depth ({'; '.join(sorted(srcs))})", fontsize=9)
        h, l = axes[0][0].get_legend_handles_labels()
        fig.legend(h, l, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7)
        save(fig, "fig18_breadth_vs_depth")

In [ ]:
# Fig 19: second base model (Table 19). Rollout think-rate, completion length and rollout accuracy over GRPO steps for
# both base models (R1's recipe solid, group std dashed), and forced think − forced short by MATH level after SFT and
# after GRPO (same problems; Table 16 for the math model, Table 19c for the second model)
if "T19D" in globals() or "T19C" in globals():
    fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
    for run, ls in (("grpo_ours", "-"), ("grpo_ours_gstd", "--"), ("m2_grpo", "-"), ("m2_grpo_gstd", "--")):
        h = run_log(run)
        if h.empty or "step" not in h:
            continue
        sm = lambda c: h[c].rolling(10, min_periods=1).mean()
        cols = (next((c for c in h.columns if "think_rate" in c and c.endswith("/mean")), None),
                "completions/mean_length" if "completions/mean_length" in h else None,
                next((c for c in h.columns if "strict_accuracy" in c and c.endswith("/mean")), None))
        for ax, c, scale in zip(axes[:3], cols, (100, 1, 100)):
            if c:
                ax.plot(h.step, sm(c) * scale, color=COLOR[run], lw=2, ls=ls, label=LABEL[run])
    parts = []
    if "T16" in globals() and len(T16):
        parts.append(T16[(T16.bench == "math500") & T16.key.isin(["sft_v2", "grpo_ours"])])
    if "T19C" in globals() and len(T19C):
        parts.append(T19C)
    x = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    x = x[x.subset != "all"] if len(x) else x
    for j, (key, g) in enumerate(x.groupby("key", sort=False) if len(x) else []):
        lv = g.subset.str[1:].astype(int).values + (j - 1.5) * 0.07          # small dodge so the CIs do not overlap
        axes[3].errorbar(lv, g.think_minus_short * 100, yerr=[np.maximum(0, (g.think_minus_short - g.ci_low) * 100),
                         np.maximum(0, (g.ci_high - g.think_minus_short) * 100)], fmt="o", color=COLOR[key], ls="-",
                         lw=1.6, ms=4, capsize=2, label=LABEL[key])
    axes[3].axhline(0, color="#52514e", lw=0.9); axes[3].set_xticks([1, 2, 3, 4, 5], [f"L{i}" for i in range(1, 6)])
    for ax, t, yl in zip(axes, ("rollout think-rate", "mean completion length", "rollout strict accuracy",
                                "MATH-500: forced think − forced short"),
                         ("% of rollouts", "tokens", "%", "points")):
        ax.set_title(t); ax.set_ylabel(yl); ax.set_xlabel("MATH level" if ax is axes[3] else "GRPO step")
    handles = {}                                     # every run once, whichever panel has it
    for ax in axes:
        for h_, l_ in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(l_ if ax is not axes[3] else l_ + " (think − short)", h_)
    if handles:
        fig.legend(list(handles.values()), list(handles), loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    save(fig, "fig19_second_model")

## 41a) Failure gallery

In [ ]:
# 41a) Failure gallery
S = ALL[(ALL.model == "grpo_ours") & (ALL.condition == "single") & (ALL.kind == "greedy")]
texts = {}
for p in (EVAL / "preds").glob("grpo_ours__single.jsonl"):
    texts = {(r["pid"], r["kind"], r["sample"]): r["text"] for r in read_jsonl(p)}
if len(S) and texts:
    fails = S[~S.correct & S.bench.isin(["gsm8k", "math500"])]
    print("Failure types:", fails.status.value_counts().to_dict(), "| by mode:", fails["mode"].value_counts().to_dict())
    for r in fails.sample(min(4, len(fails)), random_state=CFG["seed"]).itertuples():
        print(f"\n### {r.pid} gold={BENCH.set_index('pid').loc[r.pid, 'gold']} pred={r.final_answer} mode={r.mode}\n",
              texts[(r.pid, 'greedy', 0)][:600])

## 41) Bundle  ·  42) Close the session (carbon report, results zip)

In [ ]:
# 41) Bundle everything to Drive
if STAGES["P11_bundle"]:
    shutil.copy(CODE / "backend_api.py", ROOT / "backend_api.py"); shutil.copy(CODE / "common.py", ROOT / "common.py")
    zip_path = ROOT / "v2_paper_bundle.zip"
    import zipfile
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in (FIG, TAB, EXP, DATA):
            for f in folder.rglob("*"):
                if f.is_file() and f.stat().st_size < 200 * 2**20:
                    z.write(f, f.relative_to(ROOT))
        for run in ("sft_v2", "grpo_ours", "grpo_balanced", "grpo_judge", "grpo_stdlen", "grpo_ours_gstd",
                    "grpo_stdlen_gstd", "grpo_ours_s2", "grpo_ours_s3", "grpo_nobonus_gstd", "grpo_ours_nostd",
                    "grpo_ours_gstd_b01", "m2_sft", "m2_grpo", "m2_grpo_gstd"):
            for f in (RUNS / run / "adapter").glob("*") if (RUNS / run / "adapter").exists() else []:
                z.write(f, f.relative_to(ROOT))
            for name in ("trainer_log_history.json", "resolved_grpo_config.json"):
                if (RUNS / run / name).exists():
                    z.write(RUNS / run / name, (RUNS / run / name).relative_to(ROOT))
        for f in (ROOT / "config.json", ROOT / "emissions.csv", ROOT / "backend_api.py", ROOT / "common.py"):
            if f.exists():
                z.write(f, f.relative_to(ROOT))
    print(f"Bundle: {zip_path} ({zip_path.stat().st_size/2**20:.1f} MB)")

In [ ]:
# 42) Close the session: energy and CO2, Table 4, results zip
emissions = tracker.stop()
print(f"Session carbon: {emissions:.4f} kg CO2eq")
if (ROOT / "emissions.csv").exists():   # Table 4 again, now including this session (40.3 ran before the tracker stopped)
    _e = pd.read_csv(ROOT / "emissions.csv")
    pd.DataFrame([dict(energy_kwh=float(_e.energy_consumed.sum()), co2_kg=float(_e.emissions.sum()),
                       source="codecarbon (GPU + CPU + RAM), all completed sessions")]
                 ).to_csv(TAB / "table4_energy_carbon.csv", index=False)
    print(f"Table 4 (energy, CO2) incl. this session: {_e.energy_consumed.sum():.3f} kWh, {_e.emissions.sum():.3f} kg CO2eq")

# One small zip of the results (tables, figures, predictions, training logs) for analysis outside Colab. Weights,
# optimizer states, checkpoints and tokenizer copies are left out.
def _share_file(p):
    rel = p.relative_to(ROOT)
    if rel.parts[0] in ("paper_tables", "paper_figures", "eval", "experiment_data") or len(rel.parts) == 1:
        return p.stat().st_size < 200e6
    if rel.parts[0] == "runs":
        return ("checkpoints" not in rel.parts and not p.name.startswith("tokenizer")
                and p.suffix in (".json", ".csv", ".txt", ".png") and p.stat().st_size < 50e6)
    return rel.parts[0] == "data" and (p.suffix in (".json", ".csv") or p.name == "benchmarks.jsonl") and p.stat().st_size < 20e6

SHARE_ZIP = ROOT.parent / f"{ROOT.name}_results_{SESSION}.zip"
try:
    import zipfile
    _tmp = Path("/content") / SHARE_ZIP.name
    with zipfile.ZipFile(_tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob("*")):
            if p.is_file() and _share_file(p):
                z.write(p, Path(ROOT.name) / p.relative_to(ROOT))
    shutil.copy(_tmp, SHARE_ZIP)
    print(f"Results zip: MyDrive/{SHARE_ZIP.name} ({SHARE_ZIP.stat().st_size / 1e6:.0f} MB)")
except Exception as e:                                   # never let the zip break the session close
    print(f"Results zip not written ({type(e).__name__}: {e}).")
print("Done. Runtime → Disconnect and delete runtime.")